# Coding Practice Book — 40 Questions (Practice & Test)

[Open in Colab](https://colab.research.google.com/github/mudit1729/mlwiki-colabs/blob/main/pytorch-code-samples/coding-practice-book.ipynb)

Forty DSA / implementation questions: arrays & prefix sums, sliding window, intervals, stateful object design (transactional / nested key-value stores, LRU, event-timeout detectors, rate limiters, log aggregation), stack parsing, matrix backtracking, graphs / BFS / DFS, trees, heaps, binary search, and bounding-box geometry (IoU / NMS).

Each question is a **two-pane study card**: the left half has the *question, basic funda, things to take care, complexity, and related questions*; the right half has the *dense pseudocode and the commented code*. Below every card is a **runnable code + test cell** — run it to verify the solution (or blank out the function body to practice, then re-run the tests).

**How to use:** run the **Setup** cell once, then for each question run its code cell to see `Qn: ALL PASS`. The final cell prints a full scoreboard.

🟢 = high-frequency / commonly asked · 🟡 = good-to-know / specialized.


In [ ]:
# ============================================================================
# Setup & test helpers  — run this cell first.
# ============================================================================
import copy, math
from collections import defaultdict, Counter, deque
import heapq, bisect

RESULTS = {}

def report(qid, checks):
    """checks: list of (label, ok_bool). Records & prints PASS/FAIL."""
    ok = all(c[1] for c in checks)
    RESULTS[qid] = ok
    print(f"{qid}: {'ALL PASS' if ok else 'FAILED'}  ({sum(c[1] for c in checks)}/{len(checks)})")
    for label, good in checks:
        print(("   ok   " if good else "  FAIL  ") + label)
    return ok

def eq(a, b): return a == b
def eq_sorted(a, b): return sorted(a) == sorted(b)
def eq_tuples(a, b): return sorted(map(tuple, a)) == sorted(map(tuple, b))
def approx(a, b, tol=1e-4): return abs(a - b) <= tol

# Binary tree node + level-order builder
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def build_tree(vals):
    """vals: level-order list with None for missing children."""
    if not vals or vals[0] is None: return None
    it = iter(vals)
    root = TreeNode(next(it)); q = deque([root])
    while q:
        node = q.popleft()
        try: lv = next(it)
        except StopIteration: break
        if lv is not None:
            node.left = TreeNode(lv); q.append(node.left)
        try: rv = next(it)
        except StopIteration: break
        if rv is not None:
            node.right = TreeNode(rv); q.append(node.right)
    return root

def find_node(root, target_val):
    if not root: return None
    if root.val == target_val: return root
    return find_node(root.left, target_val) or find_node(root.right, target_val)

def preorder(root):
    return [] if not root else [root.val] + preorder(root.left) + preorder(root.right)

# Graph node (for Clone Graph) + adjacency builder
class Node:
    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []

def build_graph(adj):
    """adj: 1-indexed adjacency list; adj[i-1] = neighbors of node i. Returns node 1."""
    if not adj: return None
    nodes = {i: Node(i) for i in range(1, len(adj) + 1)}
    for i, nbrs in enumerate(adj, 1):
        nodes[i].neighbors = [nodes[j] for j in nbrs]
    return nodes[1]

def graphs_isomorphic(a, b):
    """Structurally equal AND made of distinct objects (true deep clone)."""
    if a is None and b is None: return True
    if a is None or b is None: return False
    seen = {}
    def dfs(x, y):
        if x.val != y.val: return False
        if x is y: return False            # must be a copy, not the same object
        if id(x) in seen: return seen[id(x)] is y
        seen[id(x)] = y
        if len(x.neighbors) != len(y.neighbors): return False
        xs = sorted(x.neighbors, key=lambda z: z.val)
        ys = sorted(y.neighbors, key=lambda z: z.val)
        return all(dfs(xx, yy) for xx, yy in zip(xs, ys))
    return dfs(a, b)

print("setup ready")

<h3 style="margin:6px 0">Subarray Sum Equals K 🟢 (prefix-sum + hashmap)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Maintain a running prefix sum and count how many earlier prefix sums equal <code>prefix - k</code>, using a hash map seeded with <code>{0: 1}</code>.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given an integer array <code>nums</code> (values may be negative) and an integer <code>k</code>, return the total number of contiguous subarrays whose elements sum to <code>k</code>. A subarray is a contiguous portion of the array, and the same element may appear in multiple counted subarrays. Constraints: <code>1 ≤ nums.length ≤ 2 × 10⁴</code>, <code>-1000 ≤ nums[i] ≤ 1000</code>, <code>-10⁷ ≤ k ≤ 10⁷</code>.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>nums=[1,1,1], k=2</code></td><td><code>2</code></td><td>subarrays <code>[1,1]</code> at indices 0–1 and 1–2</td></tr><tr><td><code>nums=[1,2,3], k=3</code></td><td><code>2</code></td><td>subarrays <code>[3]</code> and <code>[1,2]</code></td></tr><tr><td><code>nums=[-1,1,0], k=0</code></td><td><code>3</code></td><td>negatives allowed; <code>[-1,1]</code>, <code>[0]</code>, <code>[-1,1,0]</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">If <code>prefix[j]</code> is the sum of <code>nums[0..j]</code> and <code>prefix[i-1]</code> is the sum up to the element before position <code>i</code>, then the subarray <code>nums[i..j]</code> sums to <code>k</code> exactly when <code>prefix[j] - prefix[i-1] == k</code>, i.e., <code>prefix[i-1] == prefix[j] - k</code>. So for each new prefix sum, we ask: how many previously seen prefix sums equal <code>prefix - k</code>? The hash map stores frequency of each prefix value encountered so far. Seeding <code>seen[0] = 1</code> handles the case where the subarray starts at index 0 (the prefix itself equals <code>k</code>).</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Forget to seed <code>seen[0] = 1</code></td><td>Subarrays starting at index 0 (where prefix itself equals <code>k</code>) are missed</td><td>Always initialize <code>seen = {0: 1}</code> before the loop</td></tr><tr><td>Use a sliding window instead of prefix sums</td><td>Sliding window can't handle negative values — it can't contract correctly</td><td>Prefix-sum + hash map works for all integer values</td></tr><tr><td>Check <code>seen[prefix - k]</code> after updating <code>seen[prefix]</code></td><td>Double-counts the current element as its own subarray complement</td><td>Lookup before inserting the current prefix</td></tr><tr><td>Use division or assume positives only</td><td>Division fails when <code>k = 0</code> or with negatives</td><td>Always use the <code>prefix - k</code> complement approach</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>All negatives</td><td><code>[-1,-1,-1], k=-2</code></td><td><code>2</code></td></tr><tr><td><code>k = 0</code> with zeros</td><td><code>[0,0,0], k=0</code></td><td><code>6</code></td></tr><tr><td>Single element equals k</td><td><code>[5], k=5</code></td><td><code>1</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n)</td><td>Single pass through <code>nums</code>; hash map lookups are O(1) average</td></tr><tr><td>Space</td><td>O(n)</td><td>Hash map stores at most n+1 distinct prefix sums</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>560</td><td>Subarray Sum Equals K</td><td>This exact problem</td></tr><tr><td>974</td><td>Subarray Sums Divisible by K</td><td>Same prefix-sum pattern; complement is <code>prefix % k</code></td></tr><tr><td>525</td><td>Contiguous Array</td><td>Prefix-sum on a transformed array (0→-1) with hash map</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">count = 0, prefix = 0
seen = {0: 1}                # seed: empty prefix has sum 0
for x in nums:
    prefix += x
    count += seen.get(prefix - k, 0)   # subarrays ending here summing to k
    seen[prefix] += 1
return count</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">from collections import defaultdict

def subarraySum(nums, k):
    count = prefix = 0
    seen = defaultdict(int)
    seen[0] = 1                      # empty prefix
    for x in nums:
        prefix += x
        count += seen[prefix - k]   # subarrays ending here with sum k
        seen[prefix] += 1
    return count</pre></td></tr></table>

In [ ]:
# ===== Q1: Subarray Sum Equals K  (prefix-sum + hashmap) =====
from collections import defaultdict

def subarraySum(nums, k):
    count = prefix = 0
    seen = defaultdict(int)
    seen[0] = 1                      # empty prefix
    for x in nums:
        prefix += x
        count += seen[prefix - k]   # subarrays ending here with sum k
        seen[prefix] += 1
    return count
report("Q1", [
    ("[1,1,1], k=2 -> 2", subarraySum([1,1,1], 2) == 2),
    ("[1,2,3], k=3 -> 2", subarraySum([1,2,3], 3) == 2),
    ("[-1,1,0], k=0 -> 3 (negatives)", subarraySum([-1,1,0], 0) == 3),
])


<h3 style="margin:6px 0">Top K Frequent Elements 🟡 (bucket sort by frequency)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Count frequencies, then bucket by frequency (index = count); scan buckets high→low to collect the top k elements in O(n).</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given an integer array <code>nums</code> and an integer <code>k</code>, return the <code>k</code> most frequent elements. The answer is guaranteed to be unique, and <code>k</code> is always valid (1 ≤ k ≤ number of distinct elements). The order of the output does not matter. Constraints: <code>1 ≤ nums.length ≤ 10⁵</code>, <code>-10⁴ ≤ nums[i] ≤ 10⁴</code>.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>nums=[1,1,1,2,2,3], k=2</code></td><td><code>[1,2]</code></td><td>1 appears 3×, 2 appears 2×</td></tr><tr><td><code>nums=[1], k=1</code></td><td><code>[1]</code></td><td>single distinct element</td></tr><tr><td><code>nums=[4,4,2,2,3], k=3</code></td><td><code>[4,2,3]</code></td><td>all three distinct values qualify</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Build a frequency map with <code>Counter</code>. Since frequency ranges from 1 to <code>len(nums)</code>, we can create <code>len(nums)+1</code> buckets indexed by frequency. Each value goes into the bucket matching its count. Then scanning from the highest index downward picks elements in order of decreasing frequency; collect until we have <code>k</code> of them. This avoids sorting and runs in O(n). The heap alternative — <code>heapq.nlargest(k, freq, key=freq.get)</code> — is O(n log k) and simpler to state if O(n) is not required.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Use a min-heap of size n</td><td>O(n log n) — worse than bucket sort</td><td>Use bucket sort O(n) or a max-heap of size k for O(n log k)</td></tr><tr><td>Sort the frequency dict by value descending</td><td>O(n log n)</td><td>Bucket sort achieves O(n) since frequency ≤ n</td></tr><tr><td>Forget <code>buckets</code> has indices 0..n, loop 0..n-1</td><td>Can miss highest-frequency bucket</td><td>Loop <code>range(len(buckets)-1, 0, -1)</code></td></tr><tr><td>Return before collecting exactly k elements</td><td>Output truncated or over-collected</td><td>Track <code>len(out) == k</code> inside the inner loop</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>All same element</td><td><code>[5,5,5], k=1</code></td><td><code>[5]</code></td></tr><tr><td>All distinct, k=1</td><td><code>[1,2,3], k=1</code></td><td>any one of <code>[1]</code>, <code>[2]</code>, or <code>[3]</code></td></tr><tr><td>k equals total distinct count</td><td><code>[1,1,2,3], k=3</code></td><td><code>[1,2,3]</code> in any order</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n)</td><td>Counter O(n) + bucket fill O(distinct) + bucket scan O(n)</td></tr><tr><td>Space</td><td>O(n)</td><td>Frequency map + bucket array both O(n)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>692</td><td>Top K Frequent Words</td><td>Same pattern; add lexicographic tiebreak</td></tr><tr><td>451</td><td>Sort Characters By Frequency</td><td>Bucket sort by char frequency</td></tr><tr><td>973</td><td>K Closest Points to Origin</td><td>"Top k by metric" — heap or quickselect</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">freq = Counter(nums)                      # value -&gt; count
buckets = [[] for _ in range(len(nums) + 1)]
for val, c in freq.items():
    buckets[c].append(val)               # bucket index = frequency

out = []
for c in range(len(buckets) - 1, 0, -1):  # high frequency first
    for val in buckets[c]:
        out.append(val)
        if len(out) == k:
            return out</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">from collections import Counter

def topKFrequent(nums, k):
    freq = Counter(nums)
    buckets = [[] for _ in range(len(nums) + 1)]
    for val, c in freq.items():
        buckets[c].append(val)
    out = []
    for c in range(len(buckets) - 1, 0, -1):
        for val in buckets[c]:
            out.append(val)
            if len(out) == k:
                return out
    return out</pre></td></tr></table>

In [ ]:
# ===== Q2: Top K Frequent Elements  (bucket sort by frequency) =====
from collections import Counter

def topKFrequent(nums, k):
    freq = Counter(nums)
    buckets = [[] for _ in range(len(nums) + 1)]
    for val, c in freq.items():
        buckets[c].append(val)
    out = []
    for c in range(len(buckets) - 1, 0, -1):
        for val in buckets[c]:
            out.append(val)
            if len(out) == k:
                return out
    return out
report("Q2", [
    ("[1,1,1,2,2,3], k=2 -> {1,2}", eq_sorted(topKFrequent([1,1,1,2,2,3], 2), [1,2])),
    ("[1], k=1 -> [1]", eq_sorted(topKFrequent([1], 1), [1])),
])


<h3 style="margin:6px 0">Product of Array Except Self 🟡 (prefix/suffix, no division)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Two-pass accumulation: first pass stores prefix products in the result array; second pass sweeps a rolling suffix product and multiplies in-place.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given an integer array <code>nums</code>, return an array <code>answer</code> where <code>answer[i]</code> is the product of all elements of <code>nums</code> except <code>nums[i]</code>. The solution must run in O(n) time and must not use the division operation. Constraints: <code>2 ≤ nums.length ≤ 10⁵</code>, <code>-30 ≤ nums[i] ≤ 30</code>. The product of any prefix or suffix fits in a 32-bit integer.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>nums=[1,2,3,4]</code></td><td><code>[24,12,8,6]</code></td><td><code>24=2×3×4</code>, <code>12=1×3×4</code>, …</td></tr><tr><td><code>nums=[-1,1,0,-3,3]</code></td><td><code>[0,0,9,0,0]</code></td><td>zero causes most outputs to be 0</td></tr><tr><td><code>nums=[2,3]</code></td><td><code>[3,2]</code></td><td>minimum length</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">For each index <code>i</code>, the desired product is <code>(product of nums[0..i-1]) × (product of nums[i+1..n-1])</code>. Build those two parts separately. In the first left-to-right pass, <code>res[i]</code> accumulates the product of all elements to the left of <code>i</code>. In the second right-to-left pass, maintain a running suffix product and multiply it into <code>res[i]</code>. No division is needed, and no zeros create divide-by-zero issues, because we never actually divide.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Using division: <code>total / nums[i]</code></td><td>Fails when any element is 0 (ZeroDivisionError or wrong result)</td><td>Two-pass prefix/suffix — no division</td></tr><tr><td>Extra O(n) space for suffix array</td><td>Wastes memory; problem asks O(1) extra</td><td>Use a rolling <code>suffix</code> scalar variable</td></tr><tr><td>Off-by-one in prefix pass: <code>res[i] = res[i-1] * nums[i]</code></td><td>Includes <code>nums[i]</code> in its own prefix</td><td>Should be <code>res[i] = res[i-1] * nums[i-1]</code></td></tr><tr><td>Not initializing <code>res[0] = 1</code></td><td>First element has no left product — it should be 1</td><td>Initialize <code>res = [1] * n</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Contains a zero</td><td><code>[1,0,3]</code></td><td><code>[0,3,0]</code></td></tr><tr><td>Contains two zeros</td><td><code>[0,0,2]</code></td><td><code>[0,0,0]</code></td></tr><tr><td>All ones</td><td><code>[1,1,1]</code></td><td><code>[1,1,1]</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n)</td><td>Two linear passes</td></tr><tr><td>Space</td><td>O(1) extra</td><td>Output array excluded; suffix is a scalar</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>42</td><td>Trapping Rain Water</td><td>Two-pass prefix/suffix accumulation (max instead of product)</td></tr><tr><td>152</td><td>Maximum Product Subarray</td><td>Product over subarrays, track min and max</td></tr><tr><td>724</td><td>Find Pivot Index</td><td>Prefix sum comparison — same two-sweep idea</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">n = len(nums)
res = [1] * n
# Pass 1: prefix products (product of everything left of i)
for i in range(1, n):
    res[i] = res[i-1] * nums[i-1]
# Pass 2: suffix products (multiply into res)
suffix = 1
for i in range(n-1, -1, -1):
    res[i] *= suffix
    suffix *= nums[i]
return res</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def productExceptSelf(nums):
    n = len(nums)
    res = [1] * n
    for i in range(1, n):
        res[i] = res[i - 1] * nums[i - 1]   # prefix product
    suffix = 1
    for i in range(n - 1, -1, -1):
        res[i] *= suffix                    # multiply suffix product
        suffix *= nums[i]
    return res</pre></td></tr></table>

In [ ]:
# ===== Q3: Product of Array Except Self  (prefix/suffix, no division) =====
def productExceptSelf(nums):
    n = len(nums)
    res = [1] * n
    for i in range(1, n):
        res[i] = res[i - 1] * nums[i - 1]   # prefix product
    suffix = 1
    for i in range(n - 1, -1, -1):
        res[i] *= suffix                    # multiply suffix product
        suffix *= nums[i]
    return res
report("Q3", [
    ("[1,2,3,4] -> [24,12,8,6]", productExceptSelf([1,2,3,4]) == [24,12,8,6]),
    ("[-1,1,0,-3,3] -> [0,0,9,0,0]", productExceptSelf([-1,1,0,-3,3]) == [0,0,9,0,0]),
])


<h3 style="margin:6px 0">Longest Consecutive Sequence 🟡 (hashset, start-only expansion)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Load all numbers into a set; for each number that has no left neighbor (it is a run's head), extend rightward by checking consecutive membership.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given an unsorted array of integers <code>nums</code>, return the length of the longest consecutive integer sequence (e.g., <code>[100, 101, 102, 103]</code> has length 4). The algorithm must run in O(n) time. Constraints: <code>0 ≤ nums.length ≤ 10⁵</code>, <code>-10⁹ ≤ nums[i] ≤ 10⁹</code>.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>nums=[100,4,200,1,3,2]</code></td><td><code>4</code></td><td>sequence <code>1,2,3,4</code></td></tr><tr><td><code>nums=[0,3,7,2,5,8,4,6,0,1]</code></td><td><code>9</code></td><td><code>0..8</code>, duplicate 0 ignored</td></tr><tr><td><code>nums=[]</code></td><td><code>0</code></td><td>empty array</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Sorting would cost O(n log n). Instead, dump everything into a hash set for O(1) membership queries. An element <code>x</code> is the start of a consecutive run only if <code>x - 1</code> is not in the set. Expanding from every element would revisit elements O(n²) times in the worst case; checking the start condition ensures each element is touched at most twice total across all runs, giving O(n). Duplicate values in the input are harmlessly deduplicated by the set.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Expand from every element, not just run heads</td><td>O(n²) for a fully consecutive array</td><td>Guard with <code>if x - 1 not in s</code></td></tr><tr><td>Sort first</td><td>O(n log n); violates problem constraint</td><td>Use a hash set for O(n)</td></tr><tr><td>Iterate over <code>nums</code> instead of <code>s</code></td><td>Duplicate values trigger redundant expansions</td><td>Iterate over the set <code>s</code></td></tr><tr><td>Use <code>range</code> or arithmetic instead of set lookup</td><td>Set lookup is O(1); arithmetic over gaps is fragile</td><td>Use <code>while x + length in s</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Empty array</td><td><code>[]</code></td><td><code>0</code></td></tr><tr><td>All duplicates</td><td><code>[5,5,5]</code></td><td><code>1</code></td></tr><tr><td>Single element</td><td><code>[42]</code></td><td><code>1</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n)</td><td>Set construction O(n); each element entered and exited at most one run</td></tr><tr><td>Space</td><td>O(n)</td><td>Hash set stores all distinct values</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>128</td><td>Longest Consecutive Sequence</td><td>This exact problem</td></tr><tr><td>298</td><td>Binary Tree Longest Consecutive Sequence</td><td>Consecutive path in a tree</td></tr><tr><td>594</td><td>Longest Harmonious Subsequence</td><td>Hashmap frequency; difference-of-1 window</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">s = set(nums)
best = 0
for x in s:
    if x - 1 not in s:           # x is the start of a run
        length = 1
        while x + length in s:
            length += 1
        best = max(best, length)
return best</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def longestConsecutive(nums):
    s = set(nums)
    best = 0
    for x in s:
        if x - 1 not in s:                  # only begin at a run's head
            length = 1
            while x + length in s:
                length += 1
            best = max(best, length)
    return best</pre></td></tr></table>

In [ ]:
# ===== Q4: Longest Consecutive Sequence  (hashset, start-only expansion) =====
def longestConsecutive(nums):
    s = set(nums)
    best = 0
    for x in s:
        if x - 1 not in s:                  # only begin at a run's head
            length = 1
            while x + length in s:
                length += 1
            best = max(best, length)
    return best
report("Q4", [
    ("[100,4,200,1,3,2] -> 4", longestConsecutive([100,4,200,1,3,2]) == 4),
    ("[0,3,7,2,5,8,4,6,0,1] -> 9", longestConsecutive([0,3,7,2,5,8,4,6,0,1]) == 9),
    ("[] -> 0", longestConsecutive([]) == 0),
])


<h3 style="margin:6px 0">Longest Substring Without Repeating Characters 🟢 (variable window)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Expand <code>right</code> character by character; when a duplicate is found, jump <code>left</code> to one past the previous occurrence of that character (guarded to never go backward).</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given a string <code>s</code>, find the length of the longest substring that contains no repeating characters. A substring is a contiguous sequence of characters within <code>s</code>. Constraints: <code>0 ≤ s.length ≤ 5 × 10⁴</code>; <code>s</code> consists of English letters, digits, symbols, and spaces.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>s="abcabcbb"</code></td><td><code>3</code></td><td><code>"abc"</code></td></tr><tr><td><code>s="bbbbb"</code></td><td><code>1</code></td><td><code>"b"</code></td></tr><tr><td><code>s="pwwkew"</code></td><td><code>3</code></td><td><code>"wke"</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Maintain a window <code>[left, right]</code> guaranteed to have no duplicates. Store the most recent index of each character in <code>last</code>. When the character at <code>right</code> was seen before at index <code>last[ch]</code>, the duplicate is inside our window only if <code>last[ch] &gt;= left</code>. In that case, advance <code>left</code> to <code>last[ch] + 1</code>, which evicts the duplicate. Always update <code>last[ch] = right</code> and track the window length. The critical guard <code>last[ch] &gt;= left</code> prevents shrinking the window due to a duplicate that is already outside it.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Missing <code>last[ch] &gt;= left</code> guard</td><td>Shrinks window due to a character already outside it</td><td>Always check <code>last[ch] &gt;= left</code> before updating <code>left</code></td></tr><tr><td>Advance <code>left</code> by one step at a time</td><td>O(n²) when many duplicates</td><td>Jump <code>left</code> directly to <code>last[ch] + 1</code></td></tr><tr><td>Use a set instead of an index map</td><td>Can't jump <code>left</code> efficiently; must scan backward</td><td>Use <code>last = {}</code> mapping char to index</td></tr><tr><td><code>best = right - left</code> (off by one)</td><td>Window length is <code>right - left + 1</code></td><td>Add 1 for inclusive bounds</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Empty string</td><td><code>""</code></td><td><code>0</code></td></tr><tr><td>All same character</td><td><code>"aaaa"</code></td><td><code>1</code></td></tr><tr><td>No repeats at all</td><td><code>"abcdef"</code></td><td><code>6</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n)</td><td>Each character visited once by <code>right</code>; <code>left</code> only moves right</td></tr><tr><td>Space</td><td>O(min(n, charset))</td><td><code>last</code> dict bounded by alphabet size</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>3</td><td>Longest Substring Without Repeating Characters</td><td>This exact problem</td></tr><tr><td>159</td><td>Longest Substring with At Most Two Distinct Characters</td><td>Same variable-window; expand the count to 2</td></tr><tr><td>424</td><td>Longest Repeating Character Replacement</td><td>Variable window with a character-count twist</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">last = {}                        # char -&gt; most recent index
left = best = 0
for right, ch in enumerate(s):
    if ch in last and last[ch] &gt;= left:
        left = last[ch] + 1      # jump past the old occurrence
    last[ch] = right
    best = max(best, right - left + 1)
return best</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def lengthOfLongestSubstring(s):
    last = {}
    left = best = 0
    for right, ch in enumerate(s):
        if ch in last and last[ch] &gt;= left:
            left = last[ch] + 1             # jump past the previous occurrence
        last[ch] = right
        best = max(best, right - left + 1)
    return best</pre></td></tr></table>

In [ ]:
# ===== Q5: Longest Substring Without Repeating Characters  (variable window) =====
def lengthOfLongestSubstring(s):
    last = {}
    left = best = 0
    for right, ch in enumerate(s):
        if ch in last and last[ch] >= left:
            left = last[ch] + 1             # jump past the previous occurrence
        last[ch] = right
        best = max(best, right - left + 1)
    return best
report("Q5", [
    ('"abcabcbb" -> 3', lengthOfLongestSubstring("abcabcbb") == 3),
    ('"bbbbb" -> 1', lengthOfLongestSubstring("bbbbb") == 1),
    ('"pwwkew" -> 3', lengthOfLongestSubstring("pwwkew") == 3),
    ('"" -> 0', lengthOfLongestSubstring("") == 0),
])


<h3 style="margin:6px 0">Minimum Window Substring 🟢 (expand/contract with need-counter)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Expand <code>right</code> to satisfy all character requirements; once satisfied, contract <code>left</code> to minimize the window; track the smallest valid window seen.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given strings <code>s</code> and <code>t</code>, return the minimum window substring of <code>s</code> such that every character in <code>t</code> (including duplicates) appears in the window. If no such window exists, return <code>""</code>. Constraints: <code>1 ≤ s.length, t.length ≤ 10⁵</code>; <code>s</code> and <code>t</code> consist of uppercase and lowercase English letters.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>s="ADOBECODEBANC", t="ABC"</code></td><td><code>"BANC"</code></td><td>shortest window containing A, B, C</td></tr><tr><td><code>s="a", t="a"</code></td><td><code>"a"</code></td><td>exact match</td></tr><tr><td><code>s="a", t="aa"</code></td><td><code>""</code></td><td>t needs two a's; s has only one</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Use a <code>Counter</code> for the characters still needed (<code>need</code>). The integer <code>missing</code> counts how many characters from <code>t</code> are not yet satisfied in the current window. Expand <code>right</code>: if <code>need[ch] &gt; 0</code>, this character fills a genuine shortage — decrement <code>missing</code>. Always decrement <code>need[ch]</code> (even if it goes negative, meaning we have surplus copies). When <code>missing == 0</code>, the window is valid: record it if it is the shortest so far, then contract from <code>left</code> — increment <code>need[s[left]]</code>, and if that makes it positive again, increment <code>missing</code> and stop contracting.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Check <code>missing == 0</code> only once per right step, not in a <code>while</code> loop</td><td>Misses smaller windows obtainable by contracting further</td><td>Use <code>while missing == 0</code> to contract maximally each time</td></tr><tr><td>Decrement <code>missing</code> for every character, not just when <code>need[ch] &gt; 0</code></td><td><code>missing</code> goes negative; never reaches 0</td><td>Only decrement <code>missing</code> when <code>need[ch] &gt; 0</code></td></tr><tr><td>Use a separate boolean "valid" flag instead of <code>missing</code> counter</td><td>Can't count duplicate requirements in <code>t</code></td><td>The <code>missing</code> integer correctly handles multi-occurrence chars</td></tr><tr><td>Forget to handle empty <code>s</code> or <code>t</code></td><td><code>Counter("")</code> or empty loops cause wrong return values</td><td>Guard at the top: <code>if not s or not t: return ""</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td><code>t</code> has repeated chars</td><td><code>s="aa", t="aa"</code></td><td><code>"aa"</code></td></tr><tr><td>No valid window exists</td><td><code>s="abc", t="d"</code></td><td><code>""</code></td></tr><tr><td><code>s == t</code></td><td><code>s="xyz", t="xyz"</code></td><td><code>"xyz"</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(</td><td>s</td><td>+</td><td>t</td><td>)</td><td>Each character in <code>s</code> pushed and popped by <code>left</code>/<code>right</code> at most once</td></tr><tr><td>Space</td><td>O(</td><td>t</td><td>+ charset)</td><td><code>need</code> counter bounded by distinct chars in <code>t</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>76</td><td>Minimum Window Substring</td><td>This exact problem</td></tr><tr><td>567</td><td>Permutation in String</td><td>Fixed-size window; same need-counter idea</td></tr><tr><td>438</td><td>Find All Anagrams in a String</td><td>Fixed window; slide and compare freq maps</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">need = Counter(t)
missing = len(t)
left = 0
best = (inf, 0, 0)
for right, ch in enumerate(s):
    if need[ch] &gt; 0:
        missing -= 1
    need[ch] -= 1
    while missing == 0:
        if right - left + 1 &lt; best[0]:
            best = (right - left + 1, left, right)
        need[s[left]] += 1
        if need[s[left]] &gt; 0:
            missing += 1
        left += 1
return "" if best[0] == inf else s[best[1]:best[2]+1]</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">from collections import Counter

def minWindow(s, t):
    if not s or not t:
        return ""
    need = Counter(t)
    missing = len(t)
    left = 0
    best = (float("inf"), 0, 0)
    for right, ch in enumerate(s):
        if need[ch] &gt; 0:
            missing -= 1
        need[ch] -= 1
        while missing == 0:                 # valid window → try to shrink
            if right - left + 1 &lt; best[0]:
                best = (right - left + 1, left, right)
            need[s[left]] += 1
            if need[s[left]] &gt; 0:
                missing += 1
            left += 1
    return "" if best[0] == float("inf") else s[best[1]:best[2] + 1]</pre></td></tr></table>

In [ ]:
# ===== Q6: Minimum Window Substring  (expand/contract with need-counter) =====
from collections import Counter

def minWindow(s, t):
    if not s or not t:
        return ""
    need = Counter(t)
    missing = len(t)
    left = 0
    best = (float("inf"), 0, 0)
    for right, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        while missing == 0:                 # valid window → try to shrink
            if right - left + 1 < best[0]:
                best = (right - left + 1, left, right)
            need[s[left]] += 1
            if need[s[left]] > 0:
                missing += 1
            left += 1
    return "" if best[0] == float("inf") else s[best[1]:best[2] + 1]
report("Q6", [
    ('("ADOBECODEBANC","ABC") -> "BANC"', minWindow("ADOBECODEBANC","ABC") == "BANC"),
    ('("a","a") -> "a"', minWindow("a","a") == "a"),
    ('("a","aa") -> ""', minWindow("a","aa") == ""),
])


<h3 style="margin:6px 0">Sliding Window Maximum 🟡 (monotonic deque)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Maintain a deque of indices whose values are strictly decreasing; the front always holds the maximum of the current window; evict from the back on entry, and from the front when it slides out of the window.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given an integer array <code>nums</code> and a sliding window of size <code>k</code> that moves from left to right, return an array of the maximum value in each window position. The window slides one position at a time. Constraints: <code>1 ≤ nums.length ≤ 10⁵</code>, <code>-10⁴ ≤ nums[i] ≤ 10⁴</code>, <code>1 ≤ k ≤ nums.length</code>.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>nums=[1,3,-1,-3,5,3,6,7], k=3</code></td><td><code>[3,3,5,5,6,7]</code></td><td>windows: <code>[1,3,-1]</code>, <code>[3,-1,-3]</code>, …</td></tr><tr><td><code>nums=[1], k=1</code></td><td><code>[1]</code></td><td>single element</td></tr><tr><td><code>nums=[1,-1], k=1</code></td><td><code>[1,-1]</code></td><td>window size 1, output equals input</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">A naive approach checks all k elements per window in O(nk). The monotonic deque achieves O(n) by maintaining a decreasing sequence of candidates. When a new element <code>x</code> arrives, any element in the deque that is ≤ <code>x</code> can never be a future window maximum (it is both older and smaller), so pop it from the back. Append the new index. The front is the maximum of the current window. Before reading the front, evict it if its index has slid out of the window (<code>dq[0] &lt;= i - k</code>). Output the front value once the first full window is formed (<code>i &gt;= k - 1</code>).</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Store values instead of indices</td><td>Cannot check if front has exited the window</td><td>Store indices; look up <code>nums[dq[0]]</code> for the value</td></tr><tr><td>Use <code>&lt;</code> instead of <code>&lt;=</code> when popping from back</td><td>Keeps equal elements that are strictly dominated; deque grows unnecessarily</td><td>Use <code>&lt;=</code> so weaker-or-equal candidates are evicted</td></tr><tr><td>Evict front with <code>&lt; i - k</code> instead of <code>&lt;= i - k</code></td><td>Off-by-one: front at <code>i - k</code> is already outside a window of size <code>k</code></td><td>Use <code>dq[0] &lt;= i - k</code></td></tr><tr><td>Append output before window is full</td><td>Outputs garbage for the first <code>k-1</code> indices</td><td>Guard with <code>if i &gt;= k - 1</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>k equals n</td><td><code>[2,1,5,3], k=4</code></td><td><code>[5]</code></td></tr><tr><td>k equals 1</td><td><code>[4,3,2,1], k=1</code></td><td><code>[4,3,2,1]</code></td></tr><tr><td>All same values</td><td><code>[3,3,3], k=2</code></td><td><code>[3,3]</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n)</td><td>Each index is pushed once and popped at most once</td></tr><tr><td>Space</td><td>O(k)</td><td>Deque holds at most k indices at any time</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>239</td><td>Sliding Window Maximum</td><td>This exact problem</td></tr><tr><td>862</td><td>Shortest Subarray with Sum at Least K</td><td>Monotonic deque on prefix sums</td></tr><tr><td>1438</td><td>Longest Continuous Subarray With Absolute Diff ≤ Limit</td><td>Two deques (min + max) for variable window</td></tr></table>
<p style="margin:4px 0"><b>See also:</b> [[wiki/syntheses/coding-dsa/queues]].</p></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">dq = deque()                    # indices, values strictly decreasing
out = []
for i, x in enumerate(nums):
    while dq and nums[dq[-1]] &lt;= x:
        dq.pop()                # can never be max while x is in window
    dq.append(i)
    if dq[0] &lt;= i - k:
        dq.popleft()            # front has slid out of window
    if i &gt;= k - 1:
        out.append(nums[dq[0]])
return out</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">from collections import deque

def maxSlidingWindow(nums, k):
    dq = deque()                            # indices, values decreasing
    out = []
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] &lt;= x:
            dq.pop()
        dq.append(i)
        if dq[0] &lt;= i - k:                  # evict out-of-window front
            dq.popleft()
        if i &gt;= k - 1:
            out.append(nums[dq[0]])
    return out</pre></td></tr></table>

In [ ]:
# ===== Q7: Sliding Window Maximum  (monotonic deque) =====
from collections import deque

def maxSlidingWindow(nums, k):
    dq = deque()                            # indices, values decreasing
    out = []
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:                  # evict out-of-window front
            dq.popleft()
        if i >= k - 1:
            out.append(nums[dq[0]])
    return out
report("Q7", [
    ("[1,3,-1,-3,5,3,6,7], k=3 -> [3,3,5,5,6,7]", maxSlidingWindow([1,3,-1,-3,5,3,6,7],3) == [3,3,5,5,6,7]),
    ("[1], k=1 -> [1]", maxSlidingWindow([1],1) == [1]),
])


<h3 style="margin:6px 0">Merge Intervals (sort by start, sweep)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Sort by start time; for each interval, either extend the current merged interval's end (if overlapping) or emit it and start a new one.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given an array of intervals <code>intervals</code> where <code>intervals[i] = [start_i, end_i]</code>, merge all overlapping intervals and return an array of the non-overlapping intervals that cover all the intervals in the input. Two intervals <code>[a,b]</code> and <code>[c,d]</code> overlap if <code>c ≤ b</code> (touching counts as overlapping). Constraints: <code>1 ≤ intervals.length ≤ 10⁴</code>, <code>0 ≤ start_i ≤ end_i ≤ 10⁴</code>.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>[[1,3],[2,6],[8,10],[15,18]]</code></td><td><code>[[1,6],[8,10],[15,18]]</code></td><td><code>[1,3]</code> and <code>[2,6]</code> merge</td></tr><tr><td><code>[[1,4],[4,5]]</code></td><td><code>[[1,5]]</code></td><td>touching intervals merge</td></tr><tr><td><code>[[1,4],[0,4]]</code></td><td><code>[[0,4]]</code></td><td>unsorted input; sort handles it</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">After sorting by start time, any new interval either overlaps the last merged interval (its start ≤ last merged end) or it does not. If it overlaps, extend the end to <code>max(last_end, new_end)</code> — the <code>max</code> is critical because the new interval might be fully contained. If it does not overlap, push the new interval onto the result. Using <code>≤</code> for the overlap check ensures touching intervals (e.g., <code>[1,3]</code> and <code>[3,5]</code>) are merged, as they share the point 3.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Use <code>&lt;</code> instead of <code>&lt;=</code> for overlap check</td><td>Touching intervals (e.g., <code>[1,3],[3,5]</code>) are not merged</td><td>Use <code>s &lt;= res[-1][1]</code></td></tr><tr><td><code>res[-1][1] = e</code> without <code>max</code></td><td>A fully contained interval shrinks the merged end</td><td>Always <code>res[-1][1] = max(res[-1][1], e)</code></td></tr><tr><td>Forget to sort before sweeping</td><td>Merging assumes sorted order; skips overlaps out of order</td><td>Sort by <code>x[0]</code> first</td></tr><tr><td>Mutate the input without sorting</td><td>May pass some test cases but fails on unsorted inputs</td><td>Sort a copy or sort in-place explicitly</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Single interval</td><td><code>[[1,5]]</code></td><td><code>[[1,5]]</code></td></tr><tr><td>All intervals identical</td><td><code>[[2,3],[2,3]]</code></td><td><code>[[2,3]]</code></td></tr><tr><td>Fully nested</td><td><code>[[1,10],[2,5],[3,4]]</code></td><td><code>[[1,10]]</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n log n)</td><td>Dominated by sort; sweep is O(n)</td></tr><tr><td>Space</td><td>O(n)</td><td>Output array in the worst case (no merges)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>56</td><td>Merge Intervals</td><td>This exact problem</td></tr><tr><td>57</td><td>Insert Interval</td><td>Merge after inserting a new interval (next question)</td></tr><tr><td>252</td><td>Meeting Rooms</td><td>Check for any overlap — simplified merge</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">sort intervals by start
res = [intervals[0]]
for s, e in intervals[1:]:
    if s &lt;= res[-1][1]:              # overlaps or touches
        res[-1][1] = max(res[-1][1], e)
    else:
        res.append([s, e])
return res</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def merge(intervals):
    intervals.sort(key=lambda x: x[0])
    res = [intervals[0]]
    for s, e in intervals[1:]:
        if s &lt;= res[-1][1]:
            res[-1][1] = max(res[-1][1], e) # max guards a contained interval
        else:
            res.append([s, e])
    return res</pre></td></tr></table>

In [ ]:
# ===== Q8: Merge Intervals (sort by start, sweep) =====
def merge(intervals):
    intervals.sort(key=lambda x: x[0])
    res = [intervals[0]]
    for s, e in intervals[1:]:
        if s <= res[-1][1]:
            res[-1][1] = max(res[-1][1], e) # max guards a contained interval
        else:
            res.append([s, e])
    return res
report("Q8", [
    ("merge overlaps", eq_tuples(merge([[1,3],[2,6],[8,10],[15,18]]), [[1,6],[8,10],[15,18]])),
    ("touching merge", eq_tuples(merge([[1,4],[4,5]]), [[1,5]])),
])


<h3 style="margin:6px 0">Insert Interval (three phases)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Three-phase linear scan: copy intervals that end before the new one starts, absorb all overlapping intervals into the new one, then copy the rest.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given a list of non-overlapping, sorted intervals <code>intervals</code> and a new interval <code>newInterval</code>, insert <code>newInterval</code> into the correct position and merge all overlapping intervals. Return the resulting list. You may assume <code>intervals</code> is already sorted by start and contains no overlapping intervals before the insertion. Constraints: <code>0 ≤ intervals.length ≤ 10⁴</code>, <code>intervals[i] = [start_i, end_i]</code>, <code>0 ≤ start_i ≤ end_i ≤ 10⁵</code>.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>intervals=[[1,3],[6,9]], newInterval=[2,5]</code></td><td><code>[[1,5],[6,9]]</code></td><td>new interval overlaps <code>[1,3]</code></td></tr><tr><td><code>intervals=[[1,2],[3,5],[6,7],[8,10],[12,16]], newInterval=[4,8]</code></td><td><code>[[1,2],[3,10],[12,16]]</code></td><td>merges three intervals</td></tr><tr><td><code>intervals=[], newInterval=[5,7]</code></td><td><code>[[5,7]]</code></td><td>empty list</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Because the input is sorted and non-overlapping, the insert can be handled in three non-overlapping phases with a single pointer <code>i</code>: (1) while the current interval ends strictly before the new interval starts (<code>intervals[i][1] &lt; s</code>), it cannot overlap — just copy it; (2) while the current interval's start is ≤ the new interval's end (<code>intervals[i][0] &lt;= e</code>), it overlaps — expand <code>[s, e]</code> to absorb it; (3) after absorbing all overlaps, emit the merged <code>[s, e]</code> and copy the remaining intervals unchanged. No sorting needed since the input is already sorted.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Phase 1 condition <code>intervals[i][0] &lt; s</code> instead of <code>intervals[i][1] &lt; s</code></td><td>Uses start instead of end; skips intervals that end before the new one starts</td><td>Compare end of existing vs start of new: <code>intervals[i][1] &lt; s</code></td></tr><tr><td>Phase 2 condition <code>intervals[i][1] &lt;= e</code> instead of <code>intervals[i][0] &lt;= e</code></td><td>Uses end instead of start; misses intervals that start inside the new range</td><td>Compare start of existing vs end of new: <code>intervals[i][0] &lt;= e</code></td></tr><tr><td>Forget <code>min</code>/<code>max</code> when absorbing</td><td>New interval may be absorbed into an existing one, shrinking bounds</td><td>Always <code>s = min(s, intervals[i][0])</code> and <code>e = max(e, intervals[i][1])</code></td></tr><tr><td>Emit <code>[s, e]</code> inside the merge loop</td><td>Emits partial merges before all overlaps are absorbed</td><td>Emit once, after the merge loop exits</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>New interval before all existing</td><td><code>[[3,5]], [1,2]</code></td><td><code>[[1,2],[3,5]]</code></td></tr><tr><td>New interval after all existing</td><td><code>[[1,2]], [3,5]</code></td><td><code>[[1,2],[3,5]]</code></td></tr><tr><td>New interval swallows all</td><td><code>[[1,2],[3,4]], [0,10]</code></td><td><code>[[0,10]]</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n)</td><td>Single linear scan; each interval visited once</td></tr><tr><td>Space</td><td>O(n)</td><td>Output array</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>57</td><td>Insert Interval</td><td>This exact problem</td></tr><tr><td>56</td><td>Merge Intervals</td><td>Sort then sweep; generalizes insert</td></tr><tr><td>715</td><td>Range Module</td><td>Interval insert/remove on a sorted set</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">res = []
i, n = 0, len(intervals)
s, e = newInterval
# Phase 1: copy non-overlapping intervals before newInterval
while i &lt; n and intervals[i][1] &lt; s:
    res.append(intervals[i]); i += 1
# Phase 2: merge overlapping intervals
while i &lt; n and intervals[i][0] &lt;= e:
    s = min(s, intervals[i][0])
    e = max(e, intervals[i][1])
    i += 1
res.append([s, e])
# Phase 3: copy remaining
while i &lt; n:
    res.append(intervals[i]); i += 1
return res</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def insert(intervals, newInterval):
    res = []
    i, n = 0, len(intervals)
    s, e = newInterval
    while i &lt; n and intervals[i][1] &lt; s:    # before
        res.append(intervals[i]); i += 1
    while i &lt; n and intervals[i][0] &lt;= e:   # overlapping → absorb
        s = min(s, intervals[i][0])
        e = max(e, intervals[i][1])
        i += 1
    res.append([s, e])
    while i &lt; n:                            # after
        res.append(intervals[i]); i += 1
    return res</pre></td></tr></table>

In [ ]:
# ===== Q9: Insert Interval (three phases) =====
def insert(intervals, newInterval):
    res = []
    i, n = 0, len(intervals)
    s, e = newInterval
    while i < n and intervals[i][1] < s:    # before
        res.append(intervals[i]); i += 1
    while i < n and intervals[i][0] <= e:   # overlapping → absorb
        s = min(s, intervals[i][0])
        e = max(e, intervals[i][1])
        i += 1
    res.append([s, e])
    while i < n:                            # after
        res.append(intervals[i]); i += 1
    return res
report("Q9", [
    ("insert overlap", eq_tuples(insert([[1,3],[6,9]],[2,5]), [[1,5],[6,9]])),
    ("merge three", eq_tuples(insert([[1,2],[3,5],[6,7],[8,10],[12,16]],[4,8]), [[1,2],[3,10],[12,16]])),
    ("empty", eq_tuples(insert([],[5,7]), [[5,7]])),
])


<h3 style="margin:6px 0">Meeting Rooms II (min-heap of end times)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Sort by start; use a min-heap of end times representing active rooms; if the earliest-ending room frees before the next meeting starts, reuse it; otherwise allocate a new room.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given an array of meeting time intervals <code>intervals</code> where <code>intervals[i] = [start_i, end_i]</code>, return the minimum number of conference rooms required to hold all meetings without conflict. Two meetings conflict if one starts strictly before the other ends. Constraints: <code>1 ≤ intervals.length ≤ 10⁴</code>, <code>0 ≤ start_i &lt; end_i ≤ 10⁶</code>.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>[[0,30],[5,10],[15,20]]</code></td><td><code>2</code></td><td><code>[0,30]</code> overlaps both others</td></tr><tr><td><code>[[7,10],[2,4]]</code></td><td><code>1</code></td><td>non-overlapping; one room suffices</td></tr><tr><td><code>[[1,5],[2,6],[3,7]]</code></td><td><code>3</code></td><td>all three overlap; need 3 rooms</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">The heap always holds the end times of meetings currently occupying rooms. After sorting by start time, for each meeting check if the earliest-ending room (heap top) finishes at or before this meeting's start. If yes, that room is free — reuse it by replacing the heap top with the new end time (<code>heapreplace</code>). If no, no room is free — push a new end time, growing the heap. At the end, the heap size equals the number of rooms needed, which is the peak concurrency (the maximum overlap at any point in time).</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Use <code>&lt; s</code> instead of <code>&lt;= s</code> for the free-room check</td><td>A meeting ending exactly when the next starts could share a room; using <code>&lt;</code> misses this</td><td>Use <code>heap[0] &lt;= s</code></td></tr><tr><td>Sort by end time instead of start time</td><td>Heap logic requires processing by arrival order</td><td>Sort by <code>x[0]</code> (start)</td></tr><tr><td>Count intervals that overlap at a single moment instead of using the heap</td><td>O(n²); misses the "reuse a room" optimization</td><td>Min-heap of end times is O(n log n) and elegant</td></tr><tr><td>Forget to guard <code>if heap</code> before accessing <code>heap[0]</code></td><td><code>heap[0]</code> on empty heap raises <code>IndexError</code></td><td>Check <code>if heap and heap[0] &lt;= s</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Empty input</td><td><code>[]</code></td><td><code>0</code></td></tr><tr><td>All meetings at the same time</td><td><code>[[1,5],[1,5],[1,5]]</code></td><td><code>3</code></td></tr><tr><td>Sequential meetings (no overlap)</td><td><code>[[1,2],[2,3],[3,4]]</code></td><td><code>1</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n log n)</td><td>Sort O(n log n) + n heap ops each O(log n)</td></tr><tr><td>Space</td><td>O(n)</td><td>Heap holds at most n end times</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>253</td><td>Meeting Rooms II</td><td>This exact problem</td></tr><tr><td>252</td><td>Meeting Rooms</td><td>Simpler version: just detect any conflict</td></tr><tr><td>1353</td><td>Maximum Number of Events That Can Be Attended</td><td>Greedy with a heap; "which room do I take?"</td></tr></table>
<p style="margin:4px 0"><b>See also:</b> [[wiki/syntheses/coding-dsa/interval-problems]].</p></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">sort intervals by start
heap = []                        # end times of active meetings
for s, e in intervals:
    if heap and heap[0] &lt;= s:    # earliest-ending room is free
        heapreplace(heap, e)     # reuse it
    else:
        heappush(heap, e)        # need a new room
return len(heap)</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">import heapq

def minMeetingRooms(intervals):
    if not intervals:
        return 0
    intervals.sort(key=lambda x: x[0])
    heap = []                               # end times of live meetings
    for s, e in intervals:
        if heap and heap[0] &lt;= s:
            heapq.heapreplace(heap, e)      # reuse the room that freed earliest
        else:
            heapq.heappush(heap, e)
    return len(heap)</pre></td></tr></table>

In [ ]:
# ===== Q10: Meeting Rooms II (min-heap of end times) =====
import heapq

def minMeetingRooms(intervals):
    if not intervals:
        return 0
    intervals.sort(key=lambda x: x[0])
    heap = []                               # end times of live meetings
    for s, e in intervals:
        if heap and heap[0] <= s:
            heapq.heapreplace(heap, e)      # reuse the room that freed earliest
        else:
            heapq.heappush(heap, e)
    return len(heap)
report("Q10", [
    ("[[0,30],[5,10],[15,20]] -> 2", minMeetingRooms([[0,30],[5,10],[15,20]]) == 2),
    ("[[7,10],[2,4]] -> 1", minMeetingRooms([[7,10],[2,4]]) == 1),
])


<h3 style="margin:6px 0">Non-overlapping Intervals (greedy by earliest end)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Activity-selection greedy: sort by end time; greedily keep each interval whose start is ≥ the last kept interval's end; count removals as <code>total − kept</code>.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given an array of intervals <code>intervals</code>, return the minimum number of intervals you need to remove to make the rest non-overlapping. An interval <code>[a,b]</code> and <code>[c,d]</code> overlap if <code>a &lt; d</code> and <code>c &lt; b</code> (touching at a single point does NOT count as overlap for this problem). Constraints: <code>1 ≤ intervals.length ≤ 10⁵</code>, <code>-5 × 10⁴ ≤ start_i &lt; end_i ≤ 5 × 10⁴</code>.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>[[1,2],[2,3],[3,4],[1,3]]</code></td><td><code>1</code></td><td>remove <code>[1,3]</code>; the rest are non-overlapping</td></tr><tr><td><code>[[1,2],[1,2],[1,2]]</code></td><td><code>2</code></td><td>keep one, remove two</td></tr><tr><td><code>[[1,2],[2,3]]</code></td><td><code>0</code></td><td>touching but not overlapping; 0 removals</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">This is the classic activity-selection problem. Sorting by end time is the greedy key: among all intervals that don't conflict with what we've kept so far, choosing the one that ends earliest leaves the most room for future intervals. If the current interval's start is ≥ the last kept end (<code>s &gt;= end</code>), keep it and update <code>end</code>. Otherwise, skip it (count it as removed). The number of intervals to remove is <code>len(intervals) - kept</code>.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Sort by start instead of end</td><td>Greedy must pick the interval ending earliest to maximize future room</td><td>Sort by <code>x[1]</code> (end)</td></tr><tr><td>Use <code>s &gt; end</code> instead of <code>s &gt;= end</code></td><td>Touching intervals (problem definition: touching is NOT overlapping) are wrongly excluded</td><td>Use <code>s &gt;= end</code></td></tr><tr><td>Count removed intervals directly with a flag</td><td>More error-prone; must toggle correctly at each step</td><td>Count <code>keep</code>, return <code>len(intervals) - keep</code></td></tr><tr><td>Mistake this for Merge Intervals</td><td>Merge Intervals uses sort-by-start; this uses sort-by-end</td><td>Different greedy criterion for different goals</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>One interval</td><td><code>[[1,5]]</code></td><td><code>0</code></td></tr><tr><td>All intervals identical</td><td><code>[[1,3],[1,3],[1,3]]</code></td><td><code>2</code></td></tr><tr><td>No overlaps</td><td><code>[[1,2],[3,4],[5,6]]</code></td><td><code>0</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n log n)</td><td>Dominated by sort; linear sweep is O(n)</td></tr><tr><td>Space</td><td>O(1) extra</td><td>Only <code>end</code> and <code>keep</code> scalars (plus sort in-place)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>435</td><td>Non-overlapping Intervals</td><td>This exact problem</td></tr><tr><td>452</td><td>Minimum Number of Arrows to Burst Balloons</td><td>Same activity-selection greedy, sort by end</td></tr><tr><td>646</td><td>Maximum Length of Pair Chain</td><td>Same greedy; maximize kept count</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">sort intervals by end time
end = -inf
keep = 0
for s, e in intervals:
    if s &gt;= end:         # no conflict with last kept interval
        keep += 1
        end = e
return len(intervals) - keep</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def eraseOverlapIntervals(intervals):
    intervals.sort(key=lambda x: x[1])      # sort by END
    end = float("-inf")
    keep = 0
    for s, e in intervals:
        if s &gt;= end:
            keep += 1
            end = e
    return len(intervals) - keep</pre></td></tr></table>

In [ ]:
# ===== Q11: Non-overlapping Intervals (greedy by earliest end) =====
def eraseOverlapIntervals(intervals):
    intervals.sort(key=lambda x: x[1])      # sort by END
    end = float("-inf")
    keep = 0
    for s, e in intervals:
        if s >= end:
            keep += 1
            end = e
    return len(intervals) - keep
report("Q11", [
    ("one removal", eraseOverlapIntervals([[1,2],[2,3],[3,4],[1,3]]) == 1),
    ("keep one of three", eraseOverlapIntervals([[1,2],[1,2],[1,2]]) == 2),
    ("touching -> 0", eraseOverlapIntervals([[1,2],[2,3]]) == 0),
])


<h3 style="margin:6px 0">Employee Free Time (flatten, sort, find gaps)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Flatten all employee schedules into one list, sort by start, sweep with a running max end-time; any gap between the max end and the next interval's start is common free time.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given a list of schedules <code>schedule</code> where <code>schedule[i]</code> is a list of non-overlapping intervals sorted for the i-th employee, return the list of finite intervals representing common, positive-length free time for all employees, also in sorted order. Intervals for different employees may overlap. Constraints: <code>1 ≤ schedule.length, schedule[i].length ≤ 50</code>; <code>0 ≤ intervals[i][j] ≤ 10⁸</code>. (In LeetCode form, <code>schedule[i]</code> contains <code>Interval</code> objects with <code>.start</code>/<code>.end</code>; the logic is identical.)</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>[[[1,3],[6,7]],[[2,4]],[[2,5],[9,12]]]</code></td><td><code>[[5,6],[7,9]]</code></td><td>gaps between merged busy intervals</td></tr><tr><td><code>[[[1,3],[3,6]],[[2,7]]]</code></td><td><code>[]</code></td><td>completely covered; no free time</td></tr><tr><td><code>[[[1,2],[3,4]],[[5,6]]]</code></td><td><code>[[2,3],[4,5]]</code></td><td>two separate free-time windows</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Each employee's individual intervals are non-overlapping, but across employees there can be overlaps that together cover time continuously. The approach mirrors Merge Intervals: flatten all intervals into one sorted list, then sweep with a running <code>end</code> tracking how far the union of busy time has extended. Whenever the next interval's start exceeds <code>end</code>, there is a gap — this gap <code>[end, next_start]</code> is free for everyone. Update <code>end</code> to <code>max(end, next_end)</code> to handle containment. Because we only care about gaps, we never need to explicitly merge into a result list of busy times.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Forget <code>end = max(end, e)</code></td><td>A contained interval shrinks <code>end</code>; next gap appears too early</td><td>Always take <code>max(end, e)</code></td></tr><tr><td>Use <code>s &gt;= end</code> for the gap condition</td><td>Zero-width "gaps" (touching intervals) produce empty free-time windows</td><td>Use <code>s &gt; end</code> (strict inequality)</td></tr><tr><td>Sort individual employee schedules independently</td><td>Cross-employee overlaps are invisible; gaps are computed per-employee</td><td>Flatten ALL intervals before sorting</td></tr><tr><td>Expect <code>Interval</code> objects on LeetCode (LC 759)</td><td>LeetCode passes <code>Interval</code> namedtuples with <code>.start</code>/<code>.end</code></td><td>Access <code>.start</code>/<code>.end</code> or adapt tuple access</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>One employee, no gaps</td><td><code>[[[1,5]]]</code></td><td><code>[]</code></td></tr><tr><td>All employees busy at same time</td><td><code>[[[1,3]],[[1,3]]]</code></td><td><code>[]</code></td></tr><tr><td>Single gap between two clusters</td><td><code>[[[1,2]],[[4,5]]]</code></td><td><code>[[2,4]]</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n log n)</td><td>Flatten is O(n); sort is O(n log n); sweep is O(n)</td></tr><tr><td>Space</td><td>O(n)</td><td>Flattened list of all intervals</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>759</td><td>Employee Free Time</td><td>This exact problem (LeetCode Premium)</td></tr><tr><td>56</td><td>Merge Intervals</td><td>Core subroutine: merge busy time, find holes</td></tr><tr><td>1229</td><td>Meeting Scheduler</td><td>Find first common free slot of given duration</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">intervals = sorted(flatten(schedule), key=start)
free = []
_, end = intervals[0]
for s, e in intervals[1:]:
    if s &gt; end:                  # gap found: everyone is free in [end, s]
        free.append([end, s])
    end = max(end, e)
return free</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def employeeFreeTime(schedule):
    intervals = sorted(iv for emp in schedule for iv in emp)  # (start, end) tuples
    free = []
    _, end = intervals[0]
    for s, e in intervals[1:]:
        if s &gt; end:
            free.append([end, s])           # gap = free time for everyone
        end = max(end, e)
    return free</pre></td></tr></table>

In [ ]:
# ===== Q12: Employee Free Time (flatten, sort, find gaps) =====
def employeeFreeTime(schedule):
    intervals = sorted(iv for emp in schedule for iv in emp)  # (start, end) tuples
    free = []
    _, end = intervals[0]
    for s, e in intervals[1:]:
        if s > end:
            free.append([end, s])           # gap = free time for everyone
        end = max(end, e)
    return free
report("Q12", [
    ("gap [3,4]", eq_tuples(employeeFreeTime([[[1,2],[5,6]],[[1,3]],[[4,10]]]), [[3,4]])),
    ("two gaps", eq_tuples(employeeFreeTime([[[1,3],[6,7]],[[2,4]],[[2,5],[9,12]]]), [[5,6],[7,9]])),
])


<h3 style="margin:6px 0">Time-Based Key-Value Store 🟡 (hashmap + binary search)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Per-key sorted timestamp list; <code>bisect_right</code> finds the latest entry at or before a query timestamp.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Design a key-value store that supports versioned writes. <code>set(key, value, timestamp)</code> stores <code>value</code> at the given <code>timestamp</code>; <code>get(key, timestamp)</code> returns the value with the <b>largest timestamp ≤ the query</b> (or <code>""</code> if none exists). Timestamps passed to <code>set</code> are guaranteed to be strictly increasing per key.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Operation</th><th align="left">Arguments</th><th align="left">Returns</th><th align="left">Note</th></tr><tr><td><code>set</code></td><td><code>"foo", "bar", 1</code></td><td>—</td><td>stores bar at t=1</td></tr><tr><td><code>set</code></td><td><code>"foo", "baz", 4</code></td><td>—</td><td>stores baz at t=4</td></tr><tr><td><code>get</code></td><td><code>"foo", 3</code></td><td><code>"bar"</code></td><td>latest ts ≤ 3 is t=1</td></tr><tr><td><code>get</code></td><td><code>"foo", 4</code></td><td><code>"baz"</code></td><td>exact match</td></tr><tr><td><code>get</code></td><td><code>"foo", 0</code></td><td><code>""</code></td><td>query precedes all writes</td></tr><tr><td><code>get</code></td><td><code>"unknown", 5</code></td><td><code>""</code></td><td>key never set</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Because <code>set</code> timestamps arrive in strictly increasing order per key, each key's timestamp list is already sorted. A <code>get</code> query at time <code>t</code> wants the rightmost timestamp that is ≤ <code>t</code>, which is a textbook <code>bisect_right(timestamps, t) - 1</code> lookup. Store timestamps and values in parallel lists so both can be indexed together.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Using <code>bisect_left</code> instead of <code>bisect_right</code></td><td>An exact timestamp match returns the position of that element, not just past it; subtracting 1 yields the previous value instead of the matching one</td><td>Use <code>bisect_right</code> so an exact match returns position <code>i</code> where <code>ts[i-1] == timestamp</code></td></tr><tr><td>Not guarding <code>i == 0</code></td><td>Indexing <code>vals[i - 1]</code> when <code>i == 0</code> returns the last element (Python wraps) — silently wrong</td><td>Return <code>""</code> when <code>i == 0</code></td></tr><tr><td>Storing values in a single interleaved list</td><td>Off-by-one errors and harder to index</td><td>Keep parallel <code>times</code> and <code>vals</code> lists</td></tr><tr><td>Assuming per-key timestamps may arrive out of order</td><td>Would require re-sorting on every <code>set</code>, breaking O(1) amortized</td><td>Problem guarantees increasing order; state this assumption explicitly</td></tr><tr><td>Returning <code>None</code> instead of <code>""</code></td><td>API contract specifies the empty string sentinel</td><td>Return <code>""</code> for missing key or no valid timestamp</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Query timestamp precedes all writes</td><td><code>set("k","v",5); get("k",3)</code></td><td><code>""</code></td></tr><tr><td>Exact timestamp match</td><td><code>set("k","v",5); get("k",5)</code></td><td><code>"v"</code></td></tr><tr><td>Key never set</td><td><code>get("missing", 10)</code></td><td><code>""</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Method</th><th align="left">Time</th><th align="left">Space</th><th align="left">Why</th></tr><tr><td><code>set</code></td><td>O(1) amortized</td><td>O(1) per call</td><td>Append to list; timestamps arrive sorted</td></tr><tr><td><code>get</code></td><td>O(log n)</td><td>O(1)</td><td>Binary search over n stored timestamps for this key</td></tr><tr><td>Total space</td><td>—</td><td>O(N)</td><td>N = total set calls across all keys</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>981</td><td>Time Based Key-Value Store</td><td>This exact problem</td></tr><tr><td>729</td><td>My Calendar I</td><td>Interval-based versioned storage using binary search</td></tr><tr><td>1146</td><td>Snapshot Array</td><td>Per-index versioned snapshots; same bisect_right pattern</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">class TimeMap:
    times: dict[key -&gt; list[int]]   # monotonically increasing timestamps per key
    vals:  dict[key -&gt; list[str]]   # parallel values

    set(key, value, timestamp):
        append timestamp to times[key]
        append value to vals[key]
        # O(1) amortized — timestamps arrive in order

    get(key, timestamp):
        if key not in times: return ""
        i = bisect_right(times[key], timestamp)
        # i == 0 → all timestamps &gt; query → no valid write
        return vals[key][i - 1] if i &gt; 0 else ""</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">import bisect

class TimeMap:
    def __init__(self):
        self.times = {}                     # key -&gt; [timestamps] (increasing)
        self.vals = {}                      # key -&gt; [values]

    def set(self, key, value, timestamp):
        self.times.setdefault(key, []).append(timestamp)
        self.vals.setdefault(key, []).append(value)

    def get(self, key, timestamp):
        if key not in self.times:
            return ""
        i = bisect.bisect_right(self.times[key], timestamp)
        return self.vals[key][i - 1] if i else ""   # latest ts &lt;= query</pre></td></tr></table>

In [ ]:
# ===== Q13: Time-Based Key-Value Store  (hashmap + binary search) =====
import bisect

class TimeMap:
    def __init__(self):
        self.times = {}                     # key -> [timestamps] (increasing)
        self.vals = {}                      # key -> [values]

    def set(self, key, value, timestamp):
        self.times.setdefault(key, []).append(timestamp)
        self.vals.setdefault(key, []).append(value)

    def get(self, key, timestamp):
        if key not in self.times:
            return ""
        i = bisect.bisect_right(self.times[key], timestamp)
        return self.vals[key][i - 1] if i else ""   # latest ts <= query
tm = TimeMap(); tm.set("foo","bar",1)
report("Q13", [
    ('get foo@1 -> bar', tm.get("foo",1) == "bar"),
    ('get foo@3 -> bar', tm.get("foo",3) == "bar"),
    ('after set bar2@4, get foo@4 -> bar2', (tm.set("foo","bar2",4) or tm.get("foo",4)) == "bar2"),
    ('get foo@5 -> bar2', tm.get("foo",5) == "bar2"),
    ('missing key -> ""', tm.get("nope",1) == ""),
])


<h3 style="margin:6px 0">Transactional Key-Value Store 🟢 (single staging layer)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> One staging dict (<code>txn</code>) overlays the base store; a tombstone sentinel distinguishes "deleted inside txn" from "never set."</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Design a key-value store with transactional semantics. Implement <code>set(key, value)</code>, <code>get(key) -&gt; value | None</code>, <code>delete(key)</code>, <code>begin()</code>, <code>commit() -&gt; bool</code>, and <code>rollback() -&gt; bool</code>. Inside an active transaction, reads see staged (uncommitted) writes first; outside a transaction, operations apply directly to the base store. <code>commit</code> flushes the staged layer into the base store. <code>rollback</code> discards it. Both return <code>False</code> (not crash) when no transaction is open.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Operation</th><th align="left">State</th><th align="left">Returns</th><th align="left">Base after</th></tr><tr><td><code>set("a", 1)</code></td><td>no txn</td><td>—</td><td><code>{"a": 1}</code></td></tr><tr><td><code>begin()</code></td><td>—</td><td>—</td><td>txn = <code>{}</code></td></tr><tr><td><code>set("a", 2)</code></td><td>in txn</td><td>—</td><td>base unchanged; txn = <code>{"a": 2}</code></td></tr><tr><td><code>delete("b")</code></td><td>in txn</td><td>—</td><td>txn = <code>{"a": 2, "b": DELETED}</code></td></tr><tr><td><code>rollback()</code></td><td>in txn</td><td><code>True</code></td><td>base = <code>{"a": 1}</code>; txn gone</td></tr><tr><td><code>commit()</code></td><td>no txn</td><td><code>False</code></td><td>no change</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Keep one <code>txn</code> dict (or <code>None</code> when no transaction is open). Reads check <code>txn</code> first, then fall through to <code>base</code>. Deletes inside a transaction write a tombstone sentinel (a unique object) into <code>txn</code> — this shadows the base value so <code>get</code> returns <code>None</code> for that key even though the key still exists in <code>base</code>. On <code>commit</code>, replay <code>txn</code> into <code>base</code>, converting tombstones to actual <code>base.pop</code> calls. On <code>rollback</code>, simply discard <code>txn</code>.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td><code>commit</code>/<code>rollback</code> crashes when no transaction is open</td><td>The caller expects a <code>False</code> return (or a "NO TRANSACTION" sentinel), not an exception</td><td>Guard with <code>if self.txn is None: return False</code></td></tr><tr><td><code>delete</code> inside txn calls <code>base.pop</code> directly</td><td>The base is mutated before commit; rollback cannot undo it</td><td>Write the tombstone <code>_DELETED</code> into <code>txn</code> instead</td></tr><tr><td><code>get</code> does not check for tombstone</td><td>A deleted key returns its old base value through the transparent fallthrough</td><td>Check <code>v is _DELETED</code> and return <code>None</code></td></tr><tr><td><code>commit</code> skips delete propagation</td><td>Tombstones sit in base forever</td><td>Iterate txn and call <code>base.pop</code> for <code>_DELETED</code> entries</td></tr><tr><td>Using <code>None</code> as the tombstone</td><td>Cannot distinguish "key set to None" from "key deleted" if the store should allow <code>None</code> values</td><td>Use a dedicated sentinel object (<code>_DELETED = object()</code>)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Sequence</th><th align="left">Expected</th></tr><tr><td>Delete inside txn, then rollback</td><td><code>set("a",1); begin(); delete("a"); rollback(); get("a")</code></td><td><code>1</code> (base unchanged)</td></tr><tr><td>Commit a delete</td><td><code>set("a",1); begin(); delete("a"); commit(); get("a")</code></td><td><code>None</code></td></tr><tr><td><code>commit</code> / <code>rollback</code> with no open txn</td><td><code>commit()</code> or <code>rollback()</code> cold</td><td>Returns <code>False</code>, no crash</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Method</th><th align="left">Time</th><th align="left">Space</th></tr><tr><td><code>set</code> / <code>delete</code> / <code>get</code></td><td>O(1)</td><td>O(1) per call</td></tr><tr><td><code>commit</code></td><td>O(k) where k = keys in txn</td><td>O(1) extra</td></tr><tr><td><code>rollback</code></td><td>O(1)</td><td>O(1)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>1166</td><td>Design File System</td><td>Staged writes with path-based keys</td></tr><tr><td>146</td><td>LRU Cache</td><td>Stateful store with eviction policy</td></tr><tr><td>1381</td><td>Design a Stack With Increment Operation</td><td>Stateful object; O(1) amortized mutation</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">_DELETED = sentinel object          # unique identity, not None or ""

class KVStore:
    base: dict
    txn:  dict | None               # None = no active transaction

    set(key, value):
        active_layer = txn if txn is not None else base
        active_layer[key] = value

    delete(key):
        if txn is not None:
            txn[key] = _DELETED     # shadow; do NOT touch base yet
        else:
            base.pop(key, None)

    get(key):
        check txn first (if active and key present)
            if value is _DELETED: return None
            else: return value
        check base
            if missing or _DELETED: return None
            else: return value

    begin():
        if txn is not None: raise error (one-level only)
        txn = {}

    commit():
        if txn is None: return False
        for k, v in txn:
            if v is _DELETED: base.pop(k, None)
            else: base[k] = v
        txn = None; return True

    rollback():
        if txn is None: return False
        txn = None; return True</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">_DELETED = object()                         # tombstone sentinel

class KVStore:
    def __init__(self):
        self.base = {}
        self.txn = None                     # staged writes, or None if no active txn

    def set(self, key, value):
        (self.txn if self.txn is not None else self.base)[key] = value

    def delete(self, key):
        if self.txn is not None:
            self.txn[key] = _DELETED        # shadow the base value
        else:
            self.base.pop(key, None)

    def get(self, key):
        if self.txn is not None and key in self.txn:
            v = self.txn[key]
        else:
            v = self.base.get(key)
        return None if v is _DELETED or v is None else v

    def begin(self):
        if self.txn is not None:
            raise RuntimeError("transaction already open")  # flat: one level only
        self.txn = {}

    def commit(self):
        if self.txn is None:
            return False                    # NO TRANSACTION
        for k, v in self.txn.items():
            if v is _DELETED:
                self.base.pop(k, None)
            else:
                self.base[k] = v
        self.txn = None
        return True

    def rollback(self):
        if self.txn is None:
            return False
        self.txn = None
        return True</pre></td></tr></table>

In [ ]:
# ===== Q14: Transactional Key-Value Store  (single staging layer) =====
_DELETED = object()                         # tombstone sentinel

class KVStore:
    def __init__(self):
        self.base = {}
        self.txn = None                     # staged writes, or None if no active txn

    def set(self, key, value):
        (self.txn if self.txn is not None else self.base)[key] = value

    def delete(self, key):
        if self.txn is not None:
            self.txn[key] = _DELETED        # shadow the base value
        else:
            self.base.pop(key, None)

    def get(self, key):
        if self.txn is not None and key in self.txn:
            v = self.txn[key]
        else:
            v = self.base.get(key)
        return None if v is _DELETED or v is None else v

    def begin(self):
        if self.txn is not None:
            raise RuntimeError("transaction already open")  # flat: one level only
        self.txn = {}

    def commit(self):
        if self.txn is None:
            return False                    # NO TRANSACTION
        for k, v in self.txn.items():
            if v is _DELETED:
                self.base.pop(k, None)
            else:
                self.base[k] = v
        self.txn = None
        return True

    def rollback(self):
        if self.txn is None:
            return False
        self.txn = None
        return True
kv = KVStore(); kv.set("a","1")
c = []
c.append(('get a -> 1', kv.get("a") == "1"))
kv.begin(); kv.set("a","2")
c.append(('in txn get a -> 2', kv.get("a") == "2"))
c.append(('rollback -> True', kv.rollback() is True))
c.append(('after rollback get a -> 1', kv.get("a") == "1"))
kv.begin(); kv.delete("a")
c.append(('deleted in txn get a -> None', kv.get("a") is None))
c.append(('commit -> True', kv.commit() is True))
c.append(('after commit get a -> None', kv.get("a") is None))
c.append(('rollback with no txn -> False', kv.rollback() is False))
report("Q14", c)


<h3 style="margin:6px 0">Nested Transactional Key-Value Store 🟢 (stack of overlay dicts)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Replace the single <code>txn</code> staging dict with a stack of overlay dicts; <code>begin</code> pushes, <code>commit</code> merges top → parent, <code>rollback</code> pops; <code>get</code> walks the stack newest → oldest.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Extend the transactional KV store (Q14) to support <b>nested</b> <code>begin</code>/<code>commit</code>/<code>rollback</code> calls. Each <code>begin</code> opens a new inner transaction on top of any already-open ones. <code>commit</code> merges only the innermost transaction into its immediate parent (not all the way to base). <code>rollback</code> discards only the innermost transaction. A second <code>rollback</code> on an outer transaction is still possible after an inner commit.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Operation</th><th align="left">Stack state</th><th align="left"><code>get("a")</code></th></tr><tr><td><code>begin()</code></td><td><code>[{}]</code></td><td>—</td></tr><tr><td><code>set("a", 1)</code></td><td><code>[{"a":1}]</code></td><td><code>1</code></td></tr><tr><td><code>begin()</code></td><td><code>[{"a":1}, {}]</code></td><td><code>1</code></td></tr><tr><td><code>set("a", 2)</code></td><td><code>[{"a":1}, {"a":2}]</code></td><td><code>2</code></td></tr><tr><td><code>rollback()</code></td><td><code>[{"a":1}]</code></td><td><code>1</code></td></tr><tr><td><code>commit()</code></td><td><code>[]</code>; base=<code>{"a":1}</code></td><td><code>1</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">A stack of dicts generalizes the single-staging-dict design. The topmost dict is always the write target. <code>get</code> resolves from the top of the stack downward, stopping at the first layer that contains the key (with tombstone awareness). <code>commit</code> pops the top and merges it into the next-lower layer (or into <code>base</code> if the stack is now empty). <code>rollback</code> just pops. Depth is unbounded by design.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td><code>commit</code> merges directly to <code>base</code> instead of to parent</td><td>Inner <code>commit</code> should only surface to the enclosing transaction, not all the way out</td><td><code>parent = stack[-1] if stack else base</code> after popping</td></tr><tr><td><code>get</code> only checks <code>stack[-1]</code>, ignoring outer layers</td><td>A key set in an outer txn and not overridden in inner txn returns <code>None</code></td><td>Walk all layers in reverse (innermost first)</td></tr><tr><td>Tombstone propagated to intermediate layer as <code>base.pop</code></td><td>Popping from base when merging into a non-base parent removes the key before the outer txn commits</td><td>Only call <code>base.pop</code> when the parent IS <code>base</code>; otherwise store the tombstone in the parent dict</td></tr><tr><td><code>rollback</code> discards the entire stack</td><td>Should only discard the innermost transaction</td><td><code>stack.pop()</code> (not <code>stack.clear()</code>)</td></tr><tr><td><code>commit</code>/<code>rollback</code> crashes with empty stack</td><td>API contract requires <code>False</code> return</td><td>Guard <code>if not self.stack: return False</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Sequence</th><th align="left">Expected</th></tr><tr><td>Inner commit, outer rollback</td><td><code>begin; set a=1; begin; set a=2; commit; rollback; get a</code></td><td><code>None</code> (outer txn discarded)</td></tr><tr><td>Delete in inner txn, inner commit, outer commit</td><td><code>set a=1; begin; begin; delete a; commit; commit; get a</code></td><td><code>None</code></td></tr><tr><td>Double rollback on one begin</td><td><code>begin(); rollback(); rollback()</code></td><td>Second <code>rollback</code> returns <code>False</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Method</th><th align="left">Time</th><th align="left">Space</th></tr><tr><td><code>set</code> / <code>delete</code></td><td>O(1)</td><td>O(1)</td></tr><tr><td><code>get</code></td><td>O(depth)</td><td>O(1)</td></tr><tr><td><code>commit</code></td><td>O(keys in innermost layer)</td><td>O(1) extra</td></tr><tr><td><code>rollback</code></td><td>O(1)</td><td>—</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>1166</td><td>Design File System</td><td>Transactional path writes</td></tr><tr><td>155</td><td>Min Stack</td><td>Stack-based state layering</td></tr><tr><td>716</td><td>Max Stack</td><td>Nested undo/redo via layered state</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">_DELETED = sentinel

class NestedKVStore:
    base:  dict
    stack: list[dict]           # stack[-1] = innermost active txn

    set(key, value):
        top()[key] = value      # top() = stack[-1] if stack else base

    delete(key):
        if stack: stack[-1][key] = _DELETED
        else: base.pop(key, None)

    get(key):
        for layer in reversed(stack):   # innermost → outermost
            if key in layer:
                return None if layer[key] is _DELETED else layer[key]
        v = base.get(key)
        return None if v is _DELETED else v

    begin():
        stack.append({})

    commit():
        if not stack: return False
        top = stack.pop()
        parent = stack[-1] if stack else base
        for k, v in top.items():
            if v is _DELETED and parent is base:
                parent.pop(k, None)     # propagate delete to base
            else:
                parent[k] = v          # tombstone shadows deeper layers when merging up
        return True

    rollback():
        if not stack: return False
        stack.pop(); return True</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">_DELETED = object()

class NestedKVStore:
    def __init__(self):
        self.base = {}
        self.stack = []                     # one overlay dict per open transaction

    def _top(self):
        return self.stack[-1] if self.stack else self.base

    def set(self, key, value):
        self._top()[key] = value

    def delete(self, key):
        if self.stack:
            self.stack[-1][key] = _DELETED
        else:
            self.base.pop(key, None)

    def get(self, key):
        for layer in reversed(self.stack):  # innermost → outermost
            if key in layer:
                v = layer[key]
                return None if v is _DELETED else v
        v = self.base.get(key)
        return None if v is _DELETED else v

    def begin(self):
        self.stack.append({})

    def commit(self):                       # merge innermost into its parent
        if not self.stack:
            return False
        top = self.stack.pop()
        parent = self.stack[-1] if self.stack else self.base
        for k, v in top.items():
            if v is _DELETED and parent is self.base:
                parent.pop(k, None)
            else:
                parent[k] = v               # tombstone shadows deeper layers
        return True

    def rollback(self):                     # discard innermost only
        if not self.stack:
            return False
        self.stack.pop()
        return True</pre></td></tr></table>

In [ ]:
# ===== Q15: Nested Transactional Key-Value Store  (stack of overlay dicts) =====
_DELETED = object()

class NestedKVStore:
    def __init__(self):
        self.base = {}
        self.stack = []                     # one overlay dict per open transaction

    def _top(self):
        return self.stack[-1] if self.stack else self.base

    def set(self, key, value):
        self._top()[key] = value

    def delete(self, key):
        if self.stack:
            self.stack[-1][key] = _DELETED
        else:
            self.base.pop(key, None)

    def get(self, key):
        for layer in reversed(self.stack):  # innermost → outermost
            if key in layer:
                v = layer[key]
                return None if v is _DELETED else v
        v = self.base.get(key)
        return None if v is _DELETED else v

    def begin(self):
        self.stack.append({})

    def commit(self):                       # merge innermost into its parent
        if not self.stack:
            return False
        top = self.stack.pop()
        parent = self.stack[-1] if self.stack else self.base
        for k, v in top.items():
            if v is _DELETED and parent is self.base:
                parent.pop(k, None)
            else:
                parent[k] = v               # tombstone shadows deeper layers
        return True

    def rollback(self):                     # discard innermost only
        if not self.stack:
            return False
        self.stack.pop()
        return True
n = NestedKVStore(); n.set("x",10)
c = [('get x -> 10', n.get("x") == 10)]
n.begin(); n.set("x",20); n.begin(); n.set("x",30)
c.append(('innermost get x -> 30', n.get("x") == 30))
c.append(('rollback -> True', n.rollback() is True))
c.append(('after inner rollback get x -> 20', n.get("x") == 20))
c.append(('commit -> True', n.commit() is True))
c.append(('after commit get x -> 20', n.get("x") == 20))
c.append(('rollback no txn -> False', n.rollback() is False))
report("Q15", c)


<h3 style="margin:6px 0">LRU Cache 🟡 (ordered dict / hashmap + DLL)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> <code>OrderedDict</code> maintains insertion/access order; <code>move_to_end</code> on access + <code>popitem(last=False)</code> on overflow gives O(1) get/put.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Design a cache with <b>Least Recently Used</b> eviction. <code>get(key)</code> returns the value (or -1 if missing) and marks the key as most-recently used. <code>put(key, value)</code> inserts or updates the value and marks it most-recently used; if capacity is exceeded, evict the least-recently-used key.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Operation</th><th align="left">Cache state (MRU → LRU)</th><th align="left">Returns</th></tr><tr><td><code>put(1, 1)</code></td><td><code>[1]</code></td><td>—</td></tr><tr><td><code>put(2, 2)</code></td><td><code>[2, 1]</code></td><td>—</td></tr><tr><td><code>get(1)</code></td><td><code>[1, 2]</code></td><td><code>1</code> (1 promoted)</td></tr><tr><td><code>put(3, 3)</code></td><td><code>[3, 1, 2]</code> cap=2 → evict 2 → <code>[3, 1]</code></td><td>—</td></tr><tr><td><code>get(2)</code></td><td><code>[3, 1]</code></td><td><code>-1</code> (evicted)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">We need O(1) access (a hash map) AND O(1) recency tracking with eviction (an ordered sequence). <code>OrderedDict</code> combines both: it's a hash map with an internal doubly-linked list preserving insertion/access order. <code>move_to_end(key)</code> promotes a key to MRU in O(1). <code>popitem(last=False)</code> evicts the LRU in O(1). The explicit-DLL approach (required in "from scratch" follow-ups) maintains a <code>head</code>/<code>tail</code> sentinel doubly-linked list alongside a dict mapping keys to nodes.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td><code>put</code> on an existing key does not update recency</td><td>The old entry sits at its original position; the wrong key gets evicted next</td><td>Call <code>move_to_end</code> before or after writing the updated value</td></tr><tr><td><code>get</code> returns the value without promoting</td><td>Future eviction evicts the accessed key too soon</td><td>Always call <code>move_to_end</code> on a cache hit</td></tr><tr><td><code>put</code> checks capacity before writing</td><td>Off-by-one: the new key is counted before it is inserted, evicting an entry unnecessarily</td><td>Write first, then check <code>len &gt; cap</code></td></tr><tr><td>"From scratch" DLL forgets to update both <code>prev</code> and <code>next</code> pointers</td><td>Linked list corruption → wrong eviction order</td><td>Use sentinel head/tail nodes and a <code>_remove</code> + <code>_prepend</code> helper</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Sequence</th><th align="left">Expected</th></tr><tr><td>Capacity = 1</td><td><code>put(1,1); put(2,2); get(1)</code></td><td><code>-1</code> (1 was evicted when 2 was put)</td></tr><tr><td>Update existing key</td><td><code>put(1,1); put(1,10); get(1)</code></td><td><code>10</code></td></tr><tr><td>Get on empty cache</td><td><code>get(5)</code></td><td><code>-1</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Method</th><th align="left">Time</th><th align="left">Space</th></tr><tr><td><code>get</code></td><td>O(1)</td><td>O(1)</td></tr><tr><td><code>put</code></td><td>O(1)</td><td>O(1) per call; O(capacity) total</td></tr></table>
<p style="margin:4px 0"><b>Say this:</b> <code>OrderedDict</code> is the fast path; the "from scratch" answer is a <b>hashmap + doubly linked list</b> with sentinel head/tail (full skeleton in [[wiki/syntheses/coding-dsa/oop-design]]). Both give O(1) <code>get</code>/<code>put</code>. <b>Pitfall:</b> <code>put</code> on an existing key must update recency too.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>146</td><td>LRU Cache</td><td>This exact problem</td></tr><tr><td>460</td><td>LFU Cache</td><td>Evict least-frequently used; requires per-frequency buckets</td></tr><tr><td>432</td><td>All O`one Data Structure</td><td>O(1) min/max with ordered frequency buckets</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">class LRUCache:
    cache: OrderedDict          # key -&gt; value; order = LRU (front) to MRU (back)
    cap: int

    get(key):
        if key not in cache: return -1
        cache.move_to_end(key)  # promote to MRU
        return cache[key]

    put(key, value):
        if key in cache:
            cache.move_to_end(key)
        cache[key] = value
        if len(cache) &gt; cap:
            cache.popitem(last=False)   # evict LRU (front)</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.cache = OrderedDict()

    def get(self, key):
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)         # mark most-recent
        return self.cache[key]

    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) &gt; self.cap:
            self.cache.popitem(last=False)  # evict least-recent</pre></td></tr></table>

In [ ]:
# ===== Q16: LRU Cache  (ordered dict / hashmap + DLL) =====
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.cache = OrderedDict()

    def get(self, key):
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)         # mark most-recent
        return self.cache[key]

    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.cap:
            self.cache.popitem(last=False)  # evict least-recent
lru = LRUCache(2); lru.put(1,1); lru.put(2,2)
c = [('get 1 -> 1', lru.get(1) == 1)]
lru.put(3,3)  # evicts key 2
c.append(('get 2 -> -1 (evicted)', lru.get(2) == -1))
lru.put(4,4)  # evicts key 1
c.append(('get 1 -> -1 (evicted)', lru.get(1) == -1))
c.append(('get 3 -> 3', lru.get(3) == 3))
c.append(('get 4 -> 4', lru.get(4) == 4))
report("Q16", c)


<h3 style="margin:6px 0">Event Stream with Timeout 🟢 (deque TTL; heap variant for refresh)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Pure-TTL stream → deque (FIFO expiry, O(1) count); refreshable events (ping resets deadline) → min-heap + lazy deletion keyed by deadline.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0"><b>Variant A (pure TTL):</b> Design an event stream where each event expires <code>timeout</code> seconds after arrival. Implement <code>add_event(timestamp, event_id, etype)</code> and <code>count_active_events(etype)</code> (count of non-expired events of the given type) plus <code>expire_old_events(now)</code> to lazily flush expired entries.</p>
<p style="margin:4px 0"><b>Variant B (event-timeout detector):</b> Events can be refreshed — a <code>ping(event_id, ts)</code> resets the deadline to <code>ts + T</code>. Detect which events have timed out at query time <code>now</code>.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Variant</th><th align="left">Data structure</th><th align="left">Expiry order</th><th align="left">Per-event deadline mutable?</th></tr><tr><td>A — pure TTL</td><td>deque</td><td>FIFO (arrival order)</td><td>No</td></tr><tr><td>B — refreshable</td><td>min-heap + lazy delete</td><td>Deadline order</td><td>Yes (ping)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0"><b>Variant A:</b> Events expire in arrival order (no refresh), so a <code>deque</code> is ideal — the oldest events are always at the front. Keep a <code>defaultdict(int)</code> of active counts per type so <code>count_active_events</code> is O(1). Expire by popping from the left as long as <code>events[0].ts &lt;= now - timeout</code>.</p>
<p style="margin:4px 0"><b>Variant B:</b> A <code>ping</code> moves an event's deadline forward, breaking FIFO order. We need a priority queue keyed by deadline. But stale heap entries (from before a ping) are hard to remove, so use <b>lazy deletion</b>: when popping the heap, validate the popped entry against the <code>last</code> dict. If the stored deadline doesn't match the current expected deadline, the entry is stale — skip it. If the event has ended (<code>event_id not in last</code>), also skip it.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Using a deque for the refreshable variant</td><td><code>ping</code> changes expiry order; front-of-deque may no longer be the soonest to expire</td><td>Use a min-heap keyed by deadline</td></tr><tr><td>Not validating heap entries against <code>last</code> on pop</td><td>Stale entries (from before a ping) or ended events appear as false timeouts</td><td>Check <code>last.get(eid) + T == deadline</code>; skip otherwise</td></tr><tr><td>Decrementing count to negative when no events of that type remain</td><td>Count underflows; subsequent adds give wrong counts</td><td>Guard count deletion: <code>if count == 0: del active_by_type[etype]</code></td></tr><tr><td><code>expire_old_events</code> uses <code>&lt; now - timeout</code> instead of <code>&lt;=</code></td><td>Events at exactly the boundary are not expired</td><td>Use <code>&lt;=</code> — an event at <code>ts == now - timeout</code> has exactly reached its TTL</td></tr><tr><td><code>count_active_events</code> never expires lazily</td><td>Count drifts upward over time</td><td>Call <code>expire_old_events(now)</code> before each count query, or maintain it eagerly</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Expected</th></tr><tr><td>Event pinged once, then times out at new deadline</td><td><code>update(e,0); update(e,5); timed_out(5+T)</code></td><td><code>[e]</code></td></tr><tr><td>Event ended before timeout fires</td><td><code>update(e,0); end(e); timed_out(T)</code></td><td><code>[]</code></td></tr><tr><td>No events added; <code>expire_old_events</code> called</td><td><code>expire_old_events(100)</code></td><td>No crash, deque stays empty</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Variant</th><th align="left">Method</th><th align="left">Time</th><th align="left">Space</th></tr><tr><td>Deque (A)</td><td><code>add_event</code> / <code>expire</code> / <code>count</code></td><td>O(1) amortized</td><td>O(active events)</td></tr><tr><td>Heap (B)</td><td><code>update</code> / <code>end</code> / <code>timed_out</code></td><td>O(log n) amortized</td><td>O(total updates)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>362</td><td>Design Hit Counter</td><td>Sliding-window event count, deque-based</td></tr><tr><td>346</td><td>Moving Average from Data Stream</td><td>Fixed-size window, deque</td></tr><tr><td>355</td><td>Design Twitter</td><td>Time-ordered event stream with per-user state</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace"># Variant A — deque
class EventStream:
    timeout: int
    events: deque[(ts, event_id, etype)]
    active_by_type: dict[etype -&gt; int]

    add_event(ts, event_id, etype):
        events.append((ts, event_id, etype))
        active_by_type[etype] += 1

    expire_old_events(now):
        while events and events[0].ts &lt;= now - timeout:
            _, _, etype = events.popleft()
            active_by_type[etype] -= 1

    count_active_events(etype):
        return active_by_type.get(etype, 0)

# Variant B — min-heap + lazy deletion
class TimeoutDetector:
    T: int
    last: dict[event_id -&gt; last_update_ts]   # absent = event ended
    heap: [(deadline, event_id)]

    update(event_id, ts):           # start or ping
        last[event_id] = ts
        heap.push((ts + T, event_id))

    end(event_id):
        last.pop(event_id, None)    # next pop will see it missing → stale

    timed_out(now):
        results = []
        while heap and heap[0].deadline &lt;= now:
            deadline, eid = heap.pop()
            if eid not in last: continue              # ended
            if last[eid] + T != deadline: continue   # stale (pinged since)
            results.append(eid); del last[eid]
        return results</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">from collections import deque, defaultdict

class EventStream:
    def __init__(self, timeout):
        self.timeout = timeout
        self.events = deque()               # (timestamp, event_id, type) in arrival order
        self.active_by_type = defaultdict(int)

    def add_event(self, timestamp, event_id, etype):
        self.events.append((timestamp, event_id, etype))
        self.active_by_type[etype] += 1

    def expire_old_events(self, now):
        while self.events and self.events[0][0] &lt;= now - self.timeout:
            _, _, etype = self.events.popleft()
            self.active_by_type[etype] -= 1
            if self.active_by_type[etype] == 0:
                del self.active_by_type[etype]

    def count_active_events(self, etype):
        return self.active_by_type[etype]</pre></td></tr></table>

In [ ]:
# ===== Q17: Event Stream with Timeout  (deque TTL; heap variant for refresh) =====
from collections import deque, defaultdict

class EventStream:
    def __init__(self, timeout):
        self.timeout = timeout
        self.events = deque()               # (timestamp, event_id, type) in arrival order
        self.active_by_type = defaultdict(int)

    def add_event(self, timestamp, event_id, etype):
        self.events.append((timestamp, event_id, etype))
        self.active_by_type[etype] += 1

    def expire_old_events(self, now):
        while self.events and self.events[0][0] <= now - self.timeout:
            _, _, etype = self.events.popleft()
            self.active_by_type[etype] -= 1
            if self.active_by_type[etype] == 0:
                del self.active_by_type[etype]

    def count_active_events(self, etype):
        return self.active_by_type[etype]
es = EventStream(5)
es.add_event(1,"a","click"); es.add_event(2,"b","click"); es.add_event(3,"c","view")
es.expire_old_events(3)
c = [('active click -> 2', es.count_active_events("click") == 2),
     ('active view -> 1', es.count_active_events("view") == 1)]
es.expire_old_events(7)   # drops ts<=2 -> a,b expire
c.append(('after expire(7) click -> 0', es.count_active_events("click") == 0))
c.append(('after expire(7) view -> 1', es.count_active_events("view") == 1))
report("Q17", c)


<h3 style="margin:6px 0">Sliding Window Rate Limiter 🟡 (per-user timestamp log)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Per-user deque of request timestamps; evict entries older than <code>window</code> before each check; allow if <code>len(deque) &lt; max_requests</code>.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Design a rate limiter that enforces at most <code>max_requests</code> requests per <code>window</code> seconds for each user. Implement <code>allow_request(user_id, timestamp) -&gt; bool</code>. If the request is within the limit, record it and return <code>True</code>; otherwise return <code>False</code> and do not record it. Timestamps are non-decreasing within a single user's calls.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Call</th><th align="left">user</th><th align="left">ts</th><th align="left">window=60, max=3</th><th align="left">queue after</th><th align="left">Returns</th></tr><tr><td>1</td><td>A</td><td>1</td><td>—</td><td><code>[1]</code></td><td><code>True</code></td></tr><tr><td>2</td><td>A</td><td>30</td><td>—</td><td><code>[1,30]</code></td><td><code>True</code></td></tr><tr><td>3</td><td>A</td><td>60</td><td>—</td><td><code>[1,30,60]</code></td><td><code>True</code></td></tr><tr><td>4</td><td>A</td><td>61</td><td>evict ts=1 (61-60=1, 1&lt;=1) → <code>[30,60]</code></td><td><code>[30,60,61]</code></td><td><code>True</code></td></tr><tr><td>5</td><td>A</td><td>62</td><td><code>[30,60,61]</code> len=3</td><td>—</td><td><code>False</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Maintain a per-user <code>deque</code> of timestamps at which requests were allowed. Before each new request at <code>timestamp</code>, pop from the left any entries ≤ <code>timestamp - window</code> (they are outside the sliding window). If the remaining count is below <code>max_requests</code>, allow the request (append the timestamp) and return <code>True</code>. Otherwise return <code>False</code>. The deque is bounded by <code>max_requests</code> in the steady state, so memory is O(max_requests) per active user.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Using <code>&lt; timestamp - window</code> instead of <code>&lt;=</code></td><td>A request exactly at the left boundary is counted as still-active, making the window too permissive by one slot</td><td>Use <code>&lt;=</code>; a timestamp at exactly <code>now - window</code> is outside the half-open <code>(now-window, now]</code> window</td></tr><tr><td>Appending the timestamp even when the request is denied</td><td>The deque grows beyond <code>max_requests</code>; future queries under-count available capacity</td><td>Only <code>append</code> when returning <code>True</code></td></tr><tr><td>Single global deque instead of per-user deque</td><td>All users share one log; one busy user blocks others</td><td>Use <code>defaultdict(deque)</code> keyed by <code>user_id</code></td></tr><tr><td>Not evicting before checking length</td><td>Stale entries inflate the count; legitimate requests are rejected</td><td>Always evict before checking <code>len(q)</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Expected</th></tr><tr><td>Exactly <code>max_requests</code> in window</td><td><code>allow</code> called <code>max</code> times within <code>window</code> seconds</td><td>All <code>True</code>; next call <code>False</code></td></tr><tr><td>Large time gap resets window</td><td>After <code>window</code> seconds idle, full quota again</td><td><code>True</code> (all old timestamps evicted)</td></tr><tr><td>Multiple users isolated</td><td>User A throttled; User B's first request</td><td><code>True</code> for B regardless of A</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time per call</td><td>O(1) amortized</td><td>Each timestamp pushed and popped at most once</td></tr><tr><td>Space per user</td><td>O(max_requests)</td><td>Deque bounded by rate-limit capacity</td></tr></table>
<p style="margin:4px 0"><b>Key idea:</b> exact sliding-window log; O(1) amortized, O(max) memory per user. Mention the <b>sliding-window-counter</b> approximation (blend current+previous fixed buckets) for O(1) memory, and <b>Redis</b> for the distributed case.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>933</td><td>Number of Recent Calls</td><td>Exactly this problem: count calls in last 3000 ms</td></tr><tr><td>362</td><td>Design Hit Counter</td><td>Count hits in past 5 minutes; same deque pattern</td></tr><tr><td>1429</td><td>First Unique Number</td><td>Deque + set for stateful stream queries</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">class SlidingWindowRateLimiter:
    max_requests: int
    window: int
    log: dict[user_id -&gt; deque[timestamp]]

    allow_request(user_id, timestamp):
        q = log[user_id]
        # 1. Evict expired timestamps
        while q and q[0] &lt;= timestamp - window:
            q.popleft()
        # 2. Check capacity
        if len(q) &lt; max_requests:
            q.append(timestamp)
            return True
        return False</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">from collections import deque, defaultdict

class SlidingWindowRateLimiter:
    def __init__(self, max_requests, window):
        self.max = max_requests
        self.window = window
        self.log = defaultdict(deque)       # user_id -&gt; timestamps

    def allow_request(self, user_id, timestamp):
        q = self.log[user_id]
        while q and q[0] &lt;= timestamp - self.window:
            q.popleft()                     # drop timestamps outside the window
        if len(q) &lt; self.max:
            q.append(timestamp)
            return True
        return False</pre></td></tr></table>

In [ ]:
# ===== Q18: Sliding Window Rate Limiter  (per-user timestamp log) =====
from collections import deque, defaultdict

class SlidingWindowRateLimiter:
    def __init__(self, max_requests, window):
        self.max = max_requests
        self.window = window
        self.log = defaultdict(deque)       # user_id -> timestamps

    def allow_request(self, user_id, timestamp):
        q = self.log[user_id]
        while q and q[0] <= timestamp - self.window:
            q.popleft()                     # drop timestamps outside the window
        if len(q) < self.max:
            q.append(timestamp)
            return True
        return False
rl = SlidingWindowRateLimiter(2, 5)
c = [('t=1 allow', rl.allow_request("u",1) is True),
     ('t=2 allow', rl.allow_request("u",2) is True),
     ('t=3 deny (>2 in window)', rl.allow_request("u",3) is False),
     ('t=7 allow (window cleared)', rl.allow_request("u",7) is True)]
report("Q18", c)


<h3 style="margin:6px 0">Token Bucket Rate Limiter 🟡 (lazy refill)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Lazy token refill on each request: <code>tokens = min(capacity, tokens + elapsed * rate)</code>; allow if <code>tokens &gt;= cost</code>.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Design a token bucket rate limiter. The bucket has a <code>capacity</code> (max tokens) and refills at <code>refill_rate</code> tokens per second. On each <code>allow_request(timestamp, cost=1)</code> call: first add tokens proportional to time elapsed since the last call (clamped to <code>capacity</code>), then consume <code>cost</code> tokens if available. Return <code>True</code> if granted, <code>False</code> otherwise.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Call</th><th align="left">ts</th><th align="left">elapsed</th><th align="left">tokens before refill</th><th align="left">tokens after refill</th><th align="left">tokens after request</th><th align="left">Returns</th></tr><tr><td>init</td><td>—</td><td>—</td><td>10 (full)</td><td>—</td><td>—</td><td>—</td></tr><tr><td><code>allow(1, cost=3)</code></td><td>1</td><td>1 s</td><td>10</td><td>10 (clamp)</td><td>7</td><td><code>True</code></td></tr><tr><td><code>allow(1.5, cost=5)</code></td><td>1.5</td><td>0.5 s</td><td>7</td><td>min(10, 7+2.5)=9.5</td><td>4.5</td><td><code>True</code></td></tr><tr><td><code>allow(1.6, cost=6)</code></td><td>1.6</td><td>0.1 s</td><td>4.5</td><td>min(10, 4.5+0.5)=5</td><td>—</td><td><code>False</code></td></tr></table>
<p style="margin:4px 0">*(capacity=10, refill_rate=5 tokens/s)*</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">The token bucket decouples burst from sustained throughput: <code>capacity</code> sets the maximum burst (the most tokens ever available at once), <code>refill_rate</code> sets the maximum sustained rate. Rather than running a background thread that adds tokens periodically, use <b>lazy refill</b>: on every request, compute how many tokens have accumulated since <code>last</code> (<code>elapsed * rate</code>), add them, and clamp to <code>capacity</code>. This is mathematically equivalent and needs no concurrency. If <code>last</code> is <code>None</code> (first call), initialize it to <code>timestamp</code>.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Forgetting to clamp tokens to <code>capacity</code></td><td>An idle bucket accumulates tokens beyond capacity; the first burst after a long pause is larger than permitted</td><td><code>tokens = min(capacity, tokens + elapsed * rate)</code></td></tr><tr><td>Not initializing <code>last</code> on the first call</td><td><code>elapsed = timestamp - None</code> raises <code>TypeError</code></td><td>Guard: <code>if self.last is None: self.last = timestamp</code></td></tr><tr><td>Updating <code>last</code> only when the request is granted</td><td><code>elapsed</code> on the next denied call is measured from the last *granted* time, double-counting idle accumulation</td><td>Always update <code>last = timestamp</code>, regardless of grant/deny</td></tr><tr><td>Using integer division for tokens</td><td>Fractional tokens (e.g. 0.5 token/s) round to 0; limiter never refills at low rates</td><td>Use <code>float</code> throughout</td></tr><tr><td>Conflating <code>capacity</code> with <code>refill_rate</code></td><td>Burst size and sustained rate are independent; capacity is not tokens per second</td><td>State both clearly: "capacity caps burst, rate caps sustained throughput"</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Expected</th></tr><tr><td>First request at <code>t=0</code> (last is None)</td><td><code>allow(0, cost=1)</code></td><td><code>True</code> (bucket starts full)</td></tr><tr><td>Long idle period</td><td>No calls for 1000 s, then <code>allow(...)</code></td><td>Tokens clamped to <code>capacity</code>, not unbounded</td></tr><tr><td><code>cost &gt; capacity</code></td><td><code>allow(ts, cost=capacity+1)</code></td><td>Always <code>False</code> (impossible to ever grant)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Method</th><th align="left">Time</th><th align="left">Space</th></tr><tr><td><code>allow_request</code></td><td>O(1)</td><td>O(1)</td></tr></table>
<p style="margin:4px 0"><b>Key math:</b> <code>tokens += elapsed * rate</code>, then <b>clamp to capacity</b> (forgetting the clamp lets idle buckets accumulate unlimited burst). Capacity caps burst size; rate caps sustained throughput. Leaky bucket is the dual.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>933</td><td>Number of Recent Calls</td><td>Sliding-window rate limiting (deque alternative)</td></tr><tr><td>362</td><td>Design Hit Counter</td><td>Fixed-window rate tracking</td></tr><tr><td>1670</td><td>Design Front Middle Back Queue</td><td>Stateful queue design with O(1) ops</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">class TokenBucket:
    capacity:     int/float
    rate:         float         # tokens per second
    tokens:       float         # current token count
    last:         float | None  # timestamp of last update

    allow_request(timestamp, cost=1):
        if last is None:
            last = timestamp
        elapsed = timestamp - last
        tokens = min(capacity, tokens + elapsed * rate)   # refill + clamp
        last = timestamp
        if tokens &gt;= cost:
            tokens -= cost
            return True
        return False</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">class TokenBucket:
    def __init__(self, capacity, refill_rate):   # refill_rate = tokens per second
        self.capacity = capacity
        self.rate = refill_rate
        self.tokens = capacity
        self.last = None

    def allow_request(self, timestamp, cost=1):
        if self.last is None:
            self.last = timestamp
        elapsed = timestamp - self.last
        self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)  # refill, clamp
        self.last = timestamp
        if self.tokens &gt;= cost:
            self.tokens -= cost
            return True
        return False</pre></td></tr></table>

In [ ]:
# ===== Q19: Token Bucket Rate Limiter  (lazy refill) =====
class TokenBucket:
    def __init__(self, capacity, refill_rate):   # refill_rate = tokens per second
        self.capacity = capacity
        self.rate = refill_rate
        self.tokens = capacity
        self.last = None

    def allow_request(self, timestamp, cost=1):
        if self.last is None:
            self.last = timestamp
        elapsed = timestamp - self.last
        self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)  # refill, clamp
        self.last = timestamp
        if self.tokens >= cost:
            self.tokens -= cost
            return True
        return False
tb = TokenBucket(2, 1)   # capacity 2, 1 token/sec
c = [('t=0 allow', tb.allow_request(0) is True),
     ('t=0 allow', tb.allow_request(0) is True),
     ('t=0 deny (empty)', tb.allow_request(0) is False),
     ('t=1 allow (refilled 1)', tb.allow_request(1) is True),
     ('t=1 deny', tb.allow_request(1) is False)]
report("Q19", c)


<h3 style="margin:6px 0">Log Aggregator 🟡 (parse + per-service streaming stats)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Single-pass parse into per-service accumulators (<code>count</code>, <code>error_count</code>, <code>lat_sum</code>); compute <code>avg_latency = lat_sum / count</code> at the end — never store every latency.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given a list of log lines in the format <code>"&lt;timestamp&gt; &lt;service&gt; &lt;latency_ms&gt; &lt;status&gt;"</code>, compute per-service statistics: total request count, error count (status <code>ERROR</code> or HTTP status code ≥ 400), and average latency. Skip malformed lines. Return a dict mapping each service name to its stats.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input line</th><th align="left">Parsed</th><th align="left">Effect</th></tr><tr><td><code>"2026-01-01T00:00:00 api 120 200"</code></td><td>svc=api, lat=120, OK</td><td>api: count=1, lat_sum=120</td></tr><tr><td><code>"2026-01-01T00:00:01 api 95 500"</code></td><td>svc=api, lat=95, ERROR</td><td>api: count=2, errors=1, lat_sum=215</td></tr><tr><td><code>"2026-01-01T00:00:02 db 40 200"</code></td><td>svc=db, lat=40, OK</td><td>db: count=1, lat_sum=40</td></tr><tr><td><code>"bad line"</code></td><td>malformed (3 fields)</td><td>skipped</td></tr></table>
<p style="margin:4px 0">Output: <code>{"api": {"count": 2, "errors": 1, "avg_latency": 107.5}, "db": {"count": 1, "errors": 0, "avg_latency": 40.0}}</code></p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Accumulate <code>count</code>, <code>errors</code>, and <code>lat_sum</code> per service in a single pass. Never store individual latencies — a streaming mean (<code>lat_sum / count</code>) gives the average in O(1) memory per service. For follow-up percentiles, describe t-digest or histogram buckets (fixed-width buckets covering expected latency ranges). At the end, divide <code>lat_sum</code> by <code>count</code> only if <code>count &gt; 0</code> to avoid division-by-zero.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Storing all latencies in a list and averaging at the end</td><td>O(N) memory per service; fails on large log streams</td><td>Use streaming mean: track <code>lat_sum</code> and <code>count</code> only</td></tr><tr><td>Not guarding against <code>count == 0</code> when computing avg</td><td><code>ZeroDivisionError</code> for services seen only in malformed lines that were skipped</td><td><code>avg = lat_sum / count if count else 0.0</code></td></tr><tr><td>Classifying only <code>"ERROR"</code> string, not HTTP 4xx/5xx codes</td><td>Numeric error codes go uncounted</td><td>Check <code>status.isdigit() and int(status) &gt;= 400</code> as well</td></tr><tr><td>Crashing on non-numeric latency fields</td><td>Malformed latency fields propagate a <code>ValueError</code></td><td><code>try/except ValueError: continue</code></td></tr><tr><td><code>split()</code> on a line with no spaces gives <code>["whole_line"]</code></td><td><code>len(parts) != 4</code> guard catches it, but only if the guard is first</td><td>Check <code>len(parts) != 4</code> before any indexing</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Expected</th></tr><tr><td>All lines malformed</td><td><code>["bad", "also bad"]</code></td><td><code>{}</code> (empty result)</td></tr><tr><td>Status code exactly 400</td><td><code>"t svc 50 400"</code></td><td>Counted as error</td></tr><tr><td>Single service, single line</td><td><code>"t svc 100 200"</code></td><td><code>{"svc": {"count":1,"errors":0,"avg_latency":100.0}}</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(N)</td><td>Single pass over N log lines</td></tr><tr><td>Space</td><td>O(S)</td><td>S = number of distinct services</td></tr></table>
<p style="margin:4px 0"><b>Key idea:</b> one pass, streaming mean (<code>sum</code>/<code>count</code>) instead of storing every latency. <b>Follow-ups:</b> p95/p99 via a streaming sketch (t-digest / histogram buckets); time-windowed aggregation; top-K services by error rate (heap); map-reduce at scale. <b>Complexity:</b> O(N).</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>1396</td><td>Design Underground System</td><td>Per-route streaming average of journey times</td></tr><tr><td>2102</td><td>Sequentially Ordinal Rank Tracker</td><td>Streaming top-K with ordered state</td></tr><tr><td>703</td><td>Kth Largest Element in a Stream</td><td>Streaming aggregate with heap</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def aggregate(lines):
    agg = {}    # service -&gt; {count, errors, lat_sum}

    for line in lines:
        parts = line.split()
        if len(parts) != 4: continue        # malformed
        _, svc, latency_str, status = parts
        try:
            lat = float(latency_str)
        except ValueError:
            continue                        # non-numeric latency

        a = agg.setdefault(svc, {count:0, errors:0, lat_sum:0.0})
        a.count += 1
        a.lat_sum += lat
        if status is "ERROR" or (status is numeric and int(status) &gt;= 400):
            a.errors += 1

    return {
        svc: {count, errors, avg_latency: lat_sum/count if count else 0.0}
        for svc, a in agg.items()
    }</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">from collections import defaultdict

def aggregate(lines):
    agg = defaultdict(lambda: {"count": 0, "errors": 0, "lat_sum": 0.0})
    for line in lines:
        parts = line.split()
        if len(parts) != 4:
            continue                        # skip malformed
        _ts, svc, latency, status = parts
        try:
            lat = float(latency)
        except ValueError:
            continue
        a = agg[svc]
        a["count"] += 1
        a["lat_sum"] += lat
        if status.upper() == "ERROR" or (status.isdigit() and int(status) &gt;= 400):
            a["errors"] += 1
    return {
        svc: {
            "count": a["count"],
            "errors": a["errors"],
            "avg_latency": a["lat_sum"] / a["count"] if a["count"] else 0.0,
        }
        for svc, a in agg.items()
    }</pre></td></tr></table>

In [ ]:
# ===== Q20: Log Aggregator  (parse + per-service streaming stats) =====
from collections import defaultdict

def aggregate(lines):
    agg = defaultdict(lambda: {"count": 0, "errors": 0, "lat_sum": 0.0})
    for line in lines:
        parts = line.split()
        if len(parts) != 4:
            continue                        # skip malformed
        _ts, svc, latency, status = parts
        try:
            lat = float(latency)
        except ValueError:
            continue
        a = agg[svc]
        a["count"] += 1
        a["lat_sum"] += lat
        if status.upper() == "ERROR" or (status.isdigit() and int(status) >= 400):
            a["errors"] += 1
    return {
        svc: {
            "count": a["count"],
            "errors": a["errors"],
            "avg_latency": a["lat_sum"] / a["count"] if a["count"] else 0.0,
        }
        for svc, a in agg.items()
    }
lines = ["1 svcA 100 200","2 svcA 300 500","3 svcB 50 200","garbage line","4 svcB x 200"]
out = aggregate(lines)
c = [('svcA count 2', out["svcA"]["count"] == 2),
     ('svcA errors 1', out["svcA"]["errors"] == 1),
     ('svcA avg 200.0', approx(out["svcA"]["avg_latency"], 200.0)),
     ('svcB count 1 (bad lines skipped)', out["svcB"]["count"] == 1),
     ('svcB avg 50.0', approx(out["svcB"]["avg_latency"], 50.0))]
report("Q20", c)


<h3 style="margin:6px 0">Decode String (stack of context)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Push (current-string, repeat-count) onto a stack on <code>[</code>; pop and expand on <code>]</code>.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given an encoded string like <code>"3[a2[c]]"</code>, return the fully decoded string. The encoding rule is <code>k[encoded_string]</code>, where <code>encoded_string</code> is repeated exactly <code>k</code> times. <code>k</code> is a positive integer; brackets can be nested arbitrarily. The decoded output may be very long — constraints guarantee valid input (balanced brackets, positive <code>k</code>).</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>"3[a2[c]]"</code></td><td><code>"accaccacc"</code></td><td><code>"2[c]"</code> → <code>"cc"</code>, then <code>"a"+"cc"</code> × 3</td></tr><tr><td><code>"2[abc]3[cd]ef"</code></td><td><code>"abcabccdcdcdef"</code></td><td>two separate groups + suffix</td></tr><tr><td><code>"10[a]"</code></td><td><code>"aaaaaaaaaa"</code></td><td>multi-digit repeat count</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Process left-to-right with a stack. When you see a digit, accumulate into <code>num</code> (handles multi-digit). When you see <code>[</code>, push the current string and current number onto the stack and reset both. When you see <code>]</code>, pop the saved (prefix, repeat) and reconstruct <code>prefix + cur * k</code>. Letters just accumulate into <code>cur</code>.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Treating <code>num</code> as single digit (<code>num = int(ch)</code>)</td><td>Misparses <code>"10[a]"</code> → repeat 0 then 1 instead of 10</td><td><code>num = num * 10 + int(ch)</code></td></tr><tr><td>Forgetting to reset <code>cur</code> and <code>num</code> after <code>[</code></td><td>Inner group bleeds outer string/count into the nested decode</td><td><code>cur, num = "", 0</code> immediately after pushing</td></tr><tr><td>Using <code>num</code> directly from the outer scope on <code>]</code></td><td>Applies the wrong count — the pre-<code>[</code> count was already pushed</td><td>Always pop the count from the stack on <code>]</code></td></tr><tr><td>Not handling nested brackets (iterative stack vs. naive repeat)</td><td>Recursion gets complicated with multi-level nesting</td><td>Single stack with (prev_string, k) cleanly handles arbitrary depth</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Multi-digit repeat</td><td><code>"10[a]"</code></td><td><code>"aaaaaaaaaa"</code></td></tr><tr><td>Deep nesting</td><td><code>"2[3[a]]"</code></td><td><code>"aaaaaa"</code></td></tr><tr><td>No brackets</td><td><code>"abc"</code></td><td><code>"abc"</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(output length)</td><td>Each character of the output is written once during expansion</td></tr><tr><td>Space</td><td>O(output length)</td><td>Stack entries hold intermediate strings; worst case equals final length</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>726</td><td>Number of Atoms</td><td>Parentheses with multipliers — same push/pop pattern</td></tr><tr><td>1190</td><td>Reverse Substrings Between Each Pair of Parentheses</td><td>Stack-based bracket expansion</td></tr><tr><td>385</td><td>Mini Parser</td><td>Nested structure parsing with stack</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">stack = []          # entries: (string_before_bracket, repeat_count)
cur = ""
num = 0

for ch in s:
    if ch.isdigit():
        num = num * 10 + int(ch)    # multi-digit accumulation
    elif ch == "[":
        stack.push((cur, num))
        cur, num = "", 0            # reset for the inner scope
    elif ch == "]":
        prev, k = stack.pop()
        cur = prev + cur * k        # expand inner string
    else:
        cur += ch                   # plain letter

return cur</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def decodeString(s):
    stack = []                              # (string_before_bracket, repeat_count)
    cur, num = "", 0
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)        # multi-digit counts
        elif ch == "[":
            stack.append((cur, num))
            cur, num = "", 0
        elif ch == "]":
            prev, k = stack.pop()
            cur = prev + cur * k
        else:
            cur += ch
    return cur</pre></td></tr></table>

In [ ]:
# ===== Q21: Decode String (stack of context) =====
def decodeString(s):
    stack = []                              # (string_before_bracket, repeat_count)
    cur, num = "", 0
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)        # multi-digit counts
        elif ch == "[":
            stack.append((cur, num))
            cur, num = "", 0
        elif ch == "]":
            prev, k = stack.pop()
            cur = prev + cur * k
        else:
            cur += ch
    return cur
report("Q21", [
    ('3[a2[c]] -> accaccacc', decodeString("3[a2[c]]") == "accaccacc"),
    ('3[a]2[bc] -> aaabcbc', decodeString("3[a]2[bc]") == "aaabcbc"),
    ('2[abc]3[cd]ef', decodeString("2[abc]3[cd]ef") == "abcabccdcdcdef"),
])


<h3 style="margin:6px 0">Basic Calculator (+, −, parentheses)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Running total + sign variable; push (result, sign) on <code>(</code>, fold back on <code>)</code>.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Implement a basic calculator to evaluate a string expression containing non-negative integers, <code>+</code>, <code>-</code>, <code>(</code>, <code>)</code>, and spaces. No <code>*</code> or <code>/</code> — just addition and subtraction with arbitrary parentheses nesting. The input is always a valid expression; no leading/trailing whitespace issues.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>"1 + 1"</code></td><td><code>2</code></td><td>simple addition with spaces</td></tr><tr><td><code>"(1+(4+5+2)-3)+(6+8)"</code></td><td><code>23</code></td><td>nested parentheses</td></tr><tr><td><code>"-(3+(4+5))"</code></td><td><code>-12</code></td><td>leading negation via parenthesis</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Carry a running <code>result</code> and the current <code>sign</code> (+1 or -1). For digits, accumulate <code>num</code>. On <code>+</code>/<code>-</code> operator, flush <code>sign * num</code> into <code>result</code>, update <code>sign</code>. On <code>(</code>, push the current <code>result</code> and <code>sign</code> onto the stack then reset; the sign before <code>(</code> is what the entire sub-expression must be multiplied by. On <code>)</code>, flush the pending <code>num</code>, then pop the saved sign and result to fold the sub-expression back.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Forgetting the final <code>+ sign * num</code> after the loop</td><td>Last number token is never flushed into <code>result</code></td><td>Always add <code>sign * num</code> after the loop ends</td></tr><tr><td>Not resetting <code>num = 0</code> after flushing on <code>)</code></td><td>Stale <code>num</code> gets added again on the next operator or loop end</td><td>Set <code>num = 0</code> immediately after <code>result += sign * num</code> inside <code>)</code></td></tr><tr><td>Pushing only one value per <code>(</code></td><td>Loses either the accumulated <code>result</code> or the <code>sign</code></td><td>Push both <code>result</code> and <code>sign</code> in that order; pop in reverse</td></tr><tr><td>Treating spaces as operators</td><td>Crashes or misparses</td><td>Spaces are safely ignored — the <code>isdigit</code>/<code>in "+-"</code> checks skip them</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Expression ends with a number</td><td><code>"2-1"</code></td><td><code>1</code> (final flush matters)</td></tr><tr><td>Nested parentheses</td><td><code>"(1+(4+5+2)-3)+(6+8)"</code></td><td><code>23</code></td></tr><tr><td>Single number</td><td><code>"42"</code></td><td><code>42</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n)</td><td>Single pass over the string</td></tr><tr><td>Space</td><td>O(n)</td><td>Stack depth proportional to parenthesis nesting</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>227</td><td>Basic Calculator II</td><td>Same idea but adds <code>*</code> and <code>/</code> with precedence stack</td></tr><tr><td>772</td><td>Basic Calculator III</td><td>Combines both: parens + all four operators</td></tr><tr><td>150</td><td>Evaluate Reverse Polish Notation</td><td>Post-fix evaluation — no precedence needed</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">stack = []
result, num, sign = 0, 0, +1

for ch in s:
    if ch.isdigit():
        num = num * 10 + int(ch)
    elif ch in "+-":
        result += sign * num
        num = 0
        sign = +1 if ch == "+" else -1
    elif ch == "(":
        stack.push(result)
        stack.push(sign)
        result, sign = 0, +1        # fresh sub-expression
    elif ch == ")":
        result += sign * num
        num = 0
        result *= stack.pop()       # sign before "("
        result += stack.pop()       # accumulated result before "("

return result + sign * num          # flush last token</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def calculate(s):
    stack = []
    result, num, sign = 0, 0, 1
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch in "+-":
            result += sign * num
            num, sign = 0, (1 if ch == "+" else -1)
        elif ch == "(":
            stack.append(result); stack.append(sign)
            result, sign = 0, 1
        elif ch == ")":
            result += sign * num
            num = 0
            result *= stack.pop()           # sign that preceded "("
            result += stack.pop()           # result that preceded "("
    return result + sign * num</pre></td></tr></table>

In [ ]:
# ===== Q22: Basic Calculator (+, −, parentheses) =====
def calculate(s):
    stack = []
    result, num, sign = 0, 0, 1
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch in "+-":
            result += sign * num
            num, sign = 0, (1 if ch == "+" else -1)
        elif ch == "(":
            stack.append(result); stack.append(sign)
            result, sign = 0, 1
        elif ch == ")":
            result += sign * num
            num = 0
            result *= stack.pop()           # sign that preceded "("
            result += stack.pop()           # result that preceded "("
    return result + sign * num
report("Q22", [
    ('"1 + 1" -> 2', calculate("1 + 1") == 2),
    ('" 2-1 + 2 " -> 3', calculate(" 2-1 + 2 ") == 3),
    ('"(1+(4+5+2)-3)+(6+8)" -> 23', calculate("(1+(4+5+2)-3)+(6+8)") == 23),
])


<h3 style="margin:6px 0">Basic Calculator II (+, −, ×, ÷, no parens)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Defer the previous operand onto a stack keyed by operator precedence; sum the stack at the end.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Evaluate a string containing non-negative integers and operators <code>+</code>, <code>-</code>, <code>*</code>, <code>/</code> (no parentheses). Integer division truncates toward zero (not floor). Input is a valid expression; there may be spaces between tokens. Guaranteed no division by zero.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>"3+2*2"</code></td><td><code>7</code></td><td>multiplication before addition</td></tr><tr><td><code>" 3/2 "</code></td><td><code>1</code></td><td>truncation toward zero</td></tr><tr><td><code>"14-3/2"</code></td><td><code>13</code></td><td><code>-3/2</code> truncates to -1, then <code>14-1=13</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Track the previous operator (<code>op</code>, initialised to <code>"+"</code>). When you encounter an operator or reach the end, apply the previous <code>op</code> to the accumulated <code>num</code>: <code>+</code>/<code>-</code> push ±num; <code>*</code>/<code>/</code> pop the top, compute, and push back. At the end, sum the stack. This resolves precedence without a second pass: <code>*</code>/<code>/</code> consume the last operand immediately, while <code>+</code>/<code>-</code> defer theirs.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Using <code>//</code> for division</td><td><code>//</code> floors toward negative infinity; <code>-7 // 2 == -4</code> but expected <code>-3</code></td><td>Use <code>int(a / b)</code> which truncates toward zero</td></tr><tr><td>Not handling the last token</td><td>Loop ends before flushing the final <code>num</code></td><td>Trigger the flush condition also at <code>i == len(s) - 1</code></td></tr><tr><td>Initialising <code>op = ""</code> or <code>None</code></td><td>First number has no operator to dispatch on</td><td>Initialise <code>op = "+"</code> so the first number is pushed as-is</td></tr><tr><td>Treating spaces as operators</td><td>Space triggers the flush with <code>num = 0</code></td><td>Only dispatch when <code>ch in "+-*/"</code> or <code>i == last</code>, not on space</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Negative result from division</td><td><code>"-7/2"</code> (if negative were possible)</td><td><code>-3</code> (truncate toward zero)</td></tr><tr><td>String ending with operator token flush</td><td><code>"3+5"</code></td><td><code>8</code></td></tr><tr><td>Only multiplication</td><td><code>"3*2*2"</code></td><td><code>12</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n)</td><td>Single pass</td></tr><tr><td>Space</td><td>O(n)</td><td>Stack holds one entry per <code>+</code>/<code>-</code> term</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>224</td><td>Basic Calculator</td><td>Same structure but with parentheses instead of <code>*</code>/<code>/</code></td></tr><tr><td>772</td><td>Basic Calculator III</td><td>Combines parentheses + all four operators</td></tr><tr><td>150</td><td>Evaluate Reverse Polish Notation</td><td>Simpler — no operator precedence to resolve</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">stack = []
num, op = 0, "+"

for i, ch in enumerate(s):
    if ch.isdigit():
        num = num * 10 + int(ch)
    if ch in "+-*/" or i == last:
        if op == "+":  stack.push(+num)
        if op == "-":  stack.push(-num)
        if op == "*":  stack.push(stack.pop() * num)
        if op == "/":  stack.push(int(stack.pop() / num))  # truncate toward 0
        op, num = ch, 0

return sum(stack)</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def calculateII(s):
    stack = []
    num, op = 0, "+"
    for i, ch in enumerate(s):
        if ch.isdigit():
            num = num * 10 + int(ch)
        if ch in "+-*/" or i == len(s) - 1:
            if op == "+":   stack.append(num)
            elif op == "-": stack.append(-num)
            elif op == "*": stack.append(stack.pop() * num)
            elif op == "/": stack.append(int(stack.pop() / num))  # truncate toward 0
            op, num = ch, 0
    return sum(stack)</pre></td></tr></table>

In [ ]:
# ===== Q23: Basic Calculator II (+, −, ×, ÷, no parens) =====
def calculateII(s):
    stack = []
    num, op = 0, "+"
    for i, ch in enumerate(s):
        if ch.isdigit():
            num = num * 10 + int(ch)
        if ch in "+-*/" or i == len(s) - 1:
            if op == "+":   stack.append(num)
            elif op == "-": stack.append(-num)
            elif op == "*": stack.append(stack.pop() * num)
            elif op == "/": stack.append(int(stack.pop() / num))  # truncate toward 0
            op, num = ch, 0
    return sum(stack)
report("Q23", [
    ('"3+2*2" -> 7', calculateII("3+2*2") == 7),
    ('" 3/2 " -> 1', calculateII(" 3/2 ") == 1),
    ('" 3+5 / 2 " -> 5', calculateII(" 3+5 / 2 ") == 5),
])


<h3 style="margin:6px 0">Evaluate Reverse Polish Notation (stack)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Push operands; on operator pop two, compute, push result.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Evaluate an expression given as a list of tokens in Reverse Polish Notation (post-fix). Valid operators are <code>+</code>, <code>-</code>, <code>*</code>, <code>/</code>. Each operand is a valid integer (can be negative). Division truncates toward zero. The input always represents a valid RPN expression; exactly one value remains on the stack at the end.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>["2","1","+","3","*"]</code></td><td><code>9</code></td><td><code>(2+1)*3</code></td></tr><tr><td><code>["4","13","5","/","+"]</code></td><td><code>6</code></td><td><code>4+(13/5)=4+2=6</code></td></tr><tr><td><code>["10","6","9","3","+","-11","*","/","*","17","+","5","+"]</code></td><td><code>22</code></td><td>complex nested</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">There is no operator precedence to resolve — the order in post-fix is already correct. Maintain a stack of integers. For each token: if it is a number, push it; if it is an operator, pop <code>b</code> (top) then <code>a</code> (second), compute <code>a op b</code>, and push the result. After processing all tokens exactly one value remains.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Popping in wrong order: <code>a, b = pop(), pop()</code></td><td><code>a - b</code> becomes <code>b - a</code>; subtraction and division produce wrong sign/value</td><td>Pop <code>b</code> first (top), then <code>a</code> (deeper): <code>b, a = pop(), pop()</code></td></tr><tr><td>Using <code>//</code> for division</td><td>Floors toward -∞ for negatives: <code>int(-1/2)==0</code> but <code>-1//2==-1</code></td><td>Use <code>int(a / b)</code> to truncate toward zero</td></tr><tr><td>Not casting string tokens to <code>int</code></td><td>Arithmetic on strings raises <code>TypeError</code></td><td><code>stack.append(int(t))</code> for all non-operator tokens</td></tr><tr><td>Assuming tokens are always single characters</td><td>Negative numbers like <code>"-3"</code> are valid multi-char tokens</td><td>The <code>in {"+", "-", "*", "/"}</code> check correctly distinguishes them</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Single number</td><td><code>["42"]</code></td><td><code>42</code></td></tr><tr><td>Negative operand</td><td><code>["-2","3","*"]</code></td><td><code>-6</code></td></tr><tr><td>Division truncates toward zero</td><td><code>["7","-3","/"]</code></td><td><code>-2</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n)</td><td>One pass through the token list</td></tr><tr><td>Space</td><td>O(n)</td><td>Stack holds at most n/2 operands at once</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>224</td><td>Basic Calculator</td><td>In-fix with parentheses — requires sign tracking</td></tr><tr><td>227</td><td>Basic Calculator II</td><td>In-fix with precedence — requires precedence stack</td></tr><tr><td>856</td><td>Score of Parentheses</td><td>Stack-based evaluation of a bracket scoring formula</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">stack = []

for token in tokens:
    if token is operator:
        b = stack.pop()     # second operand (top)
        a = stack.pop()     # first operand (deeper)
        push result of a op b
    else:
        stack.push(int(token))

return stack[0]</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def evalRPN(tokens):
    stack = []
    for t in tokens:
        if t in {"+", "-", "*", "/"}:
            b, a = stack.pop(), stack.pop()
            if   t == "+": stack.append(a + b)
            elif t == "-": stack.append(a - b)
            elif t == "*": stack.append(a * b)
            else:          stack.append(int(a / b))   # truncate toward 0
        else:
            stack.append(int(t))
    return stack[0]</pre></td></tr></table>

In [ ]:
# ===== Q24: Evaluate Reverse Polish Notation (stack) =====
def evalRPN(tokens):
    stack = []
    for t in tokens:
        if t in {"+", "-", "*", "/"}:
            b, a = stack.pop(), stack.pop()
            if   t == "+": stack.append(a + b)
            elif t == "-": stack.append(a - b)
            elif t == "*": stack.append(a * b)
            else:          stack.append(int(a / b))   # truncate toward 0
        else:
            stack.append(int(t))
    return stack[0]
report("Q24", [
    ('["2","1","+","3","*"] -> 9', evalRPN(["2","1","+","3","*"]) == 9),
    ('["4","13","5","/","+"] -> 6', evalRPN(["4","13","5","/","+"]) == 6),
    ('long expr -> 22', evalRPN(["10","6","9","3","+","-11","*","/","*","17","+","5","+"]) == 22),
])


<h3 style="margin:6px 0">Valid Sudoku (three constraint sets)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Check rows, columns, and 3×3 boxes simultaneously using sets; box index is <code>(r//3)*3 + c//3</code>.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Determine if a 9×9 Sudoku board is valid. A board is valid if each row, each column, and each of the nine 3×3 sub-boxes contain no digit repeated. Empty cells are represented by <code>"."</code> and should be ignored. The board does not need to be solvable — only the filled cells are checked for validity.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td>Standard partially filled valid board</td><td><code>True</code></td><td>No row/col/box has a repeat</td></tr><tr><td>Board with <code>5</code> appearing twice in row 0</td><td><code>False</code></td><td>Duplicate in row</td></tr><tr><td>Board with <code>8</code> appearing twice in box (1,1)</td><td><code>False</code></td><td>Box index catches it</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Maintain three arrays of sets — one per row, one per column, one per 3×3 box. Scan every cell; for each digit, check all three constraint sets for a duplicate before adding to them. The only non-obvious piece is the box index: row <code>r</code> and column <code>c</code> map to box <code>(r // 3) * 3 + (c // 3)</code>, which gives a value in 0–8 numbering boxes left-to-right, top-to-bottom.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Wrong box index formula, e.g. <code>r // 3 + c // 3</code></td><td>Maps multiple distinct boxes to the same index</td><td>Correct formula: <code>(r // 3) * 3 + c // 3</code></td></tr><tr><td>Checking for duplicates after inserting</td><td>Never catches a duplicate — compares against a set that already contains the value</td><td>Check membership before adding</td></tr><tr><td>Skipping <code>"."</code> cells</td><td>Dot character is not a digit but may accidentally match or cause set pollution</td><td><code>if v == ".": continue</code> at the top of the inner loop</td></tr><tr><td>Using a single flat set with composite keys</td><td>Works but is slower to reason about; easy to get key construction wrong</td><td>Three separate arrays of sets is cleaner and idiomatic</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>All dots (empty board)</td><td>9×9 all <code>"."</code></td><td><code>True</code></td></tr><tr><td>Same digit in same box, different row and col</td><td><code>"5"</code> at (0,0) and (1,1)</td><td><code>False</code></td></tr><tr><td>Valid board that is unsolvable</td><td>Some valid but dead-end configuration</td><td><code>True</code> (validation only, not solvability)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(1)</td><td>Fixed 81 cells regardless of input</td></tr><tr><td>Space</td><td>O(1)</td><td>27 sets, each holding at most 9 elements</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>37</td><td>Solve Sudoku</td><td>Builds on this validator — adds backtracking to fill empty cells</td></tr><tr><td>36</td><td>Valid Sudoku</td><td>This problem itself (LC 36)</td></tr><tr><td>1001</td><td>Grid Illumination</td><td>Grid constraint tracking with hash sets</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">rows  = 9 sets
cols  = 9 sets
boxes = 9 sets

for r in 0..8:
    for c in 0..8:
        v = board[r][c]
        if v == ".": continue
        b = (r // 3) * 3 + c // 3
        if v in rows[r] or v in cols[c] or v in boxes[b]:
            return False
        add v to rows[r], cols[c], boxes[b]

return True</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def isValidSudoku(board):
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == ".":
                continue
            b = (r // 3) * 3 + c // 3        # box index
            if v in rows[r] or v in cols[c] or v in boxes[b]:
                return False
            rows[r].add(v); cols[c].add(v); boxes[b].add(v)
    return True</pre></td></tr></table>

In [ ]:
# ===== Q25: Valid Sudoku (three constraint sets) =====
def isValidSudoku(board):
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == ".":
                continue
            b = (r // 3) * 3 + c // 3        # box index
            if v in rows[r] or v in cols[c] or v in boxes[b]:
                return False
            rows[r].add(v); cols[c].add(v); boxes[b].add(v)
    return True
valid = [["5","3",".",".","7",".",".",".","."],
         ["6",".",".","1","9","5",".",".","."],
         [".","9","8",".",".",".",".","6","."],
         ["8",".",".",".","6",".",".",".","3"],
         ["4",".",".","8",".","3",".",".","1"],
         ["7",".",".",".","2",".",".",".","6"],
         [".","6",".",".",".",".","2","8","."],
         [".",".",".","4","1","9",".",".","5"],
         [".",".",".",".","8",".",".","7","9"]]
bad = [row[:] for row in valid]; bad[0][0] = "8"   # duplicate 8 in top-left box/col
report("Q25", [
    ('valid board -> True', isValidSudoku(valid) is True),
    ('duplicate -> False', isValidSudoku(bad) is False),
])


<h3 style="margin:6px 0">Solve Sudoku (backtracking with constraint sets)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Precompute empty cells and constraint sets; backtrack digit by digit with O(1) validity checks.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Fill an empty-cell Sudoku board in-place so that each row, column, and 3×3 box contains digits 1–9 exactly once. Input is always a valid, uniquely solvable 9×9 board with some cells pre-filled. Empty cells are represented by <code>"."</code>. Modify the board in-place; no return value needed.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td>Classic near-complete board</td><td>Board filled correctly</td><td>Standard case</td></tr><tr><td>Board with only one empty cell</td><td>That cell filled with the unique valid digit</td><td>Trivial backtrack depth</td></tr><tr><td>Board with 30+ empty cells</td><td>Solved board</td><td>Backtracking explores the search tree</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">This is a constraint-satisfaction problem solved by backtracking (choose → explore → unchoose). First scan the board: record existing digits into the three constraint sets and collect all empty cell positions. Then recurse through the empty cells in order. For each empty cell, try digits <code>"1"</code> through <code>"9"</code>; skip any that violate a row, column, or box constraint. If a digit fits, place it and update constraints, then recurse. If recursion succeeds, propagate <code>True</code> upward. If no digit works, undo (backtrack) and return <code>False</code>.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Re-scanning the full board at each step</td><td>O(81) per validation instead of O(1)</td><td>Precompute constraint sets; update incrementally on place/undo</td></tr><tr><td>Not collecting <code>empties</code> upfront — iterating board each level</td><td>Iterating inside <code>backtrack</code> is O(81) overhead per call</td><td>Collect empty positions once before recursing</td></tr><tr><td>Forgetting to <code>discard</code> all three sets on undo</td><td>Stale entry in rows/cols/boxes causes valid digits to be wrongly skipped</td><td>Discard from <code>rows[r]</code>, <code>cols[c]</code>, and <code>boxes[b]</code> together</td></tr><tr><td>Using <code>remove</code> instead of <code>discard</code> on undo</td><td>Raises <code>KeyError</code> if the value was never added (defensive)</td><td>Prefer <code>discard</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Single empty cell</td><td>One <code>"."</code> with unique valid digit</td><td>Filled in one step</td></tr><tr><td>Multiple solutions (invalid input by LC)</td><td>Not guaranteed by problem</td><td>First valid solution placed</td></tr><tr><td>All cells pre-filled and valid</td><td>No <code>"."</code> — <code>empties</code> is empty</td><td><code>backtrack(0)</code> returns <code>True</code> immediately</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(9^m) worst case, m = empty cells</td><td>Each empty cell tries up to 9 digits; pruning makes practical runtime fast</td></tr><tr><td>Space</td><td>O(m)</td><td>Recursion stack depth = number of empty cells; sets are O(1) fixed size</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>36</td><td>Valid Sudoku</td><td>Validation step extracted from this solver</td></tr><tr><td>51</td><td>N-Queens</td><td>Same choose-explore-unchoose backtracking structure</td></tr><tr><td>52</td><td>N-Queens II</td><td>Count solutions — same backtrack skeleton</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace"># Setup
rows, cols, boxes = 9 sets each
empties = []
scan board → populate constraint sets and empties list

def backtrack(i):
    if i == len(empties): return True   # all cells filled
    r, c = empties[i]
    b = (r // 3) * 3 + c // 3
    for d in "123456789":
        if d in rows[r] or d in cols[c] or d in boxes[b]:
            continue
        place d at board[r][c]; add d to rows/cols/boxes
        if backtrack(i + 1): return True
        undo: board[r][c] = "."; remove d from rows/cols/boxes
    return False

backtrack(0)</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def solveSudoku(board):
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    empties = []
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == ".":
                empties.append((r, c))
            else:
                rows[r].add(v); cols[c].add(v); boxes[(r // 3) * 3 + c // 3].add(v)

    def backtrack(i):
        if i == len(empties):
            return True
        r, c = empties[i]
        b = (r // 3) * 3 + c // 3
        for d in "123456789":
            if d in rows[r] or d in cols[c] or d in boxes[b]:
                continue
            board[r][c] = d
            rows[r].add(d); cols[c].add(d); boxes[b].add(d)
            if backtrack(i + 1):
                return True
            board[r][c] = "."               # undo
            rows[r].discard(d); cols[c].discard(d); boxes[b].discard(d)
        return False

    backtrack(0)</pre></td></tr></table>

In [ ]:
# ===== Q26: Solve Sudoku (backtracking with constraint sets) =====
def solveSudoku(board):
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    empties = []
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == ".":
                empties.append((r, c))
            else:
                rows[r].add(v); cols[c].add(v); boxes[(r // 3) * 3 + c // 3].add(v)

    def backtrack(i):
        if i == len(empties):
            return True
        r, c = empties[i]
        b = (r // 3) * 3 + c // 3
        for d in "123456789":
            if d in rows[r] or d in cols[c] or d in boxes[b]:
                continue
            board[r][c] = d
            rows[r].add(d); cols[c].add(d); boxes[b].add(d)
            if backtrack(i + 1):
                return True
            board[r][c] = "."               # undo
            rows[r].discard(d); cols[c].discard(d); boxes[b].discard(d)
        return False

    backtrack(0)
puzzle = [["5","3",".",".","7",".",".",".","."],
          ["6",".",".","1","9","5",".",".","."],
          [".","9","8",".",".",".",".","6","."],
          ["8",".",".",".","6",".",".",".","3"],
          ["4",".",".","8",".","3",".",".","1"],
          ["7",".",".",".","2",".",".",".","6"],
          [".","6",".",".",".",".","2","8","."],
          [".",".",".","4","1","9",".",".","5"],
          [".",".",".",".","8",".",".","7","9"]]
board = [r[:] for r in puzzle]; solveSudoku(board)
def _sudoku_ok(b):
    if any("." in r for r in b): return False
    grp = lambda idx: [set() for _ in range(9)]
    rows, cols, box = grp(0), grp(0), grp(0)
    for r in range(9):
        for c in range(9):
            v = b[r][c]; k=(r//3)*3+c//3
            if v in rows[r] or v in cols[c] or v in box[k]: return False
            rows[r].add(v); cols[c].add(v); box[k].add(v)
    return True
report("Q26", [('solved board complete & valid', _sudoku_ok(board))])


<h3 style="margin:6px 0">Number of Islands (flood-fill DFS)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Scan for unvisited land; DFS flood-fill each island, sinking cells in-place.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given an <code>m × n</code> grid of <code>"1"</code> (land) and <code>"0"</code> (water) characters, return the number of islands. An island is surrounded by water and formed by connecting adjacent land cells (4-directional: up, down, left, right). You may assume all four edges of the grid are water. Input cells are strings <code>"1"</code> and <code>"0"</code>, not integers.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td>4×5 grid with two disconnected land masses</td><td><code>2</code></td><td>Standard case</td></tr><tr><td>All <code>"0"</code></td><td><code>0</code></td><td>No land at all</td></tr><tr><td>All <code>"1"</code> (fully connected)</td><td><code>1</code></td><td>Single island spanning entire grid</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Iterate over every cell. When you find a <code>"1"</code>, increment the island count and immediately flood-fill the entire connected component by DFS — replacing each visited <code>"1"</code> with <code>"0"</code> (sink). This prevents double-counting without a separate <code>visited</code> set. Each cell is visited at most once across the entire outer loop.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Marking visited at dequeue (BFS) instead of enqueue</td><td>Same cell gets enqueued multiple times → O(cells²) and wrong counts</td><td>Mark <code>grid[r][c] = "0"</code> before appending to the queue, not when popping</td></tr><tr><td>Comparing to integer <code>1</code> not string <code>"1"</code></td><td><code>grid[r][c] != "1"</code> is always <code>True</code> when cells are strings</td><td>Grid cells are strings; compare to <code>"1"</code> and <code>"0"</code></td></tr><tr><td>Not restoring the grid (if caller expects it unchanged)</td><td>Input grid is mutated</td><td>Use a separate <code>visited</code> set if the grid must be preserved</td></tr><tr><td>Recursion depth overflow on large grids</td><td>Python default recursion limit ~1000</td><td>Use an explicit BFS queue or increase <code>sys.setrecursionlimit</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Empty grid</td><td><code>[]</code></td><td><code>0</code></td></tr><tr><td>Single cell <code>"1"</code></td><td><code>[["1"]]</code></td><td><code>1</code></td></tr><tr><td>Diagonal-only connection</td><td>Land at (0,0) and (1,1) only</td><td><code>2</code> (diagonals don't connect)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(rows × cols)</td><td>Each cell visited at most once</td></tr><tr><td>Space</td><td>O(rows × cols)</td><td>DFS stack depth in worst case (single snake-shaped island)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>695</td><td>Max Area of Island</td><td>Same flood-fill; return size of largest component</td></tr><tr><td>130</td><td>Surrounded Regions</td><td>Flood-fill from border to find non-surrounded <code>"O"</code> cells</td></tr><tr><td>417</td><td>Pacific Atlantic Water Flow</td><td>Multi-source BFS from two borders</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def dfs(r, c):
    if out-of-bounds or grid[r][c] != "1": return
    grid[r][c] = "0"                    # sink to mark visited
    dfs(r+1,c); dfs(r-1,c); dfs(r,c+1); dfs(r,c-1)

count = 0
for r in rows:
    for c in cols:
        if grid[r][c] == "1":
            count += 1
            dfs(r, c)
return count</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def numIslands(grid):
    if not grid:
        return 0
    rows, cols = len(grid), len(grid[0])
    count = 0

    def dfs(r, c):
        if not (0 &lt;= r &lt; rows and 0 &lt;= c &lt; cols) or grid[r][c] != "1":
            return
        grid[r][c] = "0"                    # sink to avoid revisits
        dfs(r + 1, c); dfs(r - 1, c); dfs(r, c + 1); dfs(r, c - 1)

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1":
                count += 1
                dfs(r, c)
    return count</pre></td></tr></table>

In [ ]:
# ===== Q27: Number of Islands (flood-fill DFS) =====
def numIslands(grid):
    if not grid:
        return 0
    rows, cols = len(grid), len(grid[0])
    count = 0

    def dfs(r, c):
        if not (0 <= r < rows and 0 <= c < cols) or grid[r][c] != "1":
            return
        grid[r][c] = "0"                    # sink to avoid revisits
        dfs(r + 1, c); dfs(r - 1, c); dfs(r, c + 1); dfs(r, c - 1)

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1":
                count += 1
                dfs(r, c)
    return count
g1 = [["1","1","0"],["1","0","0"],["0","0","1"]]
g2 = [["1","1","0","0","0"],["1","1","0","0","0"],["0","0","1","0","0"],["0","0","0","1","1"]]
report("Q27", [
    ('2 islands', numIslands(copy.deepcopy(g1)) == 2),
    ('3 islands', numIslands(copy.deepcopy(g2)) == 3),
    ('empty -> 0', numIslands([]) == 0),
])


<h3 style="margin:6px 0">Clone Graph (DFS + visited map)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> DFS with a <code>clones</code> dict; register the clone node before recursing into neighbors to break cycles.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given a reference to a node in a connected undirected graph, return a deep copy (clone) of the graph. Each node has a <code>val</code> (int) and a <code>neighbors</code> list of adjacent nodes. The graph may contain cycles. Node values are unique (1 to n). If the input node is <code>None</code>, return <code>None</code>.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td>Node 1 connected to nodes 2, 4; node 2 to 1, 3; etc. (4-cycle)</td><td>Deep copy of the 4-cycle</td><td>All edges cloned correctly</td></tr><tr><td>Single node, no neighbors</td><td>Clone of that single node</td><td>Trivial case</td></tr><tr><td><code>None</code></td><td><code>None</code></td><td>Empty graph</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Use a dictionary <code>clones</code> mapping original node → cloned node as both a visited set and the cloned-node registry. When DFS first visits a node: create the clone, store it in <code>clones</code> immediately (before recursing), then recursively clone each neighbor and attach to the clone's <code>neighbors</code>. The early registration is critical — if a cycle brings you back to a node already in <code>clones</code>, return the cached clone instead of creating a duplicate and looping forever.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Inserting into <code>clones</code> after recursing into neighbors</td><td>Cycle revisit finds no entry → creates a second clone → infinite recursion</td><td><code>clones[n] = copy</code> must come before the neighbor loop</td></tr><tr><td>Using <code>n.val</code> as the <code>clones</code> key instead of the node object</td><td>Different nodes may have the same value in some problem variants; also incorrect if <code>None</code> values</td><td>Use the node object itself as the key</td></tr><tr><td>Forgetting the <code>if n in clones</code> early-return</td><td>Every DFS call creates a new clone regardless of prior visits</td><td>Always check <code>clones</code> first and return the cached clone</td></tr><tr><td>Not handling <code>node is None</code> at the top level</td><td><code>dfs(None)</code> would call <code>None.val</code></td><td>Guard at the entry: <code>if not node: return None</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td><code>None</code> input</td><td><code>None</code></td><td><code>None</code></td></tr><tr><td>Single node, no neighbors</td><td>Node(1), <code>neighbors=[]</code></td><td>Clone of Node(1) with empty neighbors</td></tr><tr><td>Self-loop (node points to itself)</td><td>Node with itself in <code>neighbors</code></td><td>Clone with clone in its <code>neighbors</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(V + E)</td><td>Each node and edge visited exactly once</td></tr><tr><td>Space</td><td>O(V)</td><td><code>clones</code> dict + recursion stack, both proportional to number of nodes</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>138</td><td>Copy List with Random Pointer</td><td>Same visited-map trick for a linked list with random pointers</td></tr><tr><td>207</td><td>Course Schedule</td><td>Graph traversal — cycle detection instead of cloning</td></tr><tr><td>200</td><td>Number of Islands</td><td>Connected-component DFS on an implicit grid graph</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">clones = {}          # original_node -&gt; clone_node

def dfs(n):
    if n in clones: return clones[n]
    copy = Node(n.val)
    clones[n] = copy             # REGISTER before recursing (handles cycles)
    for neighbor in n.neighbors:
        copy.neighbors.append(dfs(neighbor))
    return copy

return dfs(node) if node else None</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def cloneGraph(node):
    if not node:
        return None
    clones = {}

    def dfs(n):
        if n in clones:
            return clones[n]
        copy = Node(n.val)
        clones[n] = copy                    # register BEFORE recursing (handles cycles)
        for nei in n.neighbors:
            copy.neighbors.append(dfs(nei))
        return copy

    return dfs(node)</pre></td></tr></table>

In [ ]:
# ===== Q28: Clone Graph (DFS + visited map) =====
def cloneGraph(node):
    if not node:
        return None
    clones = {}

    def dfs(n):
        if n in clones:
            return clones[n]
        copy = Node(n.val)
        clones[n] = copy                    # register BEFORE recursing (handles cycles)
        for nei in n.neighbors:
            copy.neighbors.append(dfs(nei))
        return copy

    return dfs(node)
src = build_graph([[2,4],[1,3],[2,4],[1,3]])
cl = cloneGraph(src)
report("Q28", [('isomorphic deep clone', graphs_isomorphic(src, cl)),
               ('None -> None', cloneGraph(None) is None)])


<h3 style="margin:6px 0">Course Schedule (cycle detection via Kahn's topo sort)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> BFS topological sort (Kahn's algorithm); a cycle exists if any node is never enqueued (in-degree never reaches zero).</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">There are <code>numCourses</code> courses labeled 0 to <code>numCourses - 1</code>. You are given a list of <code>prerequisites</code> where <code>[a, b]</code> means course <code>a</code> depends on course <code>b</code> (must take <code>b</code> before <code>a</code>). Return <code>True</code> if it is possible to finish all courses (i.e., no circular dependency exists), <code>False</code> otherwise.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>numCourses=2, prerequisites=[[1,0]]</code></td><td><code>True</code></td><td>Linear dependency</td></tr><tr><td><code>numCourses=2, prerequisites=[[1,0],[0,1]]</code></td><td><code>False</code></td><td>Mutual cycle</td></tr><tr><td><code>numCourses=3, prerequisites=[[1,0],[2,1]]</code></td><td><code>True</code></td><td>Chain: 0→1→2</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Model as a directed graph: edge <code>b → a</code> means "complete b before a." Kahn's algorithm repeatedly peels off nodes with zero in-degree (no prerequisites). If a node's prerequisite count reaches zero after removing a neighbor, it becomes eligible. If all <code>numCourses</code> nodes are processed (every course was reachable from the zero-in-degree frontier), the graph is acyclic. Any node left with in-degree &gt; 0 is trapped in a cycle and was never processed.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Reversing edge direction: <code>graph[a].append(b)</code></td><td>In-degrees computed backward; topological order is wrong</td><td>Edge should go from prerequisite to dependent: <code>graph[b].append(a)</code></td></tr><tr><td>Not seeding the queue with all zero-indegree nodes at start</td><td>Nodes with no prerequisites are never processed</td><td>Initialize queue with <code>[i for i in range(n) if indeg[i] == 0]</code></td></tr><tr><td>Checking <code>done == len(prerequisites)</code> instead of <code>done == numCourses</code></td><td>Prerequisites count ≠ course count; wrong termination</td><td>Compare against <code>numCourses</code></td></tr><tr><td>Using DFS cycle detection without visited states (white/grey/black)</td><td>Visiting a grey node correctly detects a cycle but is more error-prone</td><td>Kahn's BFS avoids tracking three states — simpler and equally correct</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>No prerequisites</td><td><code>numCourses=5, prerequisites=[]</code></td><td><code>True</code></td></tr><tr><td>All courses in a single cycle</td><td><code>[[1,0],[2,1],[0,2]]</code></td><td><code>False</code></td></tr><tr><td>Disconnected components, one cyclic</td><td>Mixed edges, one isolated cycle</td><td><code>False</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(V + E)</td><td>Each node dequeued once; each edge visited once</td></tr><tr><td>Space</td><td>O(V + E)</td><td>Adjacency list + in-degree array + queue</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>210</td><td>Course Schedule II</td><td>Return the actual topological order (collect pops into result list)</td></tr><tr><td>310</td><td>Minimum Height Trees</td><td>Kahn's from leaves inward — same peel-off structure</td></tr><tr><td>269</td><td>Alien Dictionary</td><td>Topological sort on a character order graph</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">graph = adjacency list (b -&gt; [a, ...])
indeg = [0] * numCourses
for a, b in prerequisites:
    graph[b].append(a)
    indeg[a] += 1

queue = all nodes with indeg == 0
done = 0
while queue not empty:
    node = dequeue
    done += 1
    for nxt in graph[node]:
        indeg[nxt] -= 1
        if indeg[nxt] == 0:
            enqueue(nxt)

return done == numCourses</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">from collections import defaultdict, deque

def canFinish(numCourses, prerequisites):
    graph = defaultdict(list)
    indeg = [0] * numCourses
    for a, b in prerequisites:              # a depends on b: edge b -&gt; a
        graph[b].append(a)
        indeg[a] += 1
    q = deque(i for i in range(numCourses) if indeg[i] == 0)
    done = 0
    while q:
        node = q.popleft()
        done += 1
        for nxt in graph[node]:
            indeg[nxt] -= 1
            if indeg[nxt] == 0:
                q.append(nxt)
    return done == numCourses               # all processed ⇒ acyclic</pre></td></tr></table>

In [ ]:
# ===== Q29: Course Schedule (cycle detection via Kahn's topo sort) =====
from collections import defaultdict, deque

def canFinish(numCourses, prerequisites):
    graph = defaultdict(list)
    indeg = [0] * numCourses
    for a, b in prerequisites:              # a depends on b: edge b -> a
        graph[b].append(a)
        indeg[a] += 1
    q = deque(i for i in range(numCourses) if indeg[i] == 0)
    done = 0
    while q:
        node = q.popleft()
        done += 1
        for nxt in graph[node]:
            indeg[nxt] -= 1
            if indeg[nxt] == 0:
                q.append(nxt)
    return done == numCourses               # all processed ⇒ acyclic
report("Q29", [
    ('[[1,0]] -> True', canFinish(2, [[1,0]]) is True),
    ('[[1,0],[0,1]] cycle -> False', canFinish(2, [[1,0],[0,1]]) is False),
])


<h3 style="margin:6px 0">Shortest Path in Binary Matrix (8-directional BFS)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> BFS from top-left; mark visited on enqueue (not dequeue); 8-directional neighbors.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given an <code>n × n</code> binary matrix, find the length of the shortest clear path from <code>(0, 0)</code> to <code>(n-1, n-1)</code>. A clear path uses only cells with value <code>0</code> and moves 8-directionally (horizontal, vertical, diagonal). Return the path length (number of cells visited, including start and end), or <code>-1</code> if no such path exists. If either the start or end cell is <code>1</code>, immediately return <code>-1</code>.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>[[0,1],[1,0]]</code></td><td><code>2</code></td><td>Diagonal path <code>(0,0)→(1,1)</code></td></tr><tr><td><code>[[0,0,0],[1,1,0],[1,1,0]]</code></td><td><code>4</code></td><td>BFS finds shortest 4-cell path</td></tr><tr><td><code>[[1,0,0],[1,1,0],[1,1,0]]</code></td><td><code>-1</code></td><td>Start cell is blocked</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">BFS guarantees the shortest path in an unweighted graph. Treat each <code>0</code> cell as a node with up to 8 neighbors. Initialize the queue with <code>(0, 0, distance=1)</code> and mark the start as visited by setting it to <code>1</code>. For each dequeued cell, if it is the target return the distance; otherwise enqueue all valid unvisited <code>0</code> neighbors, marking them visited immediately on enqueue to prevent re-enqueuing.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Marking visited on dequeue instead of enqueue</td><td>Same cell gets enqueued multiple times → wrong shortest distance and O(n²) per cell</td><td>Set <code>grid[nr][nc] = 1</code> before <code>q.append(...)</code>, not after dequeuing</td></tr><tr><td>Missing the n=1 edge case where <code>(0,0)</code> is also <code>(n-1,n-1)</code></td><td>Might return <code>-1</code> if the check <code>grid[0][0]</code> triggers on a 1×1 grid of <code>0</code></td><td>This implementation returns distance <code>1</code> correctly since <code>(0,0)</code> starts in queue</td></tr><tr><td>Using only 4 directions instead of 8</td><td>Misses diagonal shortest paths</td><td>Include all 8 <code>(dr, dc)</code> combinations</td></tr><tr><td>Checking bounds with <code>n</code> (column count) on non-square grids</td><td>Off-by-one for non-square inputs (this problem is always square, but be careful)</td><td>Use <code>len(grid)</code> and <code>len(grid[0])</code> separately for general grids</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Start blocked</td><td><code>grid[0][0] == 1</code></td><td><code>-1</code></td></tr><tr><td>End blocked</td><td><code>grid[n-1][n-1] == 1</code></td><td><code>-1</code></td></tr><tr><td>1×1 grid, cell is 0</td><td><code>[[0]]</code></td><td><code>1</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n²)</td><td>Each of the n² cells enqueued at most once</td></tr><tr><td>Space</td><td>O(n²)</td><td>Queue can hold O(n²) entries in worst case</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>200</td><td>Number of Islands</td><td>Grid BFS/DFS — component traversal instead of shortest path</td></tr><tr><td>1091</td><td>Shortest Path in Binary Matrix</td><td>This problem itself (LC 1091)</td></tr><tr><td>994</td><td>Rotting Oranges</td><td>Multi-source BFS on a grid</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">if grid[0][0] == 1 or grid[n-1][n-1] == 1: return -1

queue = [(0, 0, 1)]
grid[0][0] = 1          # mark visited on enqueue

while queue:
    r, c, d = dequeue
    if r == n-1 and c == n-1: return d
    for each of 8 directions (dr, dc):
        nr, nc = r+dr, c+dc
        if in-bounds and grid[nr][nc] == 0:
            grid[nr][nc] = 1    # mark on enqueue
            enqueue (nr, nc, d+1)

return -1</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">from collections import deque

def shortestPathBinaryMatrix(grid):
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1
    q = deque([(0, 0, 1)])
    grid[0][0] = 1                          # mark visited on enqueue
    dirs = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]
    while q:
        r, c, d = q.popleft()
        if r == n - 1 and c == n - 1:
            return d
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 &lt;= nr &lt; n and 0 &lt;= nc &lt; n and grid[nr][nc] == 0:
                grid[nr][nc] = 1
                q.append((nr, nc, d + 1))
    return -1</pre></td></tr></table>

In [ ]:
# ===== Q30: Shortest Path in Binary Matrix (8-directional BFS) =====
from collections import deque

def shortestPathBinaryMatrix(grid):
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1
    q = deque([(0, 0, 1)])
    grid[0][0] = 1                          # mark visited on enqueue
    dirs = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]
    while q:
        r, c, d = q.popleft()
        if r == n - 1 and c == n - 1:
            return d
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and grid[nr][nc] == 0:
                grid[nr][nc] = 1
                q.append((nr, nc, d + 1))
    return -1
report("Q30", [
    ('diagonal -> 2', shortestPathBinaryMatrix(copy.deepcopy([[0,1],[1,0]])) == 2),
    ('4-cell path', shortestPathBinaryMatrix(copy.deepcopy([[0,0,0],[1,1,0],[1,1,0]])) == 4),
    ('blocked start -> -1', shortestPathBinaryMatrix(copy.deepcopy([[1,0,0],[1,1,0],[1,1,0]])) == -1),
])


<h3 style="margin:6px 0">Network Delay Time (Dijkstra)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Dijkstra from source <code>k</code>; track finalized distances; answer is max distance (or -1 if any node unreachable).</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">You have <code>n</code> network nodes labeled 1 to <code>n</code>. Given a list of directed weighted edges <code>times</code> where <code>times[i] = (u, v, w)</code> means a signal travels from node <code>u</code> to node <code>v</code> in <code>w</code> time, send a signal from node <code>k</code>. Return the minimum time for all <code>n</code> nodes to receive the signal, or <code>-1</code> if not all nodes are reachable.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>times=[[2,1,1],[2,3,1],[3,4,1]], n=4, k=2</code></td><td><code>2</code></td><td>All reach: 2→1(1), 2→3(1), 3→4(2)</td></tr><tr><td><code>times=[[1,2,1]], n=2, k=2</code></td><td><code>-1</code></td><td>Node 1 unreachable from 2</td></tr><tr><td><code>times=[[1,2,1]], n=2, k=1</code></td><td><code>1</code></td><td>Node 2 reached in 1 unit</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Dijkstra finds single-source shortest paths on a non-negative weighted graph. Use a min-heap seeded with <code>(0, k)</code>. Each pop finalizes the shortest distance to a node. Skip already-finalized nodes (the <code>if node in dist: continue</code> guard). After the heap empties, check whether all <code>n</code> nodes are in <code>dist</code>; if not, some node was unreachable. The answer is the maximum finalized distance.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Not skipping already-finalized nodes</td><td>A node can be popped multiple times; later pops have larger distances and would overwrite the correct minimum</td><td><code>if node in dist: continue</code> at the top of the loop body</td></tr><tr><td>Checking <code>len(dist) == n</code> vs. <code>len(dist) == len(graph)</code></td><td>Nodes with no outgoing edges may not appear in <code>graph</code> but must still be counted</td><td>Count against <code>n</code> (the total number of nodes), not the graph's adjacency dict</td></tr><tr><td>Using BFS (unweighted) on a weighted graph</td><td>BFS gives wrong shortest paths when edge weights differ</td><td>Dijkstra with a min-heap is required for weighted graphs</td></tr><tr><td>Negative weights (using Dijkstra when Bellman-Ford needed)</td><td>Dijkstra's finalization assumption breaks with negative edges</td><td>This problem guarantees non-negative; for negative weights use Bellman-Ford</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Source is isolated</td><td><code>k</code> has no outgoing edges, <code>n &gt; 1</code></td><td><code>-1</code></td></tr><tr><td>All nodes directly connected to <code>k</code></td><td><code>n-1</code> edges from <code>k</code></td><td>Max of all edge weights</td></tr><tr><td>Single node</td><td><code>n=1, k=1, times=[]</code></td><td><code>0</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(E log V)</td><td>Each edge may generate a heap push; heap operations are O(log V)</td></tr><tr><td>Space</td><td>O(V + E)</td><td>Adjacency list + heap + dist dict</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>743</td><td>Network Delay Time</td><td>This problem itself (LC 743)</td></tr><tr><td>1514</td><td>Path with Maximum Probability</td><td>Dijkstra with max-product instead of min-sum</td></tr><tr><td>1631</td><td>Path with Minimum Effort</td><td>Dijkstra on a grid with edge weight = absolute height difference</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">graph = adjacency list from times
dist = {}
heap = [(0, k)]

while heap:
    d, node = heappop(heap)
    if node in dist: continue       # already finalized
    dist[node] = d
    for (neighbor, weight) in graph[node]:
        if neighbor not in dist:
            heappush(heap, (d + weight, neighbor))

return max(dist.values()) if len(dist) == n else -1</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">import heapq
from collections import defaultdict

def networkDelayTime(times, n, k):
    graph = defaultdict(list)
    for u, v, w in times:
        graph[u].append((v, w))
    dist = {}
    heap = [(0, k)]
    while heap:
        d, node = heapq.heappop(heap)
        if node in dist:
            continue                        # already finalized
        dist[node] = d
        for nei, w in graph[node]:
            if nei not in dist:
                heapq.heappush(heap, (d + w, nei))
    return max(dist.values()) if len(dist) == n else -1</pre></td></tr></table>

In [ ]:
# ===== Q31: Network Delay Time (Dijkstra) =====
import heapq
from collections import defaultdict

def networkDelayTime(times, n, k):
    graph = defaultdict(list)
    for u, v, w in times:
        graph[u].append((v, w))
    dist = {}
    heap = [(0, k)]
    while heap:
        d, node = heapq.heappop(heap)
        if node in dist:
            continue                        # already finalized
        dist[node] = d
        for nei, w in graph[node]:
            if nei not in dist:
                heapq.heappush(heap, (d + w, nei))
    return max(dist.values()) if len(dist) == n else -1
report("Q31", [
    ('reachable -> 2', networkDelayTime([[2,1,1],[2,3,1],[3,4,1]], 4, 2) == 2),
    ('unreachable -> -1', networkDelayTime([[1,2,1]], 2, 2) == -1),
])


<h3 style="margin:6px 0">Word Ladder (BFS over wildcard patterns)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Build wildcard-pattern buckets for O(L) neighbor lookup; BFS gives minimum transformation steps.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given a <code>beginWord</code>, an <code>endWord</code>, and a word list, find the length of the shortest transformation sequence from <code>beginWord</code> to <code>endWord</code> where each step changes exactly one letter and every intermediate word must exist in the word list. Return the number of words in the sequence (including begin and end), or <code>0</code> if no such sequence exists. All words have the same length <code>L</code>. <code>beginWord</code> is not necessarily in the word list.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>beginWord="hit", endWord="cog", wordList=["hot","dot","dog","lot","log","cog"]</code></td><td><code>5</code></td><td>hit→hot→dot→dog→cog</td></tr><tr><td><code>beginWord="hit", endWord="cog", wordList=["hot","dot","dog","lot","log"]</code></td><td><code>0</code></td><td><code>"cog"</code> not in list</td></tr><tr><td><code>beginWord="a", endWord="c", wordList=["a","b","c"]</code></td><td><code>2</code></td><td>a→c in one step</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Naively checking if two words differ by one character is O(L) per pair → O(N²L) total. Instead, precompute wildcard buckets: for each word <code>w</code> and each position <code>i</code>, key <code>w[:i] + "*" + w[i+1:]</code> maps to a list of all words sharing that pattern. Now neighbors of <code>word</code> are found in O(L) lookups. BFS from <code>beginWord</code> gives the minimum transformation count because all edges are weight 1.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Not checking <code>endWord in words</code> upfront</td><td>BFS may explore the full graph before concluding it's unreachable</td><td>Return <code>0</code> immediately if <code>endWord not in words</code></td></tr><tr><td>Marking seen at dequeue instead of enqueue</td><td>Same word enqueued from multiple predecessors → O(N²) extra work and wrong step counts</td><td>Add to <code>seen</code> before appending to queue</td></tr><tr><td>Building wildcard buckets only for words in <code>wordList</code> but not including <code>beginWord</code> neighbors</td><td><code>beginWord</code> is not in the list; its patterns may not appear in buckets</td><td>Buckets are built from <code>words</code> (the list); BFS generates patterns from the current word dynamically — patterns for <code>beginWord</code> are looked up against the list-built buckets, which is correct</td></tr><tr><td>Using O(N²L) pairwise comparison</td><td>For large word lists, exceeds time limit</td><td>Use precomputed wildcard buckets for O(NL²) preprocessing and O(L) per-neighbor lookup</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td><code>endWord</code> not in <code>wordList</code></td><td>Any</td><td><code>0</code></td></tr><tr><td><code>beginWord == endWord</code></td><td>Same word</td><td><code>1</code> (length-1 sequence)</td></tr><tr><td>No valid transformation</td><td>Disconnected words</td><td><code>0</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(N · L²)</td><td>N words × L patterns each × L chars per pattern string</td></tr><tr><td>Space</td><td>O(N · L²)</td><td>Wildcard bucket dict size</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>127</td><td>Word Ladder</td><td>This problem itself (LC 127)</td></tr><tr><td>126</td><td>Word Ladder II</td><td>All shortest paths — BFS + backtrack</td></tr><tr><td>433</td><td>Minimum Genetic Mutation</td><td>Identical BFS structure on a gene string alphabet</td></tr></table>
<p style="margin:4px 0"><b>See also:</b> [[wiki/syntheses/coding-dsa/breadth-first-search]], [[wiki/syntheses/coding-dsa/graphs]].</p></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">words = set(wordList)
if endWord not in words: return 0

# Precompute wildcard → [word, ...] buckets
for each word w in words:
    for i in range(L):
        pattern = w[:i] + "*" + w[i+1:]
        buckets[pattern].append(w)

queue = [(beginWord, 1)]
seen = {beginWord}

while queue:
    word, steps = dequeue
    if word == endWord: return steps
    for i in range(L):
        for nxt in buckets[word[:i] + "*" + word[i+1:]]:
            if nxt not in seen:
                seen.add(nxt)
                enqueue(nxt, steps + 1)

return 0</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">from collections import defaultdict, deque

def ladderLength(beginWord, endWord, wordList):
    words = set(wordList)
    if endWord not in words:
        return 0
    L = len(beginWord)
    patterns = defaultdict(list)
    for w in words:
        for i in range(L):
            patterns[w[:i] + "*" + w[i + 1:]].append(w)
    q = deque([(beginWord, 1)])
    seen = {beginWord}
    while q:
        word, steps = q.popleft()
        if word == endWord:
            return steps
        for i in range(L):
            for nxt in patterns[word[:i] + "*" + word[i + 1:]]:
                if nxt not in seen:
                    seen.add(nxt)
                    q.append((nxt, steps + 1))
    return 0</pre></td></tr></table>

In [ ]:
# ===== Q32: Word Ladder (BFS over wildcard patterns) =====
from collections import defaultdict, deque

def ladderLength(beginWord, endWord, wordList):
    words = set(wordList)
    if endWord not in words:
        return 0
    L = len(beginWord)
    patterns = defaultdict(list)
    for w in words:
        for i in range(L):
            patterns[w[:i] + "*" + w[i + 1:]].append(w)
    q = deque([(beginWord, 1)])
    seen = {beginWord}
    while q:
        word, steps = q.popleft()
        if word == endWord:
            return steps
        for i in range(L):
            for nxt in patterns[word[:i] + "*" + word[i + 1:]]:
                if nxt not in seen:
                    seen.add(nxt)
                    q.append((nxt, steps + 1))
    return 0
report("Q32", [
    ('hit->cog len 5', ladderLength("hit","cog",["hot","dot","dog","lot","log","cog"]) == 5),
    ('no endWord -> 0', ladderLength("hit","cog",["hot","dot","dog","lot","log"]) == 0),
])


<h3 style="margin:6px 0">Diameter of Binary Tree 🟡 (post-order depth)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Post-order DFS returning subtree depth; update a global max with <code>left_depth + right_depth</code> at each node.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given the <code>root</code> of a binary tree, return the length of the <b>diameter</b> — the longest path between any two nodes (the path does not need to pass through the root). Path length is measured in <b>edges</b>, not nodes. Constraints: the number of nodes is in <code>[1, 10⁴]</code>; node values are in <code>[-100, 100]</code>.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input (tree)</th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>[1,2,3,4,5]</code></td><td><code>3</code></td><td>Path 4→2→1→3 (or 4→2→5): 3 edges</td></tr><tr><td><code>[1,2]</code></td><td><code>1</code></td><td>Single edge</td></tr><tr><td><code>[1]</code></td><td><code>0</code></td><td>Single node; no edges</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">For every node, the longest path passing *through* it uses <code>left_depth</code> edges going left and <code>right_depth</code> edges going right, total <code>left_depth + right_depth</code>. A post-order traversal computes depths bottom-up, so when you visit a node both children's depths are already known — you can update the global max and return <code>1 + max(l, r)</code> as this node's depth to its parent. The diameter is the global max over all nodes.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Returning <code>l + r</code> instead of tracking a nonlocal/global max</td><td>The path through the root is returned, but the true diameter may pass through a subtree node</td><td>Use <code>nonlocal best</code> and update before returning depth</td></tr><tr><td>Counting nodes instead of edges (returning node count)</td><td>Off-by-one: a path through 4 nodes has 3 edges</td><td>Return <code>1 + max(l, r)</code> from <code>depth</code>, not <code>1 + l + r</code></td></tr><tr><td>Pre-order traversal (top-down height calls)</td><td>Recomputes subtree height O(n) times per node → O(n²) total</td><td>Use post-order so depths bubble up naturally, O(n)</td></tr><tr><td>Not handling <code>None</code> leaf children</td><td>Crashes on leaf nodes or single-node tree</td><td>Guard: <code>if not node: return 0</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Single node</td><td><code>[1]</code></td><td><code>0</code></td></tr><tr><td>Linear chain (skewed tree)</td><td><code>[1,2,null,3,null,4]</code></td><td><code>3</code></td></tr><tr><td>Diameter does not pass through root</td><td><code>[1,2,3,4,5,null,null,6,7]</code></td><td>path through node 2's subtree may exceed root's span</td><td>(verify with your tree shape)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n)</td><td>Every node visited exactly once in the post-order traversal</td></tr><tr><td>Space</td><td>O(h)</td><td>Recursion stack; O(log n) balanced, O(n) worst-case skewed</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>543</td><td>Diameter of Binary Tree</td><td>This problem</td></tr><tr><td>124</td><td>Binary Tree Maximum Path Sum</td><td>Same post-order pattern but with node values instead of edge counts</td></tr><tr><td>110</td><td>Balanced Binary Tree</td><td>Also returns height bottom-up; checks `</td><td>l - r</td><td>&lt;= 1`</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">best = 0

def depth(node):
    if node is None:
        return 0
    l = depth(node.left)
    r = depth(node.right)
    best = max(best, l + r)     # longest path through this node (edges)
    return 1 + max(l, r)        # this node's depth contribution upward

depth(root)
return best</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def diameterOfBinaryTree(root):
    best = 0

    def depth(node):
        nonlocal best
        if not node:
            return 0
        l, r = depth(node.left), depth(node.right)
        best = max(best, l + r)             # longest path through this node (edges)
        return 1 + max(l, r)

    depth(root)
    return best</pre></td></tr></table>

In [ ]:
# ===== Q33: Diameter of Binary Tree  (post-order depth) =====
def diameterOfBinaryTree(root):
    best = 0

    def depth(node):
        nonlocal best
        if not node:
            return 0
        l, r = depth(node.left), depth(node.right)
        best = max(best, l + r)             # longest path through this node (edges)
        return 1 + max(l, r)

    depth(root)
    return best
report("Q33", [
    ('[1,2,3,4,5] -> 3', diameterOfBinaryTree(build_tree([1,2,3,4,5])) == 3),
    ('single node -> 0', diameterOfBinaryTree(build_tree([1])) == 0),
])


<h3 style="margin:6px 0">Serialize and Deserialize Binary Tree 🟡 (preorder + null markers)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Preorder DFS emits node values and <code>#</code> for nulls; an iterator over the token stream reconstructs the tree in the same order.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Design an algorithm to serialize a binary tree to a string and deserialize that string back to the original tree. The encoding must be lossless — deserialize(serialize(root)) must produce a structurally identical tree. Constraints: nodes in <code>[0, 10⁴]</code>; values in <code>[-1000, 1000]</code>; your Codec class must expose <code>serialize(root) -&gt; str</code> and <code>deserialize(data) -&gt; TreeNode</code>.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Operation</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td><code>serialize([1,2,3,null,null,4,5])</code></td><td>root of 7-node tree</td><td><code>"1,2,#,#,3,4,#,#,5,#,#"</code></td></tr><tr><td><code>deserialize("1,2,#,#,3,4,#,#,5,#,#")</code></td><td>string</td><td>original tree reconstructed</td></tr><tr><td><code>serialize([])</code></td><td>empty tree</td><td><code>"#"</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Preorder traversal (root → left → right) uniquely reconstructs a tree when null positions are made explicit. Without null markers, the same level-order or inorder sequence can correspond to multiple different trees. Emitting <code>#</code> for a null fills every leaf's missing children — the token stream then encodes both structure and values. On deserialization, consuming tokens in the same preorder order naturally rebuilds the tree: read a token, if <code>#</code> return <code>None</code>, otherwise create a node and recursively build its left then right children.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Omitting null markers (<code>#</code>)</td><td>Inorder/preorder without nulls is ambiguous — multiple different trees serialize identically</td><td>Always emit a <code>#</code> for every null child</td></tr><tr><td>Using a list index instead of an iterator</td><td>With recursion, passing an int index and returning an updated index is error-prone</td><td>Wrap tokens in <code>iter()</code> so <code>next()</code> naturally advances the shared cursor</td></tr><tr><td>Joining on space and splitting on comma (or vice versa)</td><td>Mismatched delimiter corrupts the token stream</td><td>Use the same delimiter in <code>serialize</code> and <code>deserialize</code></td></tr><tr><td>Not handling negative values or multi-digit numbers</td><td>String <code>"-1"</code> contains <code>-</code> which might be mistaken for a separator</td><td>Use <code>","</code> as delimiter, not <code>-</code>; <code>int(v)</code> handles negatives correctly</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Empty tree</td><td><code>root = None</code></td><td><code>serialize → "#"</code>, <code>deserialize("#") → None</code></td></tr><tr><td>Single node</td><td><code>[5]</code></td><td><code>"5,#,#"</code></td></tr><tr><td>Left-skewed tree</td><td><code>[1,2,null,3]</code></td><td><code>"1,2,3,#,#,#,#"</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n)</td><td>Each node visited once in serialize; each token consumed once in deserialize</td></tr><tr><td>Space</td><td>O(n)</td><td>Output string + recursion stack (O(h) stack, O(n) output)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>297</td><td>Serialize and Deserialize Binary Tree</td><td>This problem</td></tr><tr><td>449</td><td>Serialize and Deserialize BST</td><td>BST order allows a more compact encoding (no nulls needed)</td></tr><tr><td>428</td><td>Serialize and Deserialize N-ary Tree</td><td>Extend null-marker approach; encode child count per node</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace"># Serialize
def serialize(root):
    out = []
    dfs(root, out)
    return ",".join(out)

def dfs(node, out):
    if node is None:
        out.append("#")
        return
    out.append(str(node.val))
    dfs(node.left, out)
    dfs(node.right, out)

# Deserialize
def deserialize(data):
    it = iter(data.split(","))
    return build(it)

def build(it):
    v = next(it)
    if v == "#":
        return None
    node = TreeNode(int(v))
    node.left = build(it)
    node.right = build(it)
    return node</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">class Codec:
    def serialize(self, root):
        out = []
        def dfs(node):
            if not node:
                out.append("#"); return
            out.append(str(node.val))
            dfs(node.left); dfs(node.right)
        dfs(root)
        return ",".join(out)

    def deserialize(self, data):
        vals = iter(data.split(","))
        def build():
            v = next(vals)
            if v == "#":
                return None
            node = TreeNode(int(v))
            node.left = build()
            node.right = build()
            return node
        return build()</pre></td></tr></table>

In [ ]:
# ===== Q34: Serialize and Deserialize Binary Tree  (preorder + null markers) =====
class Codec:
    def serialize(self, root):
        out = []
        def dfs(node):
            if not node:
                out.append("#"); return
            out.append(str(node.val))
            dfs(node.left); dfs(node.right)
        dfs(root)
        return ",".join(out)

    def deserialize(self, data):
        vals = iter(data.split(","))
        def build():
            v = next(vals)
            if v == "#":
                return None
            node = TreeNode(int(v))
            node.left = build()
            node.right = build()
            return node
        return build()
codec = Codec()
root = build_tree([1,2,3,None,None,4,5])
s1 = codec.serialize(root); rebuilt = codec.deserialize(s1)
report("Q34", [('round-trip preserves structure', preorder(rebuilt) == preorder(root)),
               ('serialize is stable', codec.serialize(rebuilt) == s1)])


<h3 style="margin:6px 0">All Nodes Distance K 🟡 (tree → graph, then BFS)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Build a parent-pointer graph (making the tree undirected), then BFS exactly <code>k</code> hops from the target node.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given the root of a binary tree, a <code>target</code> node, and an integer <code>k</code>, return a list of the values of all nodes that are exactly <code>k</code> edges away from <code>target</code>. The answer can be in any order. Constraints: nodes in <code>[1, 500]</code>; values unique; <code>0 &lt;= k &lt;= 1000</code>.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left"><code>k</code></th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>[3,5,1,6,2,0,8,null,null,7,4]</code>, target=5</td><td><code>2</code></td><td><code>[7,4,1]</code></td><td>7 and 4 are children of 2 (child of 5); 1 is the sibling of 5</td></tr><tr><td><code>[1]</code>, target=1</td><td><code>3</code></td><td><code>[]</code></td><td>Only one node; k=3 unreachable</td></tr><tr><td><code>[1,2,3]</code>, target=2</td><td><code>1</code></td><td><code>[1,3]</code> — wait, <code>[1]</code> is parent of 2; no, <code>[1]</code> yes plus child check</td><td><code>[1]</code> (parent) and no children → <code>[1]</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">A binary tree only has downward edges — you cannot traverse upward to a parent or to the other subtree. Adding parent pointers makes the tree behave like an undirected graph. Once undirected, BFS from the target naturally explores in all directions (up, down, sideways) and reports every node reached at depth exactly <code>k</code>. A <code>seen</code> set prevents revisiting nodes and handles the fact that the undirected graph has cycles (each parent-child edge is bidirectional).</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Forgetting the <code>seen</code> set</td><td>BFS oscillates between parent and child forever (undirected cycle)</td><td>Initialize <code>seen = {target}</code> and check before enqueuing</td></tr><tr><td>Only traversing downward (DFS on children only)</td><td>Misses nodes in the other subtree or above the target</td><td>Convert to undirected graph with <code>graph[parent].append(node)</code> both ways</td></tr><tr><td>Using node value as graph key when values are non-unique</td><td>Two nodes with the same value alias each other</td><td>Key by node object reference, not <code>.val</code></td></tr><tr><td>Continuing to expand after reaching depth <code>k</code></td><td>Enqueues <code>k+1</code> depth nodes unnecessarily (correctness is fine but wastes time)</td><td><code>continue</code> instead of <code>break</code> when <code>d == k</code> to still drain the same-depth nodes</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>Target is root, <code>k = 0</code></td><td>target=root, k=0</td><td><code>[root.val]</code></td></tr><tr><td><code>k</code> larger than tree depth</td><td>any tree, k=1000</td><td><code>[]</code></td></tr><tr><td>Target is a leaf node</td><td>leaf target, k=1</td><td><code>[parent.val]</code> only (no children)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n)</td><td>Build phase visits all n nodes; BFS also visits at most n nodes</td></tr><tr><td>Space</td><td>O(n)</td><td>Adjacency list + BFS queue + seen set each O(n)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>863</td><td>All Nodes Distance K in Binary Tree</td><td>This problem</td></tr><tr><td>994</td><td>Rotting Oranges</td><td>Multi-source BFS on a grid — same BFS-from-source pattern</td></tr><tr><td>1448</td><td>Count Good Nodes in Binary Tree</td><td>Tree traversal with a running value; same DFS skeleton</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace"># Phase 1: build parent pointers as an undirected adjacency list
graph = defaultdict(list)
def build(node, parent):
    if node is None: return
    if parent:
        graph[node].append(parent)
        graph[parent].append(node)
    build(node.left, node)
    build(node.right, node)
build(root, None)

# Phase 2: BFS from target
q = deque([(target, 0)])
seen = {target}
result = []
while q:
    node, d = q.popleft()
    if d == k:
        result.append(node.val)
        continue
    for nei in graph[node]:
        if nei not in seen:
            seen.add(nei)
            q.append((nei, d + 1))
return result</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">from collections import defaultdict, deque

def distanceK(root, target, k):
    graph = defaultdict(list)
    def build(node, parent):
        if not node:
            return
        if parent:
            graph[node].append(parent); graph[parent].append(node)
        build(node.left, node); build(node.right, node)
    build(root, None)

    q = deque([(target, 0)])
    seen = {target}
    res = []
    while q:
        node, d = q.popleft()
        if d == k:
            res.append(node.val); continue
        for nei in graph[node]:
            if nei not in seen:
                seen.add(nei); q.append((nei, d + 1))
    return res</pre></td></tr></table>

In [ ]:
# ===== Q35: All Nodes Distance K  (tree → graph, then BFS) =====
from collections import defaultdict, deque

def distanceK(root, target, k):
    graph = defaultdict(list)
    def build(node, parent):
        if not node:
            return
        if parent:
            graph[node].append(parent); graph[parent].append(node)
        build(node.left, node); build(node.right, node)
    build(root, None)

    q = deque([(target, 0)])
    seen = {target}
    res = []
    while q:
        node, d = q.popleft()
        if d == k:
            res.append(node.val); continue
        for nei in graph[node]:
            if nei not in seen:
                seen.add(nei); q.append((nei, d + 1))
    return res
root = build_tree([3,5,1,6,2,0,8,None,None,7,4])
target = find_node(root, 5)
report("Q35", [('distance 2 from node 5 -> {7,4,1}', eq_sorted(distanceK(root, target, 2), [7,4,1]))])


<h3 style="margin:6px 0">K Closest Points to Origin 🟡 (heap by squared distance)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Use <code>heapq.nsmallest</code> keyed by squared Euclidean distance; avoids <code>sqrt</code> and runs in O(n log k).</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given an array <code>points</code> of <code>(x, y)</code> coordinates and an integer <code>k</code>, return the <code>k</code> points closest to the origin <code>(0, 0)</code>. Distance is Euclidean; ties may be broken arbitrarily. Constraints: <code>1 &lt;= k &lt;= points.length &lt;= 10⁴</code>; coordinates in <code>[-10⁴, 10⁴]</code>.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left"><code>k</code></th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>[[1,3],[-2,2]]</code></td><td><code>1</code></td><td><code>[[-2,2]]</code></td><td>dist² = 8 &lt; 10</td></tr><tr><td><code>[[3,3],[5,-1],[-2,4]]</code></td><td><code>2</code></td><td><code>[[3,3],[-2,4]]</code></td><td>dist² = 18, 26, 20; two smallest are 18, 20</td></tr><tr><td><code>[[0,1],[1,0]]</code></td><td><code>2</code></td><td><code>[[0,1],[1,0]]</code></td><td>both equidistant; return both</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Euclidean distance is <code>sqrt(x² + y²)</code>, but since <code>sqrt</code> is monotonically increasing, sorting by <code>x² + y²</code> gives the same ranking — no floating-point <code>sqrt</code> needed. <code>heapq.nsmallest(k, …)</code> internally maintains a max-heap of size <code>k</code>, evicting the farthest seen point when a closer one is found, resulting in O(n log k). For very large n and small k, quickselect (O(n) average) is the theoretically optimal alternative, but <code>nsmallest</code> is idiomatic and sufficient.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Using <code>sqrt</code> in the key</td><td>Unnecessary floating-point computation and potential precision issues</td><td>Compare squared distances directly: <code>p[0]*p[0] + p[1]*p[1]</code></td></tr><tr><td>Building a full sort O(n log n) instead of a partial sort</td><td>Wastes time when k &lt;&lt; n</td><td>Use <code>heapq.nsmallest</code> (O(n log k)) or quickselect (O(n) average)</td></tr><tr><td>Using a min-heap of size n and popping k times</td><td>O(n) to heapify, O(k log n) to pop — correct but less efficient than max-heap of size k</td><td>For k &lt;&lt; n prefer a size-k max-heap; <code>nsmallest</code> handles this</td></tr><tr><td>Returning indices instead of the points themselves</td><td>Problem asks for the actual <code>[x, y]</code> pairs</td><td>Return <code>points[i]</code> not just <code>i</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td><code>k == len(points)</code></td><td><code>[[1,1],[2,2]]</code>, k=2</td><td>both points</td></tr><tr><td>Point at origin</td><td><code>[[0,0],[1,1]]</code>, k=1</td><td><code>[[0,0]]</code></td></tr><tr><td>All equidistant</td><td><code>[[1,0],[0,1],[-1,0],[0,-1]]</code>, k=2</td><td>any 2 (tie-breaking is arbitrary)</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n log k)</td><td><code>nsmallest</code> maintains a max-heap of size k while scanning n points</td></tr><tr><td>Space</td><td>O(k)</td><td>The heap holds at most k entries at any time</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>973</td><td>K Closest Points to Origin</td><td>This problem</td></tr><tr><td>215</td><td>Kth Largest Element in an Array</td><td>Same partial-sort idea; size-k min-heap</td></tr><tr><td>347</td><td>Top K Frequent Elements</td><td>Heap keyed by frequency instead of distance</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def kClosest(points, k):
    return heapq.nsmallest(
        k,
        points,
        key=lambda p: p[0]*p[0] + p[1]*p[1]   # squared distance, no sqrt
    )</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">import heapq

def kClosest(points, k):
    return heapq.nsmallest(k, points, key=lambda p: p[0] * p[0] + p[1] * p[1])</pre></td></tr></table>

In [ ]:
# ===== Q36: K Closest Points to Origin  (heap by squared distance) =====
import heapq

def kClosest(points, k):
    return heapq.nsmallest(k, points, key=lambda p: p[0] * p[0] + p[1] * p[1])
report("Q36", [
    ('k=1 -> [[-2,2]]', eq_tuples(kClosest([[1,3],[-2,2]],1), [[-2,2]])),
    ('k=2 -> {[3,3],[-2,4]}', eq_tuples(kClosest([[3,3],[5,-1],[-2,4]],2), [[3,3],[-2,4]])),
])


<h3 style="margin:6px 0">Find Median from Data Stream 🟡 (two balanced heaps)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Maintain a max-heap for the lower half and a min-heap for the upper half, kept balanced within one element; median is at one or both tops.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Design a data structure that supports streaming integer insertions and median queries. <code>addNum(num)</code> adds an integer; <code>findMedian()</code> returns the median of all elements inserted so far (exact if odd count, average of two middle values if even count). Constraints: <code>-10⁵ &lt;= num &lt;= 10⁵</code>; at most <code>5 × 10⁴</code> calls total.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Operation sequence</th><th align="left">After ops</th><th align="left"><code>findMedian()</code></th></tr><tr><td><code>addNum(1), addNum(2)</code></td><td>stream = [1,2]</td><td><code>1.5</code></td></tr><tr><td><code>addNum(3)</code></td><td>stream = [1,2,3]</td><td><code>2.0</code></td></tr><tr><td><code>addNum(1), addNum(7), addNum(3)</code></td><td>stream = [1,3,7]</td><td><code>3.0</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Split the sorted stream at the median point: the lower half sits in a <b>max-heap</b> (<code>small</code>, storing negated values since Python's <code>heapq</code> is a min-heap), and the upper half in a <b>min-heap</b> (<code>large</code>). Two invariants are maintained after every insert: (1) <b>value balance</b> — <code>small</code>'s maximum ≤ <code>large</code>'s minimum (every element in the lower half is ≤ every element in the upper half); (2) <b>size balance</b> — <code>|len(small) - len(large)| &lt;= 1</code>. Enforcing both requires pushing onto <code>small</code> first, then bouncing the max of <code>small</code> to <code>large</code>, then pulling back if <code>large</code> became larger. The median is then immediately readable from the tops.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Forgetting to negate values for the max-heap</td><td>Python's <code>heapq</code> is a min-heap; the top of <code>small</code> would be the minimum, not maximum</td><td>Store and retrieve as <code>-num</code>; negate on pop: <code>-heapq.heappop(self.small)</code></td></tr><tr><td>Skipping the value-balance step (just balancing sizes)</td><td><code>small</code>'s max might exceed <code>large</code>'s min, invalidating the split</td><td>Always push to <code>small</code> first, bounce the max to <code>large</code>, then size-rebalance</td></tr><tr><td>Allowing <code>large</code> to be larger than <code>small</code></td><td><code>findMedian</code> must know which half has the extra element; convention here gives <code>small</code> the extra</td><td>After balancing, ensure <code>len(small) &gt;= len(large)</code></td></tr><tr><td>Integer division in <code>findMedian</code></td><td><code>(a + b) // 2</code> floors the average; <code>3.5</code> becomes <code>3</code></td><td>Use <code>/</code> for float division</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input sequence</th><th align="left"><code>findMedian()</code></th></tr><tr><td>Single element</td><td><code>addNum(5)</code></td><td><code>5.0</code></td></tr><tr><td>Two elements</td><td><code>addNum(2), addNum(3)</code></td><td><code>2.5</code></td></tr><tr><td>All same value</td><td><code>addNum(1), addNum(1), addNum(1)</code></td><td><code>1.0</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(log n) per <code>addNum</code>, O(1) per <code>findMedian</code></td><td>Each <code>addNum</code> does a constant number of heap pushes/pops (each O(log n)); median reads from heap tops</td></tr><tr><td>Space</td><td>O(n)</td><td>Both heaps together hold all n elements</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>295</td><td>Find Median from Data Stream</td><td>This problem</td></tr><tr><td>480</td><td>Sliding Window Median</td><td>Same two-heap idea with add/remove for a fixed-size window</td></tr><tr><td>502</td><td>IPO</td><td>Max-heap + min-heap combined to greedily pick the highest-profit available project</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">small = []   # max-heap (negated) — lower half
large = []   # min-heap — upper half

def addNum(num):
    heappush(small, -num)                          # push to lower half
    heappush(large, -heappop(small))               # balance values: push small's max to large
    if len(large) &gt; len(small):                    # balance sizes
        heappush(small, -heappop(large))

def findMedian():
    if len(small) &gt; len(large):
        return -small[0]                           # odd total: lower half has extra element
    return (-small[0] + large[0]) / 2             # even total: average of two midpoints</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">import heapq

class MedianFinder:
    def __init__(self):
        self.small = []                     # max-heap (negated) — lower half
        self.large = []                     # min-heap — upper half

    def addNum(self, num):
        heapq.heappush(self.small, -num)
        heapq.heappush(self.large, -heapq.heappop(self.small))   # value balance
        if len(self.large) &gt; len(self.small):                    # size balance
            heapq.heappush(self.small, -heapq.heappop(self.large))

    def findMedian(self):
        if len(self.small) &gt; len(self.large):
            return -self.small[0]
        return (-self.small[0] + self.large[0]) / 2</pre></td></tr></table>

In [ ]:
# ===== Q37: Find Median from Data Stream  (two balanced heaps) =====
import heapq

class MedianFinder:
    def __init__(self):
        self.small = []                     # max-heap (negated) — lower half
        self.large = []                     # min-heap — upper half

    def addNum(self, num):
        heapq.heappush(self.small, -num)
        heapq.heappush(self.large, -heapq.heappop(self.small))   # value balance
        if len(self.large) > len(self.small):                    # size balance
            heapq.heappush(self.small, -heapq.heappop(self.large))

    def findMedian(self):
        if len(self.small) > len(self.large):
            return -self.small[0]
        return (-self.small[0] + self.large[0]) / 2
mf = MedianFinder(); mf.addNum(1); mf.addNum(2)
c = [('median(1,2) -> 1.5', approx(mf.findMedian(), 1.5))]
mf.addNum(3)
c.append(('median(1,2,3) -> 2.0', approx(mf.findMedian(), 2.0)))
report("Q37", c)


<h3 style="margin:6px 0">Search in Rotated Sorted Array 🟡 (identify sorted half, test membership)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> At each binary search step, one of the two halves is guaranteed sorted — identify which one, check if target falls in it, and discard the other.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">An integer array <code>nums</code> was sorted in ascending order, then rotated at some pivot. Given <code>nums</code> and an integer <code>target</code>, return its index if present, or <code>-1</code>. All values are <b>distinct</b>. Constraints: <code>1 &lt;= nums.length &lt;= 5000</code>; <code>-10⁴ &lt;= nums[i], target &lt;= 10⁴</code>; no duplicates.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Input</th><th align="left"><code>target</code></th><th align="left">Output</th><th align="left">Note</th></tr><tr><td><code>[4,5,6,7,0,1,2]</code></td><td><code>0</code></td><td><code>4</code></td><td>Rotated at index 4; left half [4,5,6,7] sorted</td></tr><tr><td><code>[4,5,6,7,0,1,2]</code></td><td><code>3</code></td><td><code>-1</code></td><td>Not present</td></tr><tr><td><code>[1]</code></td><td><code>0</code></td><td><code>-1</code></td><td>Single element, not equal</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">After a rotation, at least one of <code>[lo..mid]</code> or <code>[mid..hi]</code> is always a contiguous sorted range. Determine which half is sorted by comparing <code>nums[lo]</code> with <code>nums[mid]</code>: if <code>nums[lo] &lt;= nums[mid]</code>, the left half is sorted; otherwise the right half is sorted. Then ask whether <code>target</code> falls inside that sorted half's range — if yes, discard the other half; if no, discard the sorted half. This gives O(log n) binary search despite the rotation.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Using strict <code>&lt;</code> instead of <code>&lt;=</code> when checking which half is sorted (<code>nums[lo] &lt; nums[mid]</code>)</td><td>Fails when <code>lo == mid</code> (single-element left half); a two-element array with no rotation fails</td><td>Use <code>nums[lo] &lt;= nums[mid]</code> (equal means left half is trivially sorted)</td></tr><tr><td>Not handling duplicates (LC 81)</td><td>With duplicates, <code>nums[lo] == nums[mid]</code> is ambiguous — can't tell which half is sorted</td><td>For LC 81 add <code>if nums[lo] == nums[mid]: lo += 1; continue</code> to shrink the window</td></tr><tr><td>Checking <code>target &lt;= nums[hi]</code> without <code>nums[mid] &lt;</code> on the right-sorted branch</td><td>Boundary errors at exact <code>mid</code> or <code>hi</code> values</td><td>Include <code>nums[mid] &lt; target &lt;= nums[hi]</code> — both bounds required</td></tr><tr><td>Off-by-one on <code>hi = mid - 1</code> vs <code>hi = mid</code></td><td>Infinite loop when <code>lo == hi</code></td><td>Always move past <code>mid</code>: <code>hi = mid - 1</code> or <code>lo = mid + 1</code>, never <code>hi = mid</code> without convergence guarantee</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>No rotation</td><td><code>[1,2,3,4,5]</code>, target=3</td><td><code>2</code></td></tr><tr><td>Target at boundary</td><td><code>[4,5,6,7,0,1,2]</code>, target=4</td><td><code>0</code></td></tr><tr><td>Single element, match</td><td><code>[5]</code>, target=5</td><td><code>0</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(log n)</td><td>Search space halved at each step</td></tr><tr><td>Space</td><td>O(1)</td><td>Only index variables; no auxiliary structure</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>33</td><td>Search in Rotated Sorted Array</td><td>This problem</td></tr><tr><td>81</td><td>Search in Rotated Sorted Array II</td><td>Same algorithm; add <code>lo++</code> skip when <code>nums[lo]==nums[mid]</code> to handle duplicates</td></tr><tr><td>153</td><td>Find Minimum in Rotated Sorted Array</td><td>Same "identify sorted half" logic; return <code>nums[mid]</code> when right half is sorted</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">lo, hi = 0, len(nums) - 1
while lo &lt;= hi:
    mid = (lo + hi) // 2
    if nums[mid] == target:
        return mid
    if nums[lo] &lt;= nums[mid]:          # left half is sorted
        if nums[lo] &lt;= target &lt; nums[mid]:
            hi = mid - 1              # target in sorted left half
        else:
            lo = mid + 1              # target in right half
    else:                             # right half is sorted
        if nums[mid] &lt; target &lt;= nums[hi]:
            lo = mid + 1              # target in sorted right half
        else:
            hi = mid - 1             # target in left half
return -1</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def search(nums, target):
    lo, hi = 0, len(nums) - 1
    while lo &lt;= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[lo] &lt;= nums[mid]:           # left half is sorted
            if nums[lo] &lt;= target &lt; nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:                               # right half is sorted
            if nums[mid] &lt; target &lt;= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1</pre></td></tr></table>

In [ ]:
# ===== Q38: Search in Rotated Sorted Array  (identify sorted half, test membership) =====
def search(nums, target):
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[lo] <= nums[mid]:           # left half is sorted
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:                               # right half is sorted
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1
report("Q38", [
    ('find 0 -> idx 4', search([4,5,6,7,0,1,2], 0) == 4),
    ('missing -> -1', search([4,5,6,7,0,1,2], 3) == -1),
    ('single miss -> -1', search([1], 0) == -1),
])


<h3 style="margin:6px 0">IoU for Bounding Boxes 🟡 (clamp intersection, union formula)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Compute intersection rectangle by clamping coordinate pairs with <code>max</code>/<code>min</code>; area = <code>max(0, w) * max(0, h)</code>; union = area_a + area_b − intersection.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given two axis-aligned bounding boxes <code>box_a</code> and <code>box_b</code>, each in <code>[x1, y1, x2, y2]</code> format (top-left and bottom-right corners, continuous coordinates), compute their Intersection over Union (IoU). IoU = area of intersection / area of union; return <code>0.0</code> if either box is degenerate (zero area) or they don't overlap. Coordinates are floats; <code>x1 &lt; x2</code> and <code>y1 &lt; y2</code> should hold, but defensively clamp.</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left"><code>box_a</code></th><th align="left"><code>box_b</code></th><th align="left">IoU</th><th align="left">Note</th></tr><tr><td><code>[0,0,2,2]</code></td><td><code>[1,1,3,3]</code></td><td><code>1/7 ≈ 0.143</code></td><td>Intersection 1×1=1; union = 4+4−1=7</td></tr><tr><td><code>[0,0,1,1]</code></td><td><code>[2,2,3,3]</code></td><td><code>0.0</code></td><td>No overlap</td></tr><tr><td><code>[0,0,2,2]</code></td><td><code>[0,0,2,2]</code></td><td><code>1.0</code></td><td>Identical boxes</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">The intersection rectangle's x-span is <code>[max(ax1,bx1), min(ax2,bx2)]</code> and y-span is <code>[max(ay1,by1), min(ay2,by2)]</code>. If <code>max &gt; min</code> in either dimension the boxes don't overlap in that axis and the intersection is empty. Clamping with <code>max(0, …)</code> converts a negative width or height to zero rather than producing a negative "intersection". The union is computed by inclusion-exclusion: <code>area_a + area_b − intersection</code> (subtracting once the doubly-counted intersection region). Dividing by zero occurs when both boxes are degenerate (zero area and zero union); guard with <code>if union &gt; 0 else 0.0</code>.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Not clamping intersection dimensions to 0</td><td>Disjoint boxes produce negative <code>iw</code> or <code>ih</code>; multiplying gives a positive bogus intersection area</td><td><code>iw = max(0.0, ix2 - ix1)</code> and <code>ih = max(0.0, iy2 - iy1)</code></td></tr><tr><td><code>+1</code> pixel convention for continuous coordinates</td><td>Pixel IoU uses <code>(x2 - x1 + 1)*(y2 - y1 + 1)</code>; applying it to continuous coords inflates small boxes</td><td>Only add <code>+1</code> if problem explicitly says pixel-space integer boxes</td></tr><tr><td>Dividing <code>inter / union</code> without guarding <code>union == 0</code></td><td>Zero-area input boxes give <code>union = 0</code>; Python raises <code>ZeroDivisionError</code></td><td><code>return inter / union if union &gt; 0 else 0.0</code></td></tr><tr><td>Computing union as just the bounding box area</td><td>Over-counts non-overlapping space; correct union is <code>area_a + area_b - inter</code></td><td>Inclusion-exclusion: subtract the intersection once</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>No overlap</td><td><code>[0,0,1,1]</code> vs <code>[2,2,3,3]</code></td><td><code>0.0</code></td></tr><tr><td>One box inside the other</td><td><code>[0,0,4,4]</code> vs <code>[1,1,2,2]</code></td><td><code>1/16 = 0.0625</code></td></tr><tr><td>Identical boxes</td><td><code>[1,1,3,3]</code> vs <code>[1,1,3,3]</code></td><td><code>1.0</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(1)</td><td>Fixed number of arithmetic operations regardless of box coordinates</td></tr><tr><td>Space</td><td>O(1)</td><td>No auxiliary data structures</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>836</td><td>Rectangle Overlap</td><td>Boolean version: do two boxes overlap? Same <code>max/min</code> intersection check</td></tr><tr><td>223</td><td>Rectangle Area</td><td>Computes union area of two rectangles; same inclusion-exclusion formula</td></tr><tr><td>N/A</td><td>NMS (Q40 below)</td><td>IoU is the core primitive for NMS; implement IoU first, then call it in NMS</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    # intersection corners
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    # clamp to 0 if no overlap in either axis
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union &gt; 0 else 0.0</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)   # clamp: 0 if no overlap
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union &gt; 0 else 0.0</pre></td></tr></table>

In [ ]:
# ===== Q39: IoU for Bounding Boxes  (clamp intersection, union formula) =====
def iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)   # clamp: 0 if no overlap
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0
report("Q39", [
    ('overlap IoU = 1/7', approx(iou([0,0,2,2],[1,1,3,3]), 1/7)),
    ('disjoint -> 0', approx(iou([0,0,1,1],[2,2,3,3]), 0.0)),
])


<h3 style="margin:6px 0">Non-Max Suppression 🟡 (greedy score-sorted suppression)</h3>
<p style="margin:2px 0 8px;color:#475569"><b>Pattern:</b> Sort boxes by confidence descending; greedily keep the highest-scoring box, discard all remaining boxes with IoU above the threshold, repeat.</p>
<table style="width:100%;border-collapse:collapse"><tr><td style="width:46%;vertical-align:top;padding-right:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PROBLEM</div><p style="margin:4px 0">Given a list of bounding boxes <code>boxes</code> (each <code>[x1,y1,x2,y2]</code>), a parallel list of confidence <code>scores</code>, and a float <code>iou_threshold</code>, return the indices of the boxes that survive NMS. NMS greedily keeps the highest-scoring detection and removes overlapping boxes (IoU &gt; threshold) as duplicates of the same object. This is a core post-processing step in every object detector (YOLO, Faster R-CNN, etc.).</p>
<table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left"><code>boxes</code></th><th align="left"><code>scores</code></th><th align="left"><code>iou_threshold</code></th><th align="left"><code>keep</code></th><th align="left">Note</th></tr><tr><td><code>[[0,0,2,2],[0.5,0.5,2.5,2.5],[5,5,7,7]]</code></td><td><code>[0.9,0.75,0.8]</code></td><td><code>0.5</code></td><td><code>[0, 2]</code></td><td>Box 1 suppressed by box 0 (IoU &gt; 0.5); box 2 kept (no overlap)</td></tr><tr><td><code>[[0,0,1,1]]</code></td><td><code>[0.9]</code></td><td><code>0.5</code></td><td><code>[0]</code></td><td>Single box always kept</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">BASIC FUNDA</div><p style="margin:4px 0">Multiple detector proposals often fire on the same object, producing highly overlapping boxes. NMS keeps only the most confident detection per object cluster. The algorithm is greedy: take the highest-confidence surviving box as a true detection, then remove all others whose IoU with it exceeds the threshold (they are redundant detections of the same object), and repeat. Sorting by score first ensures each chosen survivor is always the best available.</p>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">THINGS TO TAKE CARE</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Pitfall</th><th align="left">Why it breaks</th><th align="left">Fix</th></tr><tr><td>Suppressing a box with itself (IoU = 1.0)</td><td>The just-selected box <code>i</code> gets removed from its own future comparisons if you include it</td><td>Pop <code>i</code> from <code>order</code> before computing IoU comparisons; only compare remaining boxes</td></tr><tr><td>Using <code>&lt;</code> instead of <code>&lt;=</code> for the suppression threshold</td><td>Boxes at exactly <code>iou_threshold</code> are kept instead of suppressed (or vice versa, depending on convention)</td><td>Be explicit about whether the threshold is inclusive or exclusive; <code>&lt;= iou_threshold</code> keeps boxes strictly below</td></tr><tr><td>Forgetting to call per-class NMS separately</td><td>Running NMS across all classes suppresses boxes from different classes that happen to overlap</td><td>In production: split by class, run NMS independently, concatenate results</td></tr><tr><td>O(n²) plain Python NMS in production</td><td>Slow for hundreds of proposals per image</td><td>Use vectorized NumPy (broadcast box subtraction) or <code>torchvision.ops.nms</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CORNER CASES</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Case</th><th align="left">Input</th><th align="left">Output</th></tr><tr><td>All boxes identical</td><td>Three copies of <code>[0,0,1,1]</code>, scores <code>[0.9,0.8,0.7]</code></td><td><code>[0]</code> (highest score; all others suppressed)</td></tr><tr><td>No overlapping boxes</td><td>Three disjoint boxes</td><td>All indices kept</td></tr><tr><td>Single box</td><td><code>[[0,0,1,1]]</code>, score <code>[0.5]</code></td><td><code>[0]</code></td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">COMPLEXITY</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">Metric</th><th align="left">Value</th><th align="left">Why</th></tr><tr><td>Time</td><td>O(n²) worst case</td><td>For each of n survivors, IoU is computed with all remaining candidates; degenerate case: all boxes disjoint, O(n²) IoU calls</td></tr><tr><td>Space</td><td>O(n)</td><td><code>order</code> list and <code>keep</code> list each at most n elements</td></tr></table>
<div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">RELATED QUESTIONS</div><table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;margin:4px 0"><tr><th align="left">LC #</th><th align="left">Name</th><th align="left">Connection</th></tr><tr><td>836</td><td>Rectangle Overlap</td><td>Boolean overlap check — the primitive underlying IoU</td></tr><tr><td>223</td><td>Rectangle Area</td><td>Union-area formula needed inside IoU</td></tr><tr><td>N/A</td><td>Soft-NMS (Bodla et al. 2017)</td><td>Replaces hard removal with score decay; same greedy loop, different suppression step</td></tr></table></td><td style="width:54%;vertical-align:top;border-left:1px solid #cbd5e1;padding-left:16px"><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">PSEUDOCODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">order = argsort(scores, descending=True)   # indices sorted by confidence
keep = []
while order is not empty:
    i = order.pop_front()                  # highest-confidence survivor
    keep.append(i)
    order = [j for j in order if iou(boxes[i], boxes[j]) &lt;= iou_threshold]
return keep</pre><div style="font-weight:700;color:#2563eb;margin:10px 0 2px;font-size:13px;letter-spacing:.03em">CODE</div><pre style="background:#0f172a;color:#e2e8f0;padding:10px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.45;margin:4px 0;white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace">def nms(boxes, scores, iou_threshold):
    order = sorted(range(len(boxes)), key=lambda i: scores[i], reverse=True)
    keep = []
    while order:
        i = order.pop(0)                    # highest-scoring survivor
        keep.append(i)
        order = [j for j in order if iou(boxes[i], boxes[j]) &lt;= iou_threshold]
    return keep</pre></td></tr></table>

In [ ]:
# ===== Q40: Non-Max Suppression  (greedy score-sorted suppression) =====
def nms(boxes, scores, iou_threshold):
    order = sorted(range(len(boxes)), key=lambda i: scores[i], reverse=True)
    keep = []
    while order:
        i = order.pop(0)                    # highest-scoring survivor
        keep.append(i)
        order = [j for j in order if iou(boxes[i], boxes[j]) <= iou_threshold]
    return keep
# nms reuses the IoU function from Q39
def iou(a, b):
    ix1,iy1 = max(a[0],b[0]), max(a[1],b[1])
    ix2,iy2 = min(a[2],b[2]), min(a[3],b[3])
    iw,ih = max(0.0,ix2-ix1), max(0.0,iy2-iy1); inter = iw*ih
    ua = max(0.0,a[2]-a[0])*max(0.0,a[3]-a[1]); ub = max(0.0,b[2]-b[0])*max(0.0,b[3]-b[1])
    u = ua+ub-inter
    return inter/u if u>0 else 0.0
boxes = [[0,0,1,1],[0.1,0.1,1,1],[2,2,3,3]]; scores=[0.9,0.8,0.7]
report("Q40", [('keeps box 0 and 2, suppresses 1', eq_sorted(nms(boxes, scores, 0.5), [0,2]))])


## Scoreboard

In [ ]:
passed = sum(1 for v in RESULTS.values() if v)
print(f"Solved {passed}/40")
missing = [q for q in (f"Q{i}" for i in range(1,41)) if q not in RESULTS]
failed  = [q for q,v in RESULTS.items() if not v]
if missing: print("Not run yet:", ", ".join(missing))
if failed:  print("Failing   :", ", ".join(failed))
if not missing and not failed: print("All 40 pass ✅")
